In [1]:
# ============================================================
# CELDA 1 — ENTORNO Y LOCALIZACIÓN DE MODELOS
# Solo consulta información. No modifica archivos.
# ============================================================

import sys
import os
import platform
from pathlib import Path
from importlib import metadata

print("=" * 70)
print("1. ENTORNO DEL CUADERNO")
print("=" * 70)

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
print("Sistema:", platform.platform())
print("Carpeta del cuaderno:", Path.cwd())

print("\nVersiones de las bibliotecas:")

bibliotecas = [
    "numpy",
    "pandas",
    "rasterio",
    "torch",
    "torchvision",
    "scikit-learn",
    "openpyxl",
]

for biblioteca in bibliotecas:
    try:
        version = metadata.version(biblioteca)
        print(f"  {biblioteca}: {version}")
    except metadata.PackageNotFoundError:
        print(f"  {biblioteca}: NO INSTALADA EN ESTE ENTORNO")


# Rutas mencionadas en el proyecto y en los cuadernos anteriores.
# Se consultan únicamente las que existan en este computador.

RUTAS_BUSQUEDA = [
    Path(r"D:\PROYECTO_GRADO_1"),
    Path(r"C:\Users\Ivan sua\anaconda3\Modelos_UNET"),
    Path(r"C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion"),
    Path(r"C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion"),
]

print("\n" + "=" * 70)
print("2. CARPETAS DISPONIBLES")
print("=" * 70)

rutas_disponibles = []

for ruta in RUTAS_BUSQUEDA:
    existe = ruta.is_dir()
    print(f"{'ENCONTRADA' if existe else 'NO ENCONTRADA'}: {ruta}")

    if existe:
        rutas_disponibles.append(ruta)


print("\n" + "=" * 70)
print("3. ARCHIVOS DE PESOS Y CONFIGURACIONES")
print("=" * 70)

pesos_encontrados = set()
configuraciones_encontradas = set()
errores_lectura = []

# Se omiten carpetas de entornos y cachés.
omitir = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules",
    "envs",
    "pkgs",
}

def registrar_error(error):
    errores_lectura.append(str(error))

for raiz in rutas_disponibles:
    for carpeta, subcarpetas, archivos in os.walk(
        raiz,
        onerror=registrar_error
    ):
        subcarpetas[:] = [
            nombre
            for nombre in subcarpetas
            if nombre.lower() not in omitir
        ]

        for nombre in archivos:
            ruta = Path(carpeta) / nombre
            nombre_minuscula = nombre.lower()

            if ruta.suffix.lower() in {".pt", ".pth", ".ckpt"}:
                pesos_encontrados.add(str(ruta))

            elif (
                ruta.suffix.lower() == ".json"
                and any(
                    palabra in nombre_minuscula
                    for palabra in (
                        "config",
                        "metadata",
                        "class",
                        "clase",
                        "mapping",
                        "normal",
                    )
                )
            ):
                configuraciones_encontradas.add(str(ruta))


def mostrar_archivos(titulo, archivos, limite=60):
    ordenados = sorted(archivos)

    print(f"\n{titulo}: {len(ordenados)}")

    for numero, archivo in enumerate(ordenados[:limite], start=1):
        print(f"{numero}. {archivo}")

    if len(ordenados) > limite:
        print(
            f"Se muestran los primeros {limite}. "
            "La lista completa queda disponible en las variables."
        )


mostrar_archivos("ARCHIVOS DE PESOS", pesos_encontrados)

mostrar_archivos(
    "CONFIGURACIONES Y METADATOS",
    configuraciones_encontradas
)

if errores_lectura:
    print("\nCarpetas que no pudieron consultarse:")
    for error in errores_lectura[:10]:
        print(error)

if not rutas_disponibles:
    print(
        "\nNo se encontraron las carpetas indicadas. "
        "El siguiente paso será ajustar las rutas."
    )
elif not pesos_encontrados:
    print(
        "\nNo se localizaron pesos con extensiones .pt, .pth o .ckpt "
        "en las carpetas consultadas."
    )

print("\nREVISIÓN TERMINADA. No se modificó ningún archivo.")

1. ENTORNO DEL CUADERNO
Python: 3.11.16
Ejecutable: C:\Users\Ivan sua\anaconda3\envs\CLC-SEGMENTACION\python.exe
Sistema: Windows-10-10.0.26200-SP0
Carpeta del cuaderno: C:\Users\Ivan sua

Versiones de las bibliotecas:
  numpy: 1.26.4
  pandas: 2.2.3
  rasterio: 1.4.4
  torch: 2.8.0+cu126
  torchvision: 0.23.0+cu126
  scikit-learn: 1.9.1
  openpyxl: 3.1.5

2. CARPETAS DISPONIBLES
ENCONTRADA: D:\PROYECTO_GRADO_1
ENCONTRADA: C:\Users\Ivan sua\anaconda3\Modelos_UNET
ENCONTRADA: C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion
ENCONTRADA: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion

3. ARCHIVOS DE PESOS Y CONFIGURACIONES

ARCHIVOS DE PESOS: 14
1. C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_1_20260815_122755\best_model.pt
2. C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_1_20260815_122755\checkpoints\last_checkpoint.pt
3. C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121

In [2]:
# ============================================================
# CELDA 2A — CORREGIR LA FUENTE DEL DATASET
# Localiza metadatos del dataset original.
# No modifica el dataset ni carga modelos.
# ============================================================

import os
import json
from pathlib import Path
from datetime import datetime

DATASET_ORIGINAL = Path(
    r"C:\Users\Ivan sua\anaconda3\Dataset"
)

if not DATASET_ORIGINAL.is_dir():
    raise FileNotFoundError(
        f"No se encontró el dataset original: {DATASET_ORIGINAL}"
    )

print("=" * 75)
print("DATASET ORIGINAL CONFIRMADO POR EL USUARIO")
print(DATASET_ORIGINAL)
print("=" * 75)

print("\nContenido del primer nivel:")

for ruta in sorted(
    DATASET_ORIGINAL.iterdir(),
    key=lambda p: p.name.lower()
):
    tipo = "CARPETA" if ruta.is_dir() else "ARCHIVO"
    print(f"[{tipo}] {ruta.name}")


# Buscar archivos pequeños de configuración, clases y particiones.
# No se leen imágenes, máscaras ni pesos.

METADATOS_DATASET_ORIGINAL = []
errores_busqueda = []

def registrar_error(error):
    errores_busqueda.append(str(error))

for carpeta, subcarpetas, archivos in os.walk(
    DATASET_ORIGINAL,
    onerror=registrar_error
):
    subcarpetas[:] = [
        nombre for nombre in subcarpetas
        if nombre.lower() not in {
            ".git", "__pycache__", ".ipynb_checkpoints"
        }
    ]

    for nombre in archivos:
        ruta = Path(carpeta) / nombre
        extension = ruta.suffix.lower()
        nombre_minuscula = nombre.lower()

        es_json = extension == ".json"

        es_particion = (
            extension in {".csv", ".txt"}
            and any(
                palabra in nombre_minuscula
                for palabra in (
                    "train", "val", "test", "split",
                    "partition", "particion", "manifest"
                )
            )
        )

        if es_json or es_particion:
            METADATOS_DATASET_ORIGINAL.append(str(ruta))

METADATOS_DATASET_ORIGINAL.sort()

print("\n" + "=" * 75)
print("METADATOS Y POSIBLES ARCHIVOS DE PARTICIÓN")
print("=" * 75)

for numero, ruta in enumerate(
    METADATOS_DATASET_ORIGINAL,
    start=1
):
    print(f"{numero}. {ruta}")

print(
    "\nTotal de archivos encontrados:",
    len(METADATOS_DATASET_ORIGINAL)
)

if errores_busqueda:
    print("\nErrores de acceso:")
    for error in errores_busqueda:
        print(error)


# Registrar la rectificación sin borrar la inspección anterior.

if "CARPETA_AUDITORIA" not in globals():
    identificador = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    CARPETA_AUDITORIA = (
        Path(r"D:\PROYECTO_GRADO_1")
        / "Auditoria_comparacion_modelos"
        / identificador
    )
    CARPETA_AUDITORIA.mkdir(parents=True, exist_ok=False)

rectificacion = {
    "fecha_local": datetime.now().astimezone().isoformat(),
    "celda": "2A",
    "correccion": "Identificación de la fuente original del dataset",
    "dataset_original": str(DATASET_ORIGINAL),
    "fuente_descartada": (
        r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Dataset_CLC_FINAL_V3"
    ),
    "motivo": (
        "El usuario confirmó que V3 no corresponde al dataset original."
    ),
    "archivos_localizados": METADATOS_DATASET_ORIGINAL,
    "errores_busqueda": errores_busqueda,
    "pendiente": (
        "Leer los metadatos originales y comprobar su relación "
        "con las configuraciones de los checkpoints."
    ),
    "modificaciones_al_dataset": "Ninguna",
}

fecha = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
archivo_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_02A_rectificacion_dataset_{fecha}.json"
)

with archivo_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        rectificacion,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("\nRectificación registrada en:")
print(archivo_registro)

DATASET ORIGINAL CONFIRMADO POR EL USUARIO
C:\Users\Ivan sua\anaconda3\Dataset

Contenido del primer nivel:
[CARPETA] metadata
[CARPETA] Nivel_1
[CARPETA] Nivel_2
[CARPETA] Nubes

METADATOS Y POSIBLES ARCHIVOS DE PARTICIÓN
1. C:\Users\Ivan sua\anaconda3\Dataset\metadata\reporte_dataset\resumen_splits.csv
2. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\global\test_samples.csv
3. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\global\train_samples.csv
4. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\global\val_samples.csv
5. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\nivel_1\test_samples.csv
6. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\nivel_1\train_samples.csv
7. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\nivel_1\val_samples.csv
8. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\nivel_2\test_samples.csv
9. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\nivel_2\train_samples.csv
10. C:\Users\Ivan sua\anaconda3\Dataset\metadata\splits\

In [3]:
# ============================================================
# CELDA 3 — CONFIGURACIONES DE LOS SEIS MODELOS Y PARTICIONES
#
# Lee los config.json asociados a los modelos.
# Muestra columnas, número de registros y ejemplos de los CSV.
# No carga pesos, no modifica el dataset y no hace predicciones.
# ============================================================

import json
import csv
import hashlib
from pathlib import Path
from datetime import datetime


# ------------------------------------------------------------
# 1. RUTAS ORIGINALES
# ------------------------------------------------------------

BASE_ORIGINAL = Path(r"C:\Users\Ivan sua\anaconda3")
DATASET_ORIGINAL = BASE_ORIGINAL / "Dataset"

# Diccionario nuevo: no reutiliza las rutas de V3 de la celda 2.
CARPETAS_MODELOS_ORIGINALES = {
    "UNET_N1":
        BASE_ORIGINAL / "Modelos_UNET" / "Nivel 1",

    "UNET_N2":
        BASE_ORIGINAL / "Modelos_UNET" / "Nivel 2",

    "RESNET_N1":
        BASE_ORIGINAL / "Modelos_ResNet50_Segmentacion"
        / "ResNet50Seg_nivel_1_20260730_191354",

    "RESNET_N2":
        BASE_ORIGINAL / "Modelos_ResNet50_Segmentacion"
        / "ResNet50Seg_nivel_2_20260803_201945",

    "DENSENET_N1":
        BASE_ORIGINAL / "Modelos_DenseNet121_Segmentacion"
        / "DenseNet121Seg_nivel_1_20260815_122755",

    "DENSENET_N2":
        BASE_ORIGINAL / "Modelos_DenseNet121_Segmentacion"
        / "DenseNet121Seg_nivel_2_20260815_172335",
}

if "CARPETA_AUDITORIA" not in globals():
    CARPETA_AUDITORIA = Path(
        r"D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos"
        r"\20261007_191328_200990"
    )

CARPETA_AUDITORIA = Path(CARPETA_AUDITORIA)

if not CARPETA_AUDITORIA.is_dir():
    raise FileNotFoundError(
        "No se encontró la carpeta de auditoría. "
        "Ejecuta nuevamente la celda 2A."
    )


# ------------------------------------------------------------
# 2. CONFIGURACIONES ASOCIADAS A LOS MODELOS
# ------------------------------------------------------------

CONFIGS_MODELOS_ORIGINALES = {}

for modelo, carpeta in CARPETAS_MODELOS_ORIGINALES.items():

    ruta_config = carpeta / "config.json"
    ruta_pesos = carpeta / "best_model.pt"

    print("\n" + "=" * 78)
    print(modelo)
    print("Configuración:", ruta_config)
    print("Existe best_model.pt:", ruta_pesos.is_file())
    print("=" * 78)

    resultado = {
        "ruta_config": str(ruta_config),
        "ruta_pesos": str(ruta_pesos),
        "existe_pesos": ruta_pesos.is_file(),
    }

    try:
        datos_bytes = ruta_config.read_bytes()
        contenido = json.loads(datos_bytes.decode("utf-8-sig"))

        resultado["sha256_config"] = hashlib.sha256(
            datos_bytes
        ).hexdigest()

        resultado["contenido"] = contenido

        # Se muestra completo para no omitir parámetros relevantes.
        print(json.dumps(
            contenido,
            ensure_ascii=False,
            indent=2
        ))

    except Exception as error:
        resultado["error"] = f"{type(error).__name__}: {error}"
        print("ERROR:", resultado["error"])

    CONFIGS_MODELOS_ORIGINALES[modelo] = resultado


# ------------------------------------------------------------
# 3. INSPECCIONAR LOS CSV DE PARTICIONES
# Se cuentan registros y se conservan tres ejemplos por archivo.
# No se presupone el nombre de las columnas.
# ------------------------------------------------------------

ARCHIVOS_PARTICIONES = {
    f"{grupo}_{particion}":
        DATASET_ORIGINAL / "metadata" / "splits"
        / grupo / f"{particion}_samples.csv"

    for grupo in ("global", "nivel_1", "nivel_2", "nubes")
    for particion in ("train", "val", "test")
}

ARCHIVOS_PARTICIONES["resumen_splits"] = (
    DATASET_ORIGINAL / "metadata"
    / "reporte_dataset" / "resumen_splits.csv"
)

RESUMEN_PARTICIONES_ORIGINALES = {}

for identificador, ruta in ARCHIVOS_PARTICIONES.items():

    print("\n" + "-" * 78)
    print("PARTICIÓN:", identificador)
    print("Archivo:", ruta)

    resultado = {"ruta": str(ruta)}

    try:
        with ruta.open(
            "r",
            encoding="utf-8-sig",
            newline=""
        ) as archivo:

            muestra = archivo.read(16384)
            archivo.seek(0)

            if not muestra.strip():
                raise ValueError("El archivo está vacío.")

            try:
                dialecto = csv.Sniffer().sniff(
                    muestra,
                    delimiters=",;\t"
                )
                separador = dialecto.delimiter
                deteccion = "Detectado"
            except csv.Error:
                separador = ","
                deteccion = (
                    "No se pudo detectar; se usó coma. "
                    "Revisar las columnas mostradas."
                )

            lector = csv.reader(
                archivo,
                delimiter=separador
            )

            encabezado = next(lector)
            cantidad = 0
            ejemplos = []
            filas_con_longitud_distinta = 0

            for fila in lector:
                if not fila or not any(valor.strip() for valor in fila):
                    continue

                cantidad += 1

                if len(fila) != len(encabezado):
                    filas_con_longitud_distinta += 1

                if len(ejemplos) < 3:
                    ejemplos.append(fila)

        resultado.update({
            "separador": separador,
            "deteccion_separador": deteccion,
            "columnas": encabezado,
            "numero_registros": cantidad,
            "primeros_tres_registros": ejemplos,
            "filas_con_longitud_distinta": filas_con_longitud_distinta,
        })

        print("Separador:", repr(separador))
        print("Detección:", deteccion)
        print("Columnas:", encabezado)
        print("Número de registros:", cantidad)
        print(
            "Filas con diferente número de campos:",
            filas_con_longitud_distinta
        )

        print("Ejemplos — valores en el orden de las columnas:")
        for numero, fila in enumerate(ejemplos, start=1):
            print(
                f"  {numero}:",
                json.dumps(fila, ensure_ascii=False)
            )

    except Exception as error:
        resultado["error"] = f"{type(error).__name__}: {error}"
        print("ERROR:", resultado["error"])

    RESUMEN_PARTICIONES_ORIGINALES[identificador] = resultado


# ------------------------------------------------------------
# 4. GUARDAR EVIDENCIA PARA EL REPORTE FINAL
# ------------------------------------------------------------

registro_celda3 = {
    "fecha_local": datetime.now().astimezone().isoformat(),
    "celda": 3,

    "dataset_original": str(DATASET_ORIGINAL),

    "objetivo": (
        "Inspeccionar configuraciones asociadas a los seis modelos "
        "y estructura de las particiones del dataset original."
    ),

    "configuraciones_modelos": CONFIGS_MODELOS_ORIGINALES,
    "particiones": RESUMEN_PARTICIONES_ORIGINALES,

    "alcance": (
        "Se verificó lectura y estructura documental. "
        "Todavía no se ha comprobado que las configuraciones coincidan "
        "con los metadatos internos de los checkpoints, ni la "
        "independencia espacial de las particiones."
    ),

    "fuente_V3": "Excluida por rectificación del usuario",
    "cambios_en_dataset_modelos_predicciones": "Ninguno",
}

fecha = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ARCHIVO_REVISION_CELDA3 = (
    CARPETA_AUDITORIA
    / f"celda_03_configuraciones_y_particiones_{fecha}.json"
)

with ARCHIVO_REVISION_CELDA3.open(
    "x",
    encoding="utf-8"
) as archivo:
    json.dump(
        registro_celda3,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("\n" + "=" * 78)
print("CELDA 3 TERMINADA")
print("=" * 78)
print("Registro guardado en:")
print(ARCHIVO_REVISION_CELDA3)


UNET_N1
Configuración: C:\Users\Ivan sua\anaconda3\Modelos_UNET\Nivel 1\config.json
Existe best_model.pt: True
{
  "TRAIN_LEVEL": "nivel_1",
  "DATASET_ROOT": "D:\\Dataset",
  "METADATA_DIR": "D:\\Dataset\\metadata",
  "SPLITS_DIR": "D:\\Dataset\\metadata\\splits\\nivel_1",
  "RUNS_DIR": "D:\\Modelos_UNet",
  "RUN_DIR": "D:\\Modelos_UNet\\UNetV2_nivel_1_20260703_151727",
  "CHECKPOINT_DIR": "D:\\Modelos_UNet\\UNetV2_nivel_1_20260703_151727\\checkpoints",
  "REPORT_DIR": "D:\\Modelos_UNet\\UNetV2_nivel_1_20260703_151727\\reports",
  "NUM_CLASSES": 6,
  "CLASS_DICT": {
    "0": "NoData",
    "1": "2. Territorios Agricolas",
    "2": "3. Bosques y Areas Seminaturales",
    "3": "5. Superficies de Agua",
    "4": "1. Territorios Artificializados",
    "5": "4. Areas Humedas"
  },
  "SEED": 42,
  "EPOCHS": 120,
  "BATCH_SIZE": 1,
  "LEARNING_RATE": 0.0003,
  "MOMENTUM": 0.99,
  "WEIGHT_DECAY": 0.0001,
  "OPTIMIZER_TYPE": "adamw_moderno",
  "BASE_CHANNELS": 64,
  "BILINEAR": false,
  "DROPO

In [4]:
# ============================================================
# CELDA 4 — INSPECCIÓN DE LOS SEIS CHECKPOINTS
#
# Consulta estructura, metadatos y dimensiones de los tensores.
# No ejecuta inferencia ni modifica los modelos.
# Requiere haber ejecutado la celda 3.
# ============================================================

import gc
import json
import math
from pathlib import Path
from datetime import datetime
from collections.abc import Mapping

import torch


# ------------------------------------------------------------
# 1. COMPROBAR VARIABLES DE LA CELDA ANTERIOR
# ------------------------------------------------------------

requeridas = [
    "CARPETAS_MODELOS_ORIGINALES",
    "CONFIGS_MODELOS_ORIGINALES",
    "CARPETA_AUDITORIA",
]

faltantes = [
    nombre for nombre in requeridas
    if nombre not in globals()
]

if faltantes:
    raise RuntimeError(
        "Ejecuta primero la celda 3. Faltan: "
        + ", ".join(faltantes)
    )

print("PyTorch:", torch.__version__)
print("Dispositivo de lectura: CPU")
print("No se ejecutarán predicciones.")


# ------------------------------------------------------------
# 2. FUNCIONES DE INSPECCIÓN
# ------------------------------------------------------------

def resumir_valor(valor):
    """Convierte metadatos a JSON sin exportar matrices de pesos."""

    if isinstance(valor, torch.Tensor):
        if valor.numel() <= 32:
            return resumir_valor(valor.detach().cpu().tolist())

        return {
            "tipo": "Tensor",
            "dimensiones": list(valor.shape),
            "dtype": str(valor.dtype),
        }

    if isinstance(valor, Mapping):
        return {
            str(clave): resumir_valor(contenido)
            for clave, contenido in valor.items()
        }

    if isinstance(valor, (list, tuple)):
        return [resumir_valor(elemento) for elemento in valor]

    if isinstance(valor, Path):
        return str(valor)

    if isinstance(valor, float) and not math.isfinite(valor):
        return str(valor)

    if valor is None or isinstance(valor, (str, int, float, bool)):
        return valor

    return {"tipo_no_convertido": type(valor).__name__}


def es_diccionario_tensores(objeto):
    return (
        isinstance(objeto, Mapping)
        and len(objeto) > 0
        and all(
            isinstance(valor, torch.Tensor)
            for valor in objeto.values()
        )
    )


def localizar_pesos(checkpoint):
    """Reconoce formatos habituales sin asumir una arquitectura."""

    if es_diccionario_tensores(checkpoint):
        return "raiz", checkpoint

    if isinstance(checkpoint, Mapping):
        for clave in (
            "model_state_dict",
            "state_dict",
            "model",
            "net",
            "network",
        ):
            candidato = checkpoint.get(clave)

            if es_diccionario_tensores(candidato):
                return clave, candidato

    raise ValueError(
        "No se encontró un diccionario de tensores "
        "en los formatos inspeccionados."
    )


# Metadatos que interesa recuperar, si están presentes.
CLAVES_METADATOS = {
    "config", "configuration", "training_config",
    "hyperparameters", "hparams",
    "epoch", "best_epoch", "best_value",
    "best_val_iou", "best_val_loss", "best_val_f1",
    "validation_metrics", "val_metrics",
    "class_dict", "class_mapping",
    "code_to_idx", "idx_to_code", "idx_to_name",
    "class_to_idx", "num_classes", "num_channels",
    "means", "stds", "band_indexes", "band_order",
    "ignore_index", "scale_divisor", "input_scale_divisor",
    "reflectance_scale", "reflectance_add_offset",
    "saved_at", "checkpoint_type",
}


# ------------------------------------------------------------
# 3. INSPECCIONAR UN CHECKPOINT A LA VEZ
# ------------------------------------------------------------

REVISION_CHECKPOINTS = {}

for modelo, carpeta in CARPETAS_MODELOS_ORIGINALES.items():

    ruta = Path(carpeta) / "best_model.pt"

    print("\n" + "=" * 78)
    print(modelo)
    print("Archivo:", ruta)
    print("=" * 78)

    resultado = {"ruta": str(ruta)}
    checkpoint = None
    estado = None

    try:
        resultado["tamano_bytes"] = ruta.stat().st_size

        checkpoint = torch.load(
            ruta,
            map_location="cpu",
            weights_only=True,
        )

        resultado["tipo_contenedor"] = type(checkpoint).__name__

        ubicacion, estado = localizar_pesos(checkpoint)
        resultado["ubicacion_pesos"] = ubicacion

        if ubicacion != "raiz":
            resultado["claves_principales"] = [
                str(clave) for clave in checkpoint.keys()
            ]

            metadatos = {
                str(clave): resumir_valor(valor)
                for clave, valor in checkpoint.items()
                if str(clave).lower() in CLAVES_METADATOS
            }
        else:
            resultado["claves_principales"] = (
                "El archivo contiene directamente los tensores."
            )
            metadatos = {}

        resultado["metadatos"] = metadatos

        # Se guardan nombres y dimensiones, no los valores de los pesos.
        dimensiones = {
            str(nombre): list(tensor.shape)
            for nombre, tensor in estado.items()
        }

        resultado["dimensiones_tensores"] = dimensiones
        resultado["numero_tensores"] = len(dimensiones)

        print("Ubicación de pesos:", ubicacion)
        print("Número de tensores:", len(dimensiones))
        print("Claves principales:", resultado["claves_principales"])

        print("\nMetadatos recuperados:")
        print(json.dumps(metadatos, ensure_ascii=False, indent=2))

        nombres = list(dimensiones)

        # Primeros y últimos tensores, más posibles capas de salida.
        seleccion = list(dict.fromkeys(
            nombres[:4]
            + nombres[-6:]
            + [
                nombre for nombre in nombres
                if any(
                    termino in nombre.lower()
                    for termino in (
                        "outc", "outconv", "segmentation_head",
                        "classifier", "final_conv"
                    )
                )
            ]
        ))

        print("\nTensores seleccionados — nombre y dimensiones:")
        for nombre in seleccion:
            print(f"  {nombre}: {dimensiones[nombre]}")

        # Comparación documental exacta cuando existe config interna.
        config_externa = CONFIGS_MODELOS_ORIGINALES[
            modelo
        ].get("contenido", {})

        config_interna = None

        if ubicacion != "raiz":
            for clave in ("config", "configuration", "training_config"):
                candidato = checkpoint.get(clave)
                if isinstance(candidato, Mapping):
                    config_interna = resumir_valor(candidato)
                    break

        if (
            config_interna is not None
            and isinstance(config_externa, dict)
        ):
            comunes = sorted(
                set(config_externa) & set(config_interna)
            )

            diferencias = {
                clave: {
                    "config_json": config_externa[clave],
                    "checkpoint": config_interna[clave],
                }
                for clave in comunes
                if config_externa[clave] != config_interna[clave]
            }

            resultado["comparacion_config"] = {
                "claves_comunes": comunes,
                "diferencias": diferencias,
                "solo_config_json": sorted(
                    set(config_externa) - set(config_interna)
                ),
                "solo_checkpoint": sorted(
                    set(config_interna) - set(config_externa)
                ),
            }

            print("\nDiferencias en claves comunes de configuración:")
            print(json.dumps(
                diferencias,
                ensure_ascii=False,
                indent=2
            ))

        else:
            resultado["comparacion_config"] = {
                "estado": (
                    "No se pudo comparar una configuración interna "
                    "con el JSON externo."
                )
            }
            print("\nSin configuración interna comparable.")

        resultado["estado"] = (
            "Lectura completada; pendiente interpretar "
            "la correspondencia semántica y la arquitectura."
        )

    except Exception as error:
        resultado["estado"] = "Lectura no completada"
        resultado["error"] = f"{type(error).__name__}: {error}"

        print("\nERROR:")
        print(resultado["error"])
        print(
            "\nSe conserva el error para revisarlo. "
            "No se cambia automáticamente el modo de carga."
        )

    finally:
        checkpoint = None
        estado = None
        # Evitar conservar referencias auxiliares al checkpoint.
        candidato = None
        gc.collect()

    REVISION_CHECKPOINTS[modelo] = resultado


# ------------------------------------------------------------
# 4. GUARDAR EL REGISTRO DE ESTA CELDA
# ------------------------------------------------------------

registro = {
    "fecha_local": datetime.now().astimezone().isoformat(),
    "celda": 4,
    "torch_version": str(torch.__version__),
    "modo_lectura": {
        "map_location": "cpu",
        "weights_only": True,
    },
    "checkpoints": REVISION_CHECKPOINTS,
    "pendientes": [
        "Confirmar codificación de las máscaras de entrenamiento U-Net.",
        "Recuperar la transformación radiométrica original de U-Net.",
        "Verificar rutas locales e independencia espacial de particiones.",
        "Comprobar referencias y universo común de evaluación.",
    ],
    "modificaciones_a_modelos_o_datos": "Ninguna",
}

fecha = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ARCHIVO_REVISION_CELDA4 = (
    Path(CARPETA_AUDITORIA)
    / f"celda_04_checkpoints_{fecha}.json"
)

with ARCHIVO_REVISION_CELDA4.open(
    "x",
    encoding="utf-8"
) as archivo:
    json.dump(
        registro,
        archivo,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("\n" + "=" * 78)
print("CELDA 4 TERMINADA")
print("Registro guardado en:")
print(ARCHIVO_REVISION_CELDA4)

PyTorch: 2.8.0+cu126
Dispositivo de lectura: CPU
No se ejecutarán predicciones.

UNET_N1
Archivo: C:\Users\Ivan sua\anaconda3\Modelos_UNET\Nivel 1\best_model.pt
Ubicación de pesos: model_state_dict
Número de tensores: 46
Claves principales: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'scheduler_state_dict', 'best_val_iou', 'best_epoch', 'no_improve_epochs', 'config']

Metadatos recuperados:
{
  "epoch": 120,
  "best_val_iou": 0.5941851050740058,
  "best_epoch": 120,
  "config": {
    "TRAIN_LEVEL": "nivel_1",
    "DATASET_ROOT": "D:\\Dataset",
    "METADATA_DIR": "D:\\Dataset\\metadata",
    "SPLITS_DIR": "D:\\Dataset\\metadata\\splits\\nivel_1",
    "RUNS_DIR": "D:\\Modelos_UNet",
    "RUN_DIR": "D:\\Modelos_UNet\\UNetV2_nivel_1_20260703_151727",
    "CHECKPOINT_DIR": "D:\\Modelos_UNet\\UNetV2_nivel_1_20260703_151727\\checkpoints",
    "REPORT_DIR": "D:\\Modelos_UNet\\UNetV2_nivel_1_20260703_151727\\reports",
    "NUM_CLASSES": 6,
    "CLASS_DICT": {
   

In [5]:
# ============================================================
# CELDA 5 — RUTAS LOCALES Y REPETICIONES ENTRE PARTICIONES
#
# Resuelve las rutas antiguas contra el dataset original.
# Comprueba existencia de imágenes y máscaras.
# Detecta coincidencias de identificadores y rutas entre splits.
#
# No modifica los CSV, no elimina muestras y no abre los TIFF.
# ============================================================

import csv
import json
import os
from pathlib import Path
from collections import Counter
from itertools import combinations
from datetime import datetime


DATASET_ORIGINAL = Path(
    r"C:\Users\Ivan sua\anaconda3\Dataset"
).resolve()

if not DATASET_ORIGINAL.is_dir():
    raise FileNotFoundError(DATASET_ORIGINAL)

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError(
        "Ejecuta las celdas anteriores para recuperar "
        "la carpeta de auditoría."
    )


# ------------------------------------------------------------
# 1. RESOLVER RUTAS SIN BUSCAR ARCHIVOS POR NOMBRES PARECIDOS
# ------------------------------------------------------------

def resolver_ruta_dataset(texto):
    texto = str(texto).strip()

    if not texto:
        raise ValueError("Ruta vacía.")

    partes = texto.replace("\\", "/").split("/")

    posiciones = [
        i for i, parte in enumerate(partes)
        if parte.casefold() == "dataset"
    ]

    if len(posiciones) != 1:
        raise ValueError(
            "No se identificó un único segmento 'Dataset': "
            + texto
        )

    relativas = partes[posiciones[0] + 1:]

    if not relativas or any(
        parte in {"", ".", ".."} for parte in relativas
    ):
        raise ValueError("Estructura de ruta no admitida: " + texto)

    ruta_local = DATASET_ORIGINAL.joinpath(*relativas).resolve()

    # Mantener la consulta dentro del dataset original.
    ruta_local.relative_to(DATASET_ORIGINAL)

    return ruta_local


def clave_ruta(ruta):
    return os.path.normcase(os.path.normpath(str(ruta)))


# ------------------------------------------------------------
# 2. LEER LAS SEIS PARTICIONES TEMÁTICAS
# ------------------------------------------------------------

FILAS_LOCALES = {}
AUDITORIA_RUTAS = {}

for nivel in ("nivel_1", "nivel_2"):
    for particion in ("train", "val", "test"):

        identificador = f"{nivel}_{particion}"

        ruta_csv = (
            DATASET_ORIGINAL / "metadata" / "splits"
            / nivel / f"{particion}_samples.csv"
        )

        print("\n" + "=" * 75)
        print(identificador)
        print("=" * 75)

        filas_locales = []
        incidencias = []

        with ruta_csv.open(
            "r", encoding="utf-8-sig", newline=""
        ) as archivo:

            lector = csv.DictReader(archivo)

            requeridas = {
                "level", "chip_id", "image_path", "mask_path"
            }

            if not requeridas.issubset(lector.fieldnames or []):
                raise ValueError(
                    f"Columnas insuficientes en {ruta_csv}"
                )

            for numero, fila in enumerate(lector, start=2):

                fila_local = dict(fila)
                fila_local["fila_csv"] = numero

                if fila["level"].strip().casefold() != nivel:
                    incidencias.append({
                        "fila_csv": numero,
                        "problema": "Nivel distinto al esperado",
                        "valor": fila["level"],
                    })

                for campo in ("image_path", "mask_path"):
                    try:
                        ruta_local = resolver_ruta_dataset(fila[campo])
                        existe = ruta_local.is_file()

                        fila_local[campo + "_local"] = str(ruta_local)
                        fila_local[campo + "_existe"] = existe

                        if not existe:
                            incidencias.append({
                                "fila_csv": numero,
                                "campo": campo,
                                "problema": "Archivo no encontrado",
                                "ruta_local": str(ruta_local),
                            })

                    except Exception as error:
                        fila_local[campo + "_local"] = None
                        fila_local[campo + "_existe"] = False

                        incidencias.append({
                            "fila_csv": numero,
                            "campo": campo,
                            "problema": str(error),
                            "ruta_original": fila[campo],
                        })

                filas_locales.append(fila_local)

        FILAS_LOCALES[identificador] = filas_locales

        conteo_ids = Counter(
            fila["chip_id"].strip()
            for fila in filas_locales
            if fila["chip_id"].strip()
        )

        ids_repetidos = {
            chip: cantidad
            for chip, cantidad in conteo_ids.items()
            if cantidad > 1
        }

        resumen = {
            "csv_original": str(ruta_csv),
            "registros": len(filas_locales),
            "imagenes_existentes": sum(
                fila["image_path_existe"] for fila in filas_locales
            ),
            "mascaras_existentes": sum(
                fila["mask_path_existe"] for fila in filas_locales
            ),
            "ids_vacios": sum(
                not fila["chip_id"].strip() for fila in filas_locales
            ),
            "ids_repetidos_dentro_particion": ids_repetidos,
            "incidencias": incidencias,
        }

        AUDITORIA_RUTAS[identificador] = resumen

        print("Registros:", resumen["registros"])
        print("Imágenes encontradas:", resumen["imagenes_existentes"])
        print("Máscaras encontradas:", resumen["mascaras_existentes"])
        print("Identificadores vacíos:", resumen["ids_vacios"])
        print("Identificadores repetidos:", len(ids_repetidos))
        print("Incidencias:", len(incidencias))

        for incidencia in incidencias[:5]:
            print("  ", incidencia)


# ------------------------------------------------------------
# 3. COMPROBAR COINCIDENCIAS ENTRE PARTICIONES DEL MISMO NIVEL
# No equivale todavía a comprobar independencia espacial.
# ------------------------------------------------------------

CRUCES_PARTICIONES = {}

def valores_comparables(filas, campo):
    if campo == "chip_id":
        return {
            fila[campo].strip()
            for fila in filas
            if fila[campo].strip()
        }

    return {
        clave_ruta(fila[campo])
        for fila in filas
        if fila.get(campo)
    }


print("\n" + "=" * 75)
print("COINCIDENCIAS ENTRE PARTICIONES")
print("=" * 75)

for nivel in ("nivel_1", "nivel_2"):
    for primera, segunda in combinations(
        ("train", "val", "test"), 2
    ):
        nombre_cruce = f"{nivel}: {primera} / {segunda}"

        filas_a = FILAS_LOCALES[f"{nivel}_{primera}"]
        filas_b = FILAS_LOCALES[f"{nivel}_{segunda}"]

        resultado = {}

        for campo in (
            "chip_id",
            "image_path_local",
            "mask_path_local",
        ):
            coincidencias = sorted(
                valores_comparables(filas_a, campo)
                & valores_comparables(filas_b, campo)
            )

            resultado[campo] = {
                "cantidad": len(coincidencias),
                "coincidencias": coincidencias,
            }

        CRUCES_PARTICIONES[nombre_cruce] = resultado

        print("\n", nombre_cruce)
        for campo, detalle in resultado.items():
            print(f"  {campo}: {detalle['cantidad']} coincidencias")
            for ejemplo in detalle["coincidencias"][:3]:
                print("    ", ejemplo)


# ------------------------------------------------------------
# 4. GUARDAR REGISTRO SIN ALTERAR LOS CSV ORIGINALES
# ------------------------------------------------------------

registro = {
    "fecha_local": datetime.now().astimezone().isoformat(),
    "celda": 5,
    "dataset_original": str(DATASET_ORIGINAL),

    "accion": (
        "Resolución de rutas antiguas en memoria, comprobación "
        "de existencia y detección de coincidencias documentales."
    ),

    "resumen_particiones": AUDITORIA_RUTAS,
    "cruces_entre_particiones": CRUCES_PARTICIONES,
    "filas_con_rutas_locales": FILAS_LOCALES,

    "limites": [
        "Existencia de archivo no demuestra identidad con el original.",
        "No se han leído todavía las imágenes ni las máscaras.",
        "Ausencia de IDs repetidos no demuestra independencia espacial.",
        "No se han eliminado ni redistribuido muestras.",
    ],

    "pendientes": [
        "Revisar georreferenciación y solapamientos de los chips.",
        "Comprobar correspondencia entre particiones globales y por nivel.",
        "Recuperar el código de entrenamiento original de U-Net.",
    ],

    "archivos_originales_modificados": [],
}

fecha = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ARCHIVO_REVISION_CELDA5 = (
    Path(CARPETA_AUDITORIA)
    / f"celda_05_rutas_y_particiones_{fecha}.json"
)

with ARCHIVO_REVISION_CELDA5.open(
    "x", encoding="utf-8"
) as archivo:
    json.dump(
        registro,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("\nRegistro guardado en:")
print(ARCHIVO_REVISION_CELDA5)
print("\nCELDA 5 TERMINADA. Los CSV originales no se modificaron.")


nivel_1_train
Registros: 1017
Imágenes encontradas: 1017
Máscaras encontradas: 1017
Identificadores vacíos: 0
Identificadores repetidos: 0
Incidencias: 0

nivel_1_val
Registros: 218
Imágenes encontradas: 218
Máscaras encontradas: 218
Identificadores vacíos: 0
Identificadores repetidos: 0
Incidencias: 0

nivel_1_test
Registros: 219
Imágenes encontradas: 219
Máscaras encontradas: 219
Identificadores vacíos: 0
Identificadores repetidos: 0
Incidencias: 0

nivel_2_train
Registros: 3727
Imágenes encontradas: 3727
Máscaras encontradas: 3727
Identificadores vacíos: 0
Identificadores repetidos: 0
Incidencias: 0

nivel_2_val
Registros: 799
Imágenes encontradas: 799
Máscaras encontradas: 799
Identificadores vacíos: 0
Identificadores repetidos: 0
Incidencias: 0

nivel_2_test
Registros: 799
Imágenes encontradas: 799
Máscaras encontradas: 799
Identificadores vacíos: 0
Identificadores repetidos: 0
Incidencias: 0

COINCIDENCIAS ENTRE PARTICIONES

 nivel_1: train / val
  chip_id: 0 coincidencias
  ima

In [6]:
# ============================================================
# CELDA 6 — GEOMETRÍA, BANDAS Y CÓDIGOS DEL DATASET ORIGINAL
# ============================================================

from pathlib import Path
from collections import Counter
from datetime import datetime
import json

import numpy as np
import rasterio


if "FILAS_LOCALES" not in globals():
    raise RuntimeError("Primero ejecuta la celda 5.")

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")


def metadatos_raster(src):
    return {
        "ancho": src.width,
        "alto": src.height,
        "bandas": src.count,
        "tipos": list(src.dtypes),
        "crs": src.crs.to_string() if src.crs else None,
        "transformacion": list(src.transform)[:6],
        "limites": list(src.bounds),
        "resolucion": list(src.res),
        "nodata": str(src.nodata),
        "descripciones": list(src.descriptions),
        "escalas": list(src.scales),
        "offsets": list(src.offsets),
    }


REVISION_RASTERS = {}
RESUMEN_RASTERS = {}

for particion, filas in FILAS_LOCALES.items():

    print("\n" + "=" * 75)
    print(particion)
    print("=" * 75)

    registros = []
    incidencias = []

    dimensiones = Counter()
    bandas = Counter()
    sistemas = Counter()

    # Todos los valores almacenados, incluidos los píxeles enmascarados.
    codigos_totales = Counter()

    # Valores que la máscara interna/NoData del TIFF considera válidos.
    codigos_validos_tiff = Counter()

    pixeles_invalidos_tiff = 0
    pares_alineados = 0

    for numero, fila in enumerate(filas, start=1):

        registro = {
            "chip_id": fila["chip_id"],
            "image_path_local": fila["image_path_local"],
            "mask_path_local": fila["mask_path_local"],
        }

        problemas = []

        try:
            with rasterio.open(fila["image_path_local"]) as img, \
                 rasterio.open(fila["mask_path_local"]) as msk:

                registro["imagen"] = metadatos_raster(img)
                registro["mascara"] = metadatos_raster(msk)

                dimensiones[f"{img.width} x {img.height}"] += 1
                bandas[str(img.count)] += 1
                sistemas[str(img.crs)] += 1

                misma_geometria = (
                    img.crs is not None
                    and msk.crs is not None
                    and img.crs == msk.crs
                    and img.width == msk.width
                    and img.height == msk.height
                    and img.transform == msk.transform
                )

                registro["geometria_identica"] = misma_geometria

                if misma_geometria:
                    pares_alineados += 1
                else:
                    problemas.append(
                        "Imagen y máscara sin geometría idéntica "
                        "o sin CRS declarado."
                    )

                if img.count != 4:
                    problemas.append(
                        f"Imagen con {img.count} bandas; se esperaban 4."
                    )

                if msk.count != 1:
                    problemas.append(
                        f"Máscara con {msk.count} bandas; se esperaba 1."
                    )
                else:
                    valores = msk.read(1)
                    validos_tiff = msk.read_masks(1) > 0

                    if not np.isfinite(valores).all():
                        problemas.append(
                            "La máscara contiene valores no finitos."
                        )

                    finitos = valores[np.isfinite(valores)]

                    if finitos.size and np.any(finitos != np.floor(finitos)):
                        problemas.append(
                            "La máscara contiene códigos no enteros."
                        )

                    unicos, cantidades = np.unique(
                        valores, return_counts=True
                    )

                    conteo = {
                        str(codigo.item()): int(cantidad)
                        for codigo, cantidad in zip(unicos, cantidades)
                    }

                    codigos_totales.update(conteo)
                    registro["codigos_almacenados"] = conteo

                    unicos_validos, cantidades_validas = np.unique(
                        valores[validos_tiff],
                        return_counts=True
                    )

                    codigos_validos_tiff.update({
                        str(codigo.item()): int(cantidad)
                        for codigo, cantidad in zip(
                            unicos_validos, cantidades_validas
                        )
                    })

                    invalidos = int(np.count_nonzero(~validos_tiff))
                    pixeles_invalidos_tiff += invalidos
                    registro["pixeles_invalidos_tiff"] = invalidos

        except Exception as error:
            problemas.append(f"{type(error).__name__}: {error}")

        registro["problemas"] = problemas
        registros.append(registro)

        if problemas:
            incidencias.append({
                "chip_id": fila["chip_id"],
                "problemas": problemas,
            })

        if numero % 500 == 0:
            print(f"  Revisados: {numero}/{len(filas)}", flush=True)

    REVISION_RASTERS[particion] = registros

    resumen = {
        "pares_revisados": len(filas),
        "pares_con_geometria_identica": pares_alineados,
        "dimensiones_imagen": dict(dimensiones),
        "numero_bandas_imagen": dict(bandas),
        "crs_imagen": dict(sistemas),
        "codigos_almacenados_con_repeticion_entre_parches":
            dict(codigos_totales),
        "codigos_validos_segun_tiff_con_repeticion_entre_parches":
            dict(codigos_validos_tiff),
        "pixeles_invalidos_segun_tiff": pixeles_invalidos_tiff,
        "incidencias": incidencias,
    }

    RESUMEN_RASTERS[particion] = resumen

    print("Pares revisados:", len(filas))
    print("Pares con geometría idéntica:", pares_alineados)
    print("Dimensiones de las imágenes:", dict(dimensiones))
    print("Número de bandas:", dict(bandas))
    print("CRS:", dict(sistemas))

    print("\nCódigos almacenados en las máscaras:")
    for codigo, cantidad in sorted(
        codigos_totales.items(),
        key=lambda elemento: float(elemento[0])
    ):
        print(f"  Código {codigo}: {cantidad:,}")

    print("\nCódigos válidos según la máscara/NoData del TIFF:")
    for codigo, cantidad in sorted(
        codigos_validos_tiff.items(),
        key=lambda elemento: float(elemento[0])
    ):
        print(f"  Código {codigo}: {cantidad:,}")

    print("\nPíxeles inválidos según TIFF:", pixeles_invalidos_tiff)
    print("Pares con incidencias:", len(incidencias))

    for incidencia in incidencias[:5]:
        print(" ", incidencia)

    ejemplo = next(
        (r for r in registros if "imagen" in r),
        None
    )

    if ejemplo:
        print("\nMetadatos de una imagen — ejemplo, no resumen global:")
        print("  Chip:", ejemplo["chip_id"])
        for campo in ("descripciones", "escalas", "offsets", "nodata"):
            print(f"  {campo}: {ejemplo['imagen'][campo]}")


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_06_geometria_y_codigos_{marca}.json"
)

informe = {
    "celda": 6,
    "fecha": datetime.now().astimezone().isoformat(),
    "dataset_original": r"C:\Users\Ivan sua\anaconda3\Dataset",
    "resumen": RESUMEN_RASTERS,
    "detalle": REVISION_RASTERS,
    "alcance": [
        "Se leyeron cabeceras de imágenes y máscaras.",
        "Se leyeron los códigos y las máscaras de validez de los TIFF de etiquetas.",
        "No se modificaron datos, pesos ni predicciones.",
        "Los conteos suman parches y pueden repetir píxeles territoriales.",
        "La validez declarada por el TIFF no define todavía la máscara de evaluación.",
    ],
    "pendiente": [
        "Comprobar superposición espacial entre particiones.",
        "Confirmar semántica de códigos con el proceso de entrenamiento.",
        "Confirmar orden espectral y preprocesamiento de U-Net.",
        "Verificar jerarquía temática entre los niveles 1 y 2.",
    ],
}

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(informe, archivo, ensure_ascii=False, indent=2)

print("\nRegistro guardado en:")
print(ruta_registro)
print("\nCELDA 6 TERMINADA. No se modificaron los archivos originales.")


nivel_1_train
  Revisados: 500/1017
  Revisados: 1000/1017
Pares revisados: 1017
Pares con geometría idéntica: 1017
Dimensiones de las imágenes: {'256 x 256': 1017}
Número de bandas: {'4': 1017}
CRS: {'EPSG:9377': 1017}

Códigos almacenados en las máscaras:
  Código 0: 23,503
  Código 1: 39,536,781
  Código 2: 23,360,552
  Código 3: 1,011,311
  Código 4: 2,513,309
  Código 5: 204,656

Códigos válidos según la máscara/NoData del TIFF:
  Código 1: 39,536,781
  Código 2: 23,360,552
  Código 3: 1,011,311
  Código 4: 2,513,309
  Código 5: 204,656

Píxeles inválidos según TIFF: 23503
Pares con incidencias: 0

Metadatos de una imagen — ejemplo, no resumen global:
  Chip: nivel_1_minority_r3840_c3968_cls3_256
  descripciones: ['B2 - Blue', 'B3 - Green', 'B4 - Red', 'B8 - Near Infrared']
  escalas: [1.0, 1.0, 1.0, 1.0]
  offsets: [0.0, 0.0, 0.0, 0.0]
  nodata: 0.0

nivel_1_val
Pares revisados: 218
Pares con geometría idéntica: 218
Dimensiones de las imágenes: {'256 x 256': 218}
Número de banda

In [7]:
# ============================================================
# CELDA 7 — SUPERPOSICIÓN ESPACIAL ENTRE PARTICIONES
# ============================================================

from pathlib import Path
from itertools import combinations
from datetime import datetime
import json
import numpy as np

if "REVISION_RASTERS" not in globals():
    raise RuntimeError("Primero ejecuta la celda 6.")

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")


# Verificamos las condiciones necesarias para comparar
# directamente los rectángulos de los parches.
crs_detectados = set()
problemas = []

for particion, registros in REVISION_RASTERS.items():
    for registro in registros:
        imagen = registro.get("imagen")

        if imagen is None or registro.get("problemas"):
            problemas.append(
                f"{particion}: {registro['chip_id']} tiene incidencias."
            )
            continue

        crs_detectados.add(imagen["crs"])

        a, b, c, d, e, f = imagen["transformacion"]

        if b != 0 or d != 0:
            problemas.append(
                f"{particion}: {registro['chip_id']} tiene rotación."
            )

        if a == 0 or e == 0:
            problemas.append(
                f"{particion}: {registro['chip_id']} tiene escala nula."
            )

        limites = np.asarray(imagen["limites"], dtype=float)

        if (
            not np.isfinite(limites).all()
            or limites[2] <= limites[0]
            or limites[3] <= limites[1]
        ):
            problemas.append(
                f"{particion}: {registro['chip_id']} tiene límites inválidos."
            )

if problemas:
    raise RuntimeError(
        "Antes de comparar geometrías hay que resolver:\n"
        + "\n".join(problemas[:10])
    )

if None in crs_detectados or len(crs_detectados) != 1:
    raise RuntimeError(
        f"No hay un único CRS declarado: {crs_detectados}"
    )


LIMITES_PARTICIONES = {
    particion: np.asarray(
        [r["imagen"]["limites"] for r in registros],
        dtype=float
    ).reshape(-1, 4)
    for particion, registros in REVISION_RASTERS.items()
}

CRUCES_ESPACIALES = {}

# Tolerancia pequeña respecto a la resolución de los datos.
resolucion_minima = min(
    min(abs(v) for v in r["imagen"]["resolucion"])
    for registros in REVISION_RASTERS.values()
    for r in registros
)

tolerancia = resolucion_minima * 1e-7

for nombre_a, nombre_b in combinations(REVISION_RASTERS, 2):

    registros_a = REVISION_RASTERS[nombre_a]
    registros_b = REVISION_RASTERS[nombre_b]

    limites_a = LIMITES_PARTICIONES[nombre_a]
    limites_b = LIMITES_PARTICIONES[nombre_b]

    afectados_a = np.zeros(len(registros_a), dtype=bool)
    afectados_b = np.zeros(len(registros_b), dtype=bool)

    cantidad_pares = 0
    ejemplos = []

    for i, (izquierda, abajo, derecha, arriba) in enumerate(limites_a):

        ancho_comun = (
            np.minimum(derecha, limites_b[:, 2])
            - np.maximum(izquierda, limites_b[:, 0])
        )

        alto_comun = (
            np.minimum(arriba, limites_b[:, 3])
            - np.maximum(abajo, limites_b[:, 1])
        )

        # Compartir solamente un borde no cuenta como superposición.
        indices = np.flatnonzero(
            (ancho_comun > tolerancia)
            & (alto_comun > tolerancia)
        )

        if indices.size:
            afectados_a[i] = True
            afectados_b[indices] = True
            cantidad_pares += int(indices.size)

            for j in indices[:max(0, 3 - len(ejemplos))]:
                ejemplos.append({
                    "chip_a": registros_a[i]["chip_id"],
                    "chip_b": registros_b[int(j)]["chip_id"],
                })

    clave = f"{nombre_a}__{nombre_b}"

    resultado = {
        "particion_a": nombre_a,
        "particion_b": nombre_b,
        "pares_con_superposicion": cantidad_pares,
        "parches_a_afectados": int(afectados_a.sum()),
        "parches_b_afectados": int(afectados_b.sum()),
        "total_parches_a": len(registros_a),
        "total_parches_b": len(registros_b),
        "ids_a_afectados": [
            registros_a[i]["chip_id"]
            for i in np.flatnonzero(afectados_a)
        ],
        "ids_b_afectados": [
            registros_b[i]["chip_id"]
            for i in np.flatnonzero(afectados_b)
        ],
        "ejemplos": ejemplos,
    }

    CRUCES_ESPACIALES[clave] = resultado

    print("\n" + "=" * 75)
    print(f"{nombre_a} / {nombre_b}")
    print("Pares con superposición:", cantidad_pares)
    print(
        f"Parches afectados en A: "
        f"{afectados_a.sum()} / {len(registros_a)}"
    )
    print(
        f"Parches afectados en B: "
        f"{afectados_b.sum()} / {len(registros_b)}"
    )

    for ejemplo in ejemplos:
        print("  Ejemplo:", ejemplo)


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_07_superposicion_espacial_{marca}.json"
)

informe = {
    "celda": 7,
    "fecha": datetime.now().astimezone().isoformat(),
    "crs": sorted(crs_detectados),
    "tolerancia_unidades_crs": tolerancia,
    "cruces": CRUCES_ESPACIALES,
    "alcance": [
        "Intersección de huellas rectangulares sin rotación y en un mismo CRS.",
        "El contacto por un borde no cuenta como superposición.",
        "No se han cruzado las máscaras de validez dentro de las huellas.",
        "No se han calculado superficies ni píxeles territoriales únicos.",
        "No se han eliminado muestras ni modificado particiones.",
        "La ausencia de superposición no demuestra independencia espacial.",
    ],
    "hallazgo_documental_unet": {
        "archivo_revisado": "ENTRENAMIENTO_UNET(1).ipynb",
        "estado": "No coincide con los checkpoints revisados en la celda 4.",
        "diferencias": [
            "Salidas: checkpoints 6/14; cuaderno 5/13.",
            "Ignore index: configuración guardada 0; cuaderno -100.",
            "Capa final: checkpoints outc; cuaderno out.",
            "Bloques: checkpoints inc.net; cuaderno inc.block.",
            "Muestreo ponderado: configuración activado; cuaderno desactivado.",
        ],
        "pendiente": (
            "Localizar la versión de entrenamiento correspondiente "
            "a los pesos seleccionados."
        ),
    },
}

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(informe, archivo, ensure_ascii=False, indent=2)

print("\nRegistro guardado en:")
print(ruta_registro)
print("\nCELDA 7 TERMINADA. No se modificaron los archivos originales.")


nivel_1_train / nivel_1_val
Pares con superposición: 66
Parches afectados en A: 53 / 1017
Parches afectados en B: 25 / 218
  Ejemplo: {'chip_a': 'nivel_1_minority_r3840_c3968_cls3_256', 'chip_b': 'nivel_1_minority_r3712_c3968_cls3_256'}
  Ejemplo: {'chip_a': 'nivel_1_minority_r4480_c2944_cls3_256', 'chip_b': 'nivel_1_base_r4608_c2816_cls2_256'}
  Ejemplo: {'chip_a': 'nivel_1_minority_r4480_c2944_cls3_256', 'chip_b': 'nivel_1_base_r4352_c2816_cls2_256'}

nivel_1_train / nivel_1_test
Pares con superposición: 79
Parches afectados en A: 51 / 1017
Parches afectados en B: 26 / 219
  Ejemplo: {'chip_a': 'nivel_1_minority_r3840_c3968_cls3_256', 'chip_b': 'nivel_1_minority_r3712_c4096_cls3_256'}
  Ejemplo: {'chip_a': 'nivel_1_minority_r3840_c3968_cls3_256', 'chip_b': 'nivel_1_base_r3840_c4096_cls1_256'}
  Ejemplo: {'chip_a': 'nivel_1_minority_r3840_c3968_cls3_256', 'chip_b': 'nivel_1_minority_r3968_c4096_cls3_256'}

nivel_1_train / nivel_2_train
Pares con superposición: 6092
Parches afectados 

In [8]:
# ============================================================
# CELDA 8 — DISPONIBILIDAD DE PRUEBA SIN SUPERPOSICIÓN
# Diagnóstico: no modifica las particiones originales.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import rasterio

for variable in (
    "CRUCES_ESPACIALES",
    "REVISION_RASTERS",
    "CARPETA_AUDITORIA",
):
    if variable not in globals():
        raise RuntimeError(f"Falta {variable}. Ejecuta las celdas anteriores.")


def ids_superpuestos(particion_objetivo, otra_particion):
    """Recupera los identificadores afectados, sin repetirlos."""
    for cruce in CRUCES_ESPACIALES.values():

        if (
            cruce["particion_a"] == particion_objetivo
            and cruce["particion_b"] == otra_particion
        ):
            return set(cruce["ids_a_afectados"])

        if (
            cruce["particion_b"] == particion_objetivo
            and cruce["particion_a"] == otra_particion
        ):
            return set(cruce["ids_b_afectados"])

    raise RuntimeError(
        f"No se encontró el cruce: "
        f"{particion_objetivo} / {otra_particion}"
    )


DIAGNOSTICO_PRUEBA = {}

for nivel, numero_clases in (("nivel_1", 5), ("nivel_2", 13)):

    particion = f"{nivel}_test"
    registros = REVISION_RASTERS[particion]

    escenarios = {
        "separacion_dentro_del_nivel": [
            f"{nivel}_train",
            f"{nivel}_val",
        ],
        "separacion_respecto_a_ambos_niveles": [
            "nivel_1_train",
            "nivel_1_val",
            "nivel_2_train",
            "nivel_2_val",
        ],
    }

    for escenario, particiones_a_comparar in escenarios.items():

        afectados = set()

        for otra_particion in particiones_a_comparar:
            afectados.update(
                ids_superpuestos(particion, otra_particion)
            )

        candidatos = [
            registro
            for registro in registros
            if registro["chip_id"] not in afectados
        ]

        conteos = {
            codigo: 0
            for codigo in range(1, numero_clases + 1)
        }

        inesperados = set()

        for registro in candidatos:
            with rasterio.open(registro["mask_path_local"]) as src:
                etiquetas = src.read(1)
                validos = src.read_masks(1) > 0

            codigos, cantidades = np.unique(
                etiquetas[validos],
                return_counts=True
            )

            for codigo, cantidad in zip(codigos, cantidades):
                valor = codigo.item()

                if valor in conteos:
                    conteos[int(valor)] += int(cantidad)
                else:
                    inesperados.add(str(valor))

        ausentes = [
            codigo
            for codigo, cantidad in conteos.items()
            if cantidad == 0
        ]

        clave = f"{nivel}__{escenario}"

        DIAGNOSTICO_PRUEBA[clave] = {
            "nivel": nivel,
            "escenario": escenario,
            "particiones_comparadas": particiones_a_comparar,
            "parches_test_originales": len(registros),
            "parches_con_superposicion": len(afectados),
            "parches_completos_sin_superposicion": len(candidatos),
            "ids_candidatos": [
                registro["chip_id"]
                for registro in candidatos
            ],
            "conteos_por_clase_en_mascaras": conteos,
            "clases_sin_representacion": ausentes,
            "codigos_validos_inesperados": sorted(inesperados),
        }

        print("\n" + "=" * 75)
        print(nivel, "—", escenario)
        print("=" * 75)
        print("Parches originales de prueba:", len(registros))
        print("Parches con alguna superposición:", len(afectados))
        print("Parches completos sin superposición:", len(candidatos))

        print("\nPíxeles de etiqueta por clase, sumados por parche:")
        for codigo, cantidad in conteos.items():
            print(f"  Clase {codigo}: {cantidad:,}")

        print("Clases sin representación:", ausentes)
        print("Códigos válidos inesperados:", sorted(inesperados))


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_08_disponibilidad_prueba_{marca}.json"
)

informe = {
    "celda": 8,
    "fecha": datetime.now().astimezone().isoformat(),
    "diagnostico": DIAGNOSTICO_PRUEBA,
    "alcance": [
        "Diagnóstico basado en las huellas revisadas en la celda 7.",
        "No se modificaron CSV ni se adoptó una nueva partición.",
        "Se consideran parches completos, no sus posibles fragmentos libres.",
        "Los conteos pueden repetir territorio entre parches candidatos.",
        "Los conteos usan validez de las máscaras, no aún de las imágenes.",
        "Ausencia de superposición no equivale a independencia espacial.",
        "Excluir parches puede cambiar la distribución de clases.",
        "Este diagnóstico no corrige la superposición train/val del entrenamiento previo.",
        "La correspondencia del entrenamiento de U-Net sigue pendiente.",
    ],
    "referencia_metodologica": {
        "autores": "Roberts et al.",
        "anio": 2017,
        "doi": "10.1111/ecog.02881",
    },
}

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(informe, archivo, ensure_ascii=False, indent=2)

print("\nRegistro guardado en:")
print(ruta_registro)
print("\nCELDA 8 TERMINADA. Diagnóstico sin modificar las particiones.")


nivel_1 — separacion_dentro_del_nivel
Parches originales de prueba: 219
Parches con alguna superposición: 26
Parches completos sin superposición: 193

Píxeles de etiqueta por clase, sumados por parche:
  Clase 1: 7,457,117
  Clase 2: 4,745,913
  Clase 3: 143,655
  Clase 4: 299,341
  Clase 5: 78
Clases sin representación: []
Códigos válidos inesperados: []

nivel_1 — separacion_respecto_a_ambos_niveles
Parches originales de prueba: 219
Parches con alguna superposición: 219
Parches completos sin superposición: 0

Píxeles de etiqueta por clase, sumados por parche:
  Clase 1: 0
  Clase 2: 0
  Clase 3: 0
  Clase 4: 0
  Clase 5: 0
Clases sin representación: [1, 2, 3, 4, 5]
Códigos válidos inesperados: []

nivel_2 — separacion_dentro_del_nivel
Parches originales de prueba: 799
Parches con alguna superposición: 73
Parches completos sin superposición: 726

Píxeles de etiqueta por clase, sumados por parche:
  Clase 1: 4,111,271
  Clase 2: 2,604,544
  Clase 3: 1,717,662
  Clase 4: 2,832,151
  Cl

In [9]:
# ============================================================
# CELDA 9 — INVENTARIO DE RÁSTERES COMPLETOS Y REFERENCIAS
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import rasterio

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")


# Rutas mencionadas durante el proceso.
# Su existencia se comprueba; no se da por confirmada.
CARPETAS_RASTERS = [
    Path(r"C:\Users\Ivan sua\anaconda3\Rasters_CLC"),
    Path(r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC"),
]

RUTAS_ESPERADAS = {
    "imagen_completa_por_confirmar": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif"
    ),
    "referencia_nivel_1": Path(
        r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC"
        r"\Raster_Almeidas_NIVEL_1_ALINEADO.tif"
    ),
    "referencia_nivel_2": Path(
        r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC"
        r"\Raster_Almeidas_NIVEL_2_ALINEADO.tif"
    ),
}

archivos = set()
estado_carpetas = {}

print("=" * 75)
print("CARPETAS")
print("=" * 75)

for carpeta in CARPETAS_RASTERS:
    existe = carpeta.is_dir()
    estado_carpetas[str(carpeta)] = existe

    print("ENCONTRADA:" if existe else "NO ENCONTRADA:", carpeta)

    if existe:
        # Solo el primer nivel; no recorre todo el disco.
        archivos.update(
            ruta
            for ruta in carpeta.iterdir()
            if ruta.is_file()
            and ruta.suffix.lower() in (".tif", ".tiff")
        )

print("\nRUTAS ESPERADAS")

for nombre, ruta in RUTAS_ESPERADAS.items():
    print(f"{nombre}: {'ENCONTRADO' if ruta.is_file() else 'NO ENCONTRADO'}")
    print(" ", ruta)

    if ruta.is_file():
        archivos.add(ruta)


INVENTARIO_RASTERS_COMPLETOS = {}

for ruta in sorted(archivos, key=lambda p: str(p).lower()):

    print("\n" + "=" * 75)
    print(ruta.name)
    print("=" * 75)

    try:
        with rasterio.open(ruta) as src:
            registro = {
                "ruta": str(ruta),
                "ancho": src.width,
                "alto": src.height,
                "bandas": src.count,
                "tipos": list(src.dtypes),
                "crs": src.crs.to_string() if src.crs else None,
                "transformacion": list(src.transform)[:6],
                "limites": list(src.bounds),
                "resolucion": list(src.res),
                "nodata": str(src.nodata),
                "descripciones": list(src.descriptions),
                "escalas": list(src.scales),
                "offsets": list(src.offsets),
                "etiquetas_archivo": src.tags(),
                "etiquetas_bandas": {
                    str(banda): src.tags(banda)
                    for banda in src.indexes
                },
                "estado": "lectura_correcta",
            }

        for campo in (
            "ancho", "alto", "bandas", "tipos", "crs",
            "resolucion", "limites", "transformacion",
            "nodata", "descripciones", "escalas", "offsets"
        ):
            print(f"{campo}: {registro[campo]}")

    except Exception as error:
        registro = {
            "ruta": str(ruta),
            "estado": "error",
            "error": f"{type(error).__name__}: {error}",
        }
        print(registro["error"])

    INVENTARIO_RASTERS_COMPLETOS[str(ruta)] = registro


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_09_rasters_completos_{marca}.json"
)

informe = {
    "celda": 9,
    "fecha": datetime.now().astimezone().isoformat(),
    "carpetas": estado_carpetas,
    "rutas_esperadas": {
        nombre: {
            "ruta": str(ruta),
            "existe": ruta.is_file(),
        }
        for nombre, ruta in RUTAS_ESPERADAS.items()
    },
    "rasters": INVENTARIO_RASTERS_COMPLETOS,
    "decision_tras_celda_08": (
        "No se adopta todavía un subconjunto definitivo de evaluación. "
        "La eliminación de parches completos reduce fuertemente "
        "la representación de algunas clases."
    ),
    "pendientes": [
        "Confirmar la imagen completa usada para las predicciones.",
        "Verificar correspondencia espacial y temporal con el dataset.",
        "Comprobar códigos y jerarquía de las referencias completas.",
        "Cuantificar superficie común disponible en píxeles únicos.",
        "Resolver la correspondencia entre cuaderno y pesos de U-Net.",
    ],
    "alcance": "Solo lectura de metadatos; sin reproyección ni modificación.",
}

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(informe, archivo, ensure_ascii=False, indent=2)

print("\nTotal de rásteres encontrados:", len(archivos))
print("\nRegistro guardado en:")
print(ruta_registro)
print("\nCELDA 9 TERMINADA. No se modificaron los archivos originales.")

CARPETAS
NO ENCONTRADA: C:\Users\Ivan sua\anaconda3\Rasters_CLC
ENCONTRADA: D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC

RUTAS ESPERADAS
imagen_completa_por_confirmar: NO ENCONTRADO
  C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif
referencia_nivel_1: ENCONTRADO
  D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\Raster_Almeidas_NIVEL_1_ALINEADO.tif
referencia_nivel_2: ENCONTRADO
  D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\Raster_Almeidas_NIVEL_2_ALINEADO.tif

Raster_Almeidas_2025.tif
ancho: 4376
alto: 4999
bandas: 4
tipos: ['uint16', 'uint16', 'uint16', 'uint16']
crs: PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1329"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","20046"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",4],PARAMETER["cen

In [14]:
# ============================================================
# CELDA 10 — IMÁGENES, CÓDIGOS Y JERARQUÍA DE REFERENCIAS
# Solo diagnóstico. No modifica rásteres.
# ============================================================

from pathlib import Path
from datetime import datetime
from collections import Counter
from contextlib import ExitStack
import json

import numpy as np
import rasterio
from rasterio.windows import Window

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")

BASE_RASTERS = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC"
)

RUTAS_COMPROBACION = {
    "zona": BASE_RASTERS / "zona_estudio.tif",
    "imagen_2025": BASE_RASTERS / "Raster_Almeidas_2025.tif",
    "nivel_1": BASE_RASTERS / "Raster_Almeidas_NIVEL_1_ALINEADO.tif",
    "nivel_2": BASE_RASTERS / "Raster_Almeidas_NIVEL_2_ALINEADO.tif",
}

# Hipótesis según el catálogo facilitado por el usuario.
# Su aplicación a estos rásteres aún debe verificarse.
PADRE_N2_N1 = {
    1: 1, 2: 2, 3: 2, 4: 1, 5: 3,
    6: 4, 7: 4, 8: 5, 9: 4, 10: 4,
    11: 1, 12: 1, 13: 2,
}


def validez_por_banda(src, datos, ventana):
    validos = (src.read_masks(window=ventana) > 0)
    validos &= np.isfinite(datos)

    for banda, nodata in enumerate(src.nodatavals):
        if nodata is not None:
            validos[banda] &= datos[banda] != nodata

    return validos


def agregar_conteos(contador, valores):
    codigos, cantidades = np.unique(valores, return_counts=True)
    contador.update({
        int(codigo): int(cantidad)
        for codigo, cantidad in zip(codigos, cantidades)
    })


conteos_referencia = {"nivel_1": Counter(), "nivel_2": Counter()}
conteos_comunes = {"nivel_1": Counter(), "nivel_2": Counter()}
cruce_referencias = Counter()

pixeles_comunes = 0
pixeles_imagen_distintos = 0
validez_distinta = 0
comparables_jerarquia = 0
discrepancias_jerarquia = 0

with ExitStack() as pila:
    fuentes = {
        nombre: pila.enter_context(rasterio.open(ruta))
        for nombre, ruta in RUTAS_COMPROBACION.items()
    }

    base = fuentes["zona"]

    print("COMPROBACIÓN DE CUADRÍCULAS")

    for nombre, src in fuentes.items():
        coincide = (
            src.crs is not None
            and crs_equivalentes_xy(src.crs, base.crs)
            and src.transform == base.transform
            and src.width == base.width
            and src.height == base.height
        )
        print(f"  {nombre}: {coincide}")

        if not coincide:
            raise RuntimeError(
                f"{nombre}: la cuadrícula o el CRS no coincide. "
                "No se realizará una comparación por posición."
            )

    if fuentes["zona"].count != 4 or fuentes["imagen_2025"].count != 4:
        raise RuntimeError("Se esperaban cuatro bandas en ambas imágenes.")

    if any(fuentes[n].count != 1 for n in ("nivel_1", "nivel_2")):
        raise RuntimeError("Se esperaba una banda por referencia.")

    total_pixeles = base.width * base.height

    for fila in range(0, base.height, 512):
        for columna in range(0, base.width, 512):
            ventana = Window(
                columna,
                fila,
                min(512, base.width - columna),
                min(512, base.height - fila),
            )

            zona = fuentes["zona"].read(window=ventana)
            imagen = fuentes["imagen_2025"].read(window=ventana)

            vz = validez_por_banda(fuentes["zona"], zona, ventana)
            vi = validez_por_banda(fuentes["imagen_2025"], imagen, ventana)

            # Comparación de valores almacenados, incluso en zonas NoData.
            pixeles_imagen_distintos += int(
                np.count_nonzero(np.any(zona != imagen, axis=0))
            )
            validez_distinta += int(
                np.count_nonzero(np.any(vz != vi, axis=0))
            )

            referencias = {}
            validas = {}

            for nivel in ("nivel_1", "nivel_2"):
                src = fuentes[nivel]
                datos = src.read(window=ventana)
                referencias[nivel] = datos[0]

                validas[nivel] = (
                    validez_por_banda(src, datos, ventana)[0]
                    & (datos[0] != 0)
                )

                agregar_conteos(
                    conteos_referencia[nivel],
                    datos[0][validas[nivel]]
                )

            comun = (
                vz.all(axis=0)
                & vi.all(axis=0)
                & validas["nivel_1"]
                & validas["nivel_2"]
            )

            pixeles_comunes += int(comun.sum())

            r1 = referencias["nivel_1"][comun]
            r2 = referencias["nivel_2"][comun]

            agregar_conteos(conteos_comunes["nivel_1"], r1)
            agregar_conteos(conteos_comunes["nivel_2"], r2)

            if r1.size:
                pares, cantidades = np.unique(
                    np.column_stack((r1, r2)),
                    axis=0,
                    return_counts=True
                )

                for (codigo1, codigo2), cantidad in zip(pares, cantidades):
                    codigo1 = int(codigo1)
                    codigo2 = int(codigo2)
                    cantidad = int(cantidad)

                    cruce_referencias[(codigo1, codigo2)] += cantidad

                    if codigo1 in range(1, 6) and codigo2 in PADRE_N2_N1:
                        comparables_jerarquia += cantidad

                        if codigo1 != PADRE_N2_N1[codigo2]:
                            discrepancias_jerarquia += cantidad


print("\nCOMPARACIÓN DE LAS DOS IMÁGENES")
print("Píxeles de la cuadrícula:", total_pixeles)
print("Píxeles con alguna banda diferente:", pixeles_imagen_distintos)
print("Píxeles con validez diferente:", validez_distinta)

print("\nREFERENCIAS: CÓDIGOS VÁLIDOS EN CADA ARCHIVO")
for nivel, contador in conteos_referencia.items():
    print(nivel, dict(sorted(contador.items())))

print("\nSUPERFICIE VÁLIDA COMÚN PRELIMINAR")
print("Píxeles:", pixeles_comunes)

for nivel, contador in conteos_comunes.items():
    print(nivel, dict(sorted(contador.items())))

print("\nCRUCE DE REFERENCIAS: CÓDIGO NIVEL 2 → CÓDIGOS NIVEL 1")
for codigo2 in sorted({par[1] for par in cruce_referencias}):
    distribucion = {
        codigo1: cantidad
        for (codigo1, c2), cantidad in sorted(cruce_referencias.items())
        if c2 == codigo2
    }
    print(f"  Nivel 2, código {codigo2}: {distribucion}")

print("\nJERARQUÍA PROPUESTA SEGÚN LA LEYENDA")
print("Píxeles con códigos contemplados:", comparables_jerarquia)
print("Píxeles que discrepan:", discrepancias_jerarquia)
print(
    "Píxeles comunes con códigos fuera del catálogo:",
    pixeles_comunes - comparables_jerarquia
)

if comparables_jerarquia:
    print(
        "Discrepancia entre códigos contemplados:",
        f"{100 * discrepancias_jerarquia / comparables_jerarquia:.4f}%"
    )


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_10_imagenes_y_referencias_{marca}.json"
)

informe = {
    "celda": 10,
    "fecha": datetime.now().astimezone().isoformat(),
    "rutas": {nombre: str(ruta) for nombre, ruta in RUTAS_COMPROBACION.items()},
    "pixeles_cuadricula": total_pixeles,
    "pixeles_imagen_distintos": pixeles_imagen_distintos,
    "pixeles_validez_distinta": validez_distinta,
    "conteos_referencia": {
        nivel: dict(contador)
        for nivel, contador in conteos_referencia.items()
    },
    "pixeles_comunes_preliminares": pixeles_comunes,
    "conteos_comunes": {
        nivel: dict(contador)
        for nivel, contador in conteos_comunes.items()
    },
    "cruce_referencias": [
        {"nivel_1": c1, "nivel_2": c2, "pixeles": cantidad}
        for (c1, c2), cantidad in sorted(cruce_referencias.items())
    ],
    "jerarquia_propuesta": PADRE_N2_N1,
    "pixeles_comparables_jerarquia": comparables_jerarquia,
    "pixeles_discrepantes_jerarquia": discrepancias_jerarquia,
    "limites": [
        "La coincidencia numérica no prueba por sí sola la semántica de clases.",
        "No se asignaron nuevos códigos ni se aplicó remapeo óptimo.",
        "La máscara común todavía no excluye entrenamiento ni validación.",
        "Faltan verificar nubes, orden de bandas y procedencia temporal.",
        "La igualdad de valores no implica igualdad de todos los metadatos.",
    ],
}

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(informe, archivo, ensure_ascii=False, indent=2)

print("\nRegistro guardado en:")
print(ruta_registro)
print("\nCELDA 10 TERMINADA. No se modificaron los rásteres.")

COMPROBACIÓN DE CUADRÍCULAS
  zona: True
  imagen_2025: True
  nivel_1: True
  nivel_2: True

COMPARACIÓN DE LAS DOS IMÁGENES
Píxeles de la cuadrícula: 21875624
Píxeles con alguna banda diferente: 0
Píxeles con validez diferente: 0

REFERENCIAS: CÓDIGOS VÁLIDOS EN CADA ARCHIVO
nivel_1 {1: 3375868, 2: 8676599, 3: 87395, 4: 8600, 5: 174857}
nivel_2 {1: 2510241, 2: 5340779, 3: 3129265, 4: 64148, 5: 8600, 6: 145835, 7: 838897, 8: 18430, 9: 4817, 10: 174857, 11: 60720, 12: 26730}

SUPERFICIE VÁLIDA COMÚN PRELIMINAR
Píxeles: 12319921
nivel_1 {1: 3374666, 2: 8674615, 3: 87395, 4: 8586, 5: 174659}
nivel_2 {1: 2509303, 2: 5339526, 3: 3128559, 4: 64148, 5: 8586, 6: 145835, 7: 838692, 8: 18430, 9: 4817, 10: 174659, 11: 60695, 12: 26671}

CRUCE DE REFERENCIAS: CÓDIGO NIVEL 2 → CÓDIGOS NIVEL 1
  Nivel 2, código 1: {1: 2509303}
  Nivel 2, código 2: {2: 5339526}
  Nivel 2, código 3: {2: 3128559}
  Nivel 2, código 4: {3: 64148}
  Nivel 2, código 5: {4: 8586}
  Nivel 2, código 6: {2: 145835}
  Nivel 2,

In [11]:
# ============================================================
# CELDA 10A — IDENTIFICAR QUÉ CONDICIÓN NO COINCIDE
# Solo lectura. No modifica ni reproyecta archivos.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import rasterio

BASE = Path(r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC")

rutas = {
    "zona": BASE / "zona_estudio.tif",
    "nivel_1": BASE / "Raster_Almeidas_NIVEL_1_ALINEADO.tif",
    "nivel_2": BASE / "Raster_Almeidas_NIVEL_2_ALINEADO.tif",
}

DETALLE_CUADRICULAS = {}

with rasterio.open(rutas["zona"]) as base:

    for nombre, ruta in rutas.items():
        with rasterio.open(ruta) as src:

            diferencia = (
                np.asarray(tuple(src.transform)[:6], dtype=float)
                - np.asarray(tuple(base.transform)[:6], dtype=float)
            )

            detalle = {
                "ruta": str(ruta),
                "dimensiones_iguales": (
                    src.width == base.width
                    and src.height == base.height
                ),
                "transformacion_exactamente_igual": (
                    src.transform == base.transform
                ),
                "diferencia_transformacion": diferencia.tolist(),
                "crs_igual_segun_rasterio": src.crs == base.crs,
                "epsg_identificado": src.crs.to_epsg() if src.crs else None,
                "crs_wkt": src.crs.to_wkt() if src.crs else None,
                "transformacion": list(src.transform)[:6],
                "dimensiones": [src.width, src.height],
            }

            DETALLE_CUADRICULAS[nombre] = detalle

            print("\n" + "=" * 75)
            print(nombre)
            print("=" * 75)

            for campo, valor in detalle.items():
                if campo != "crs_wkt":
                    print(f"{campo}: {valor}")

            print("\nDefinición completa del CRS:")
            print(detalle["crs_wkt"])


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_10A_diagnostico_crs_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": "10A",
            "fecha": datetime.now().astimezone().isoformat(),
            "diagnostico": DETALLE_CUADRICULAS,
            "estado_celda_10": (
                "Interrumpida antes del conteo por discrepancia geométrica "
                "o de CRS. Sin resultados de comparación de píxeles."
            ),
            "accion": "Solo diagnóstico; no se modificaron archivos.",
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:")
print(ruta_registro)


zona
ruta: D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\zona_estudio.tif
dimensiones_iguales: True
transformacion_exactamente_igual: True
diferencia_transformacion: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
crs_igual_segun_rasterio: True
epsg_identificado: None
transformacion: [10.0, 0.0, 4903449.579587339, 0.0, -10.0, 2150915.3094111346]
dimensiones: [4376, 4999]

Definición completa del CRS:
PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1329"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","20046"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",4],PARAMETER["central_meridian",-73],PARAMETER["scale_factor",0.9992],PARAMETER["false_easting",5000000],PARAMETER["false_northing",2000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northin

In [12]:
# ============================================================
# CELDA 10B — EQUIVALENCIA DEL CRS SIN MODIFICAR LOS RÁSTERES
# ============================================================

from pathlib import Path
from datetime import datetime
import json

try:
    import pyproj
    from pyproj import CRS
except ImportError:
    raise RuntimeError(
        "pyproj no está disponible en este entorno. "
        "Comparte este mensaje antes de instalar o cambiar bibliotecas."
    )

if "DETALLE_CUADRICULAS" not in globals():
    raise RuntimeError("Primero ejecuta la celda 10A.")

crs_base = CRS.from_wkt(
    DETALLE_CUADRICULAS["zona"]["crs_wkt"]
)

EQUIVALENCIA_CRS = {}

print("Versión de pyproj:", pyproj.__version__)
print("Versión de PROJ:", pyproj.proj_version_str)

for nombre, detalle in DETALLE_CUADRICULAS.items():

    crs_actual = CRS.from_wkt(detalle["crs_wkt"])

    equivalencia = crs_base.equals(
        crs_actual,
        ignore_axis_order=True
    )

    cuadricula_identica = (
        detalle["dimensiones_iguales"]
        and detalle["transformacion_exactamente_igual"]
    )

    resultado = {
        "equivalentes_ignorando_orden_ejes": bool(equivalencia),
        "dimensiones_y_transformacion_identicas": bool(cuadricula_identica),
        "compatible_para_comparacion_por_posicion": bool(
            equivalencia and cuadricula_identica
        ),
        "ejes": [
            {
                "nombre": eje.name,
                "direccion": eje.direction,
                "unidad": eje.unit_name,
            }
            for eje in crs_actual.axis_info
        ],
    }

    EQUIVALENCIA_CRS[nombre] = resultado

    print("\n" + "=" * 75)
    print(nombre)
    print("=" * 75)

    for campo, valor in resultado.items():
        print(f"{campo}: {valor}")


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_10B_equivalencia_crs_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": "10B",
            "fecha": datetime.now().astimezone().isoformat(),
            "pyproj": pyproj.__version__,
            "proj": pyproj.proj_version_str,
            "resultados": EQUIVALENCIA_CRS,
            "alcance": (
                "Comparación de definiciones CRS y geometría. "
                "Sin modificación de metadatos, reproyección "
                "ni intercambio de ejes de los datos."
            ),
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:")
print(ruta_registro)

Versión de pyproj: 3.7.2
Versión de PROJ: 9.7.1

zona
equivalentes_ignorando_orden_ejes: True
dimensiones_y_transformacion_identicas: True
compatible_para_comparacion_por_posicion: True
ejes: [{'nombre': 'Easting', 'direccion': 'east', 'unidad': 'metre'}, {'nombre': 'Northing', 'direccion': 'north', 'unidad': 'metre'}]

nivel_1
equivalentes_ignorando_orden_ejes: False
dimensiones_y_transformacion_identicas: True
compatible_para_comparacion_por_posicion: False
ejes: [{'nombre': 'Northing', 'direccion': 'north', 'unidad': 'metre'}, {'nombre': 'Easting', 'direccion': 'east', 'unidad': 'metre'}]

nivel_2
equivalentes_ignorando_orden_ejes: False
dimensiones_y_transformacion_identicas: True
compatible_para_comparacion_por_posicion: False
ejes: [{'nombre': 'Northing', 'direccion': 'north', 'unidad': 'metre'}, {'nombre': 'Easting', 'direccion': 'east', 'unidad': 'metre'}]

Registro guardado en:
D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_10B_equivalencia_crs_

In [13]:
# ============================================================
# CELDA 10C — COMPARACIÓN CON EJES PROYECTADOS EN ORDEN E/N
# Solo normaliza objetos de comparación en memoria.
# ============================================================

from pathlib import Path
from datetime import datetime
import json

import rasterio
from pyproj import CRS


def crs_para_comparacion_xy(crs_raster):
    crs = CRS.from_wkt(crs_raster.to_wkt())

    if not crs.is_projected:
        raise ValueError("Esta validación requiere un CRS proyectado.")

    definicion = crs.to_json_dict()
    sistema = definicion["coordinate_system"]
    ejes = sistema["axis"]

    direcciones = [eje["direction"] for eje in ejes]

    if len(ejes) != 2 or set(direcciones) != {"east", "north"}:
        raise ValueError(
            f"Orden o direcciones de ejes no contemplados: {direcciones}"
        )

    # Reordenamos los ejes existentes, conservando sus unidades.
    sistema["axis"] = [
        next(eje for eje in ejes if eje["direction"] == direccion)
        for direccion in ("east", "north")
    ]

    # La copia normalizada puede no tener el orden oficial del EPSG.
    # Quitamos únicamente el identificador del CRS completo.
    definicion.pop("id", None)

    return CRS.from_json_dict(definicion)


def crs_equivalentes_xy(crs_a, crs_b):
    if crs_a is None or crs_b is None:
        return False

    return crs_para_comparacion_xy(crs_a).equals(
        crs_para_comparacion_xy(crs_b)
    )


BASE = Path(r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC")

rutas = {
    "zona": BASE / "zona_estudio.tif",
    "imagen_2025": BASE / "Raster_Almeidas_2025.tif",
    "nivel_1": BASE / "Raster_Almeidas_NIVEL_1_ALINEADO.tif",
    "nivel_2": BASE / "Raster_Almeidas_NIVEL_2_ALINEADO.tif",
}

VALIDACION_XY = {}

with rasterio.open(rutas["zona"]) as base:
    for nombre, ruta in rutas.items():
        with rasterio.open(ruta) as src:

            equivalentes = crs_equivalentes_xy(src.crs, base.crs)

            geometria_identica = (
                src.width == base.width
                and src.height == base.height
                and src.transform == base.transform
            )

            VALIDACION_XY[nombre] = {
                "crs_equivalente_en_orden_xy": bool(equivalentes),
                "geometria_identica": bool(geometria_identica),
                "comparacion_por_posicion_admitida": bool(
                    equivalentes and geometria_identica
                ),
            }

            print(nombre)
            for campo, valor in VALIDACION_XY[nombre].items():
                print(f"  {campo}: {valor}")


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_10C_validacion_xy_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": "10C",
            "fecha": datetime.now().astimezone().isoformat(),
            "resultados": VALIDACION_XY,
            "metodo": (
                "Comparación de CRS con ejes proyectados ordenados "
                "Este/Norte en copias en memoria, conservando "
                "datum, elipsoide, unidades y proyección."
            ),
            "archivos_modificados": False,
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:", ruta_registro)

APTO_PARA_CELDA_10 = all(
    resultado["comparacion_por_posicion_admitida"]
    for resultado in VALIDACION_XY.values()
)

print("\nAPTO_PARA_CELDA_10:", APTO_PARA_CELDA_10)

zona
  crs_equivalente_en_orden_xy: True
  geometria_identica: True
  comparacion_por_posicion_admitida: True
imagen_2025
  crs_equivalente_en_orden_xy: True
  geometria_identica: True
  comparacion_por_posicion_admitida: True
nivel_1
  crs_equivalente_en_orden_xy: True
  geometria_identica: True
  comparacion_por_posicion_admitida: True
nivel_2
  crs_equivalente_en_orden_xy: True
  geometria_identica: True
  comparacion_por_posicion_admitida: True

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_10C_validacion_xy_20261007_195159_943070.json

APTO_PARA_CELDA_10: True


In [15]:
# ============================================================
# CELDA 11 — PROCEDENCIA DE LOS CÓDIGOS DE REFERENCIA
# Lectura de metadatos y búsqueda de tablas auxiliares.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import rasterio

BASE = Path(r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC")

REFERENCIAS_A_REVISAR = [
    BASE / "Raster_CT_Almeidas_2022_Nivel1.tif",
    BASE / "Raster_CT_Almeidas_2022_Nivel2.tif",
    BASE / "Raster_Almeidas_NIVEL_1_ALINEADO.tif",
    BASE / "Raster_Almeidas_NIVEL_2_ALINEADO.tif",
]

METADATOS_REFERENCIAS = {}

for ruta in REFERENCIAS_A_REVISAR:

    print("\n" + "=" * 75)
    print(ruta.name)
    print("=" * 75)

    if not ruta.is_file():
        print("Archivo no encontrado.")
        METADATOS_REFERENCIAS[str(ruta)] = {"existe": False}
        continue

    registro = {
        "existe": True,
        "etiquetas_archivo": {},
        "etiquetas_banda": {},
        "auxiliares": [],
    }

    with rasterio.open(ruta) as src:

        for banda, destino in (
            (0, registro["etiquetas_archivo"]),
            (1, registro["etiquetas_banda"]),
        ):
            destino["principal"] = src.tags(banda)

            for espacio in src.tag_namespaces(banda):
                destino[espacio] = src.tags(banda, ns=espacio)

        print("Metadatos del archivo:")
        print(json.dumps(
            registro["etiquetas_archivo"],
            ensure_ascii=False,
            indent=2
        ))

        print("Metadatos de la banda:")
        print(json.dumps(
            registro["etiquetas_banda"],
            ensure_ascii=False,
            indent=2
        ))

    # Incluye nombres como:
    # archivo.tif.aux.xml, archivo.tif.vat.dbf, archivo.qml...
    auxiliares = sorted(
        p for p in BASE.iterdir()
        if p.is_file()
        and p != ruta
        and (
            p.name.lower().startswith(ruta.name.lower() + ".")
            or p.stem.lower() == ruta.stem.lower()
        )
    )

    print("\nArchivos auxiliares asociados:")

    if not auxiliares:
        print("  Ninguno encontrado con el mismo nombre base.")

    for auxiliar in auxiliares:
        item = {
            "ruta": str(auxiliar),
            "tamano_bytes": auxiliar.stat().st_size,
        }

        print(" ", auxiliar.name)

        # Los archivos binarios se inventarían, no se leen como texto.
        if auxiliar.suffix.lower() in {
            ".xml", ".json", ".csv", ".txt", ".qml", ".sld"
        }:
            if auxiliar.stat().st_size <= 2_000_000:
                texto = auxiliar.read_text(
                    encoding="utf-8-sig",
                    errors="replace"
                )
                item["contenido"] = texto

                print(texto[:6000])
                if len(texto) > 6000:
                    print(
                        "  [Vista abreviada; contenido completo "
                        "guardado en el registro.]"
                    )
            else:
                item["lectura"] = "Pendiente: archivo mayor a 2 MB."

        registro["auxiliares"].append(item)

    METADATOS_REFERENCIAS[str(ruta)] = registro


print("\n" + "=" * 75)
print("POSIBLES TABLAS, LEYENDAS Y FUENTES EN LA MISMA CARPETA")
print("=" * 75)

extensiones = {
    ".dbf", ".csv", ".xlsx", ".xls", ".json",
    ".xml", ".qml", ".sld", ".lyrx", ".lyr",
    ".shp", ".gpkg", ".py", ".ipynb",
}

candidatos = [
    str(p)
    for p in sorted(BASE.iterdir())
    if (
        p.is_file() and p.suffix.lower() in extensiones
    ) or (
        p.is_dir() and p.suffix.lower() == ".gdb"
    )
]

for candidato in candidatos:
    print(candidato)

if not candidatos:
    print("No se encontraron archivos candidatos en esta carpeta.")


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_11_procedencia_codigos_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 11,
            "fecha": datetime.now().astimezone().isoformat(),
            "metadatos": METADATOS_REFERENCIAS,
            "archivos_candidatos": candidatos,
            "hallazgo_celda_10": {
                "pixeles_comunes_preliminares": 12319921,
                "discrepancias_con_jerarquia_propuesta": 1315862,
                "interpretacion": (
                    "La correspondencia semántica entre códigos del "
                    "dataset y referencias sigue pendiente. "
                    "No es una medida del error de los modelos."
                ),
            },
            "accion": "Solo lectura; sin asignar nombres ni remapear.",
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:")
print(ruta_registro)
print("\nCELDA 11 TERMINADA. No se modificaron archivos.")


Raster_CT_Almeidas_2022_Nivel1.tif
Metadatos del archivo:
{
  "principal": {
    "TIFFTAG_SOFTWARE": "GDAL 3.9.2e (3090222) (4.6.0;1710;3.0.3;2.1.6;;0x10400;5.4.7;4.0.0;), PROJ 9.4.1 (EPSG.VERSION=v11.006;EPSG.DATE=2024-03-13;ESRI.VERSION=ArcGIS Pro 3.3;ESRI.DATE=2024-05-07;PROJ_DATA.VERSION=1.18;)",
    "AREA_OR_POINT": "Area"
  },
  "IMAGE_STRUCTURE": {
    "COMPRESSION": "LZW",
    "INTERLEAVE": "BAND"
  },
  "Esri": {
    "PyramidResamplingType": "NEAREST"
  },
  "DERIVED_SUBDATASETS": {
    "DERIVED_SUBDATASET_1_NAME": "DERIVED_SUBDATASET:LOGAMPLITUDE:D:\\PROYECTO_GRADO_1\\PROYECTO_GRADO\\Raster_CLC\\Raster_CT_Almeidas_2022_Nivel1.tif",
    "DERIVED_SUBDATASET_1_DESC": "log10 of amplitude of input bands from D:\\PROYECTO_GRADO_1\\PROYECTO_GRADO\\Raster_CLC\\Raster_CT_Almeidas_2022_Nivel1.tif"
  }
}
Metadatos de la banda:
{
  "principal": {},
  "IMAGE_STRUCTURE": {
    "NBITS": "4"
  }
}

Archivos auxiliares asociados:
  Ninguno encontrado con el mismo nombre base.

Raster_CT_Alme

In [16]:
# ============================================================
# CELDA 12 — HOMOLOGACIÓN DOCUMENTADA DE LAS REFERENCIAS
# Evidencia: tablas de atributos suministradas por el usuario.
# No modifica rásteres ni predicciones.
# ============================================================

from pathlib import Path
from datetime import datetime
from collections import Counter
import json

for nombre in ("cruce_referencias", "conteos_comunes", "CARPETA_AUDITORIA"):
    if nombre not in globals():
        raise RuntimeError(
            f"Falta {nombre}. Ejecuta primero la celda 10 corregida."
        )

# Value del TIFF -> código CLC de la tabla de atributos.
VALUE_A_CLC = {
    "nivel_1": {
        1: 3, 2: 2, 3: 1, 4: 4, 5: 5,
    },
    "nivel_2": {
        1: 32, 2: 24, 3: 23, 4: 11,
        5: 41, 6: 21, 7: 31, 8: 12,
        9: 13, 10: 51, 11: 22, 12: 33,
    },
}

# Código CLC -> código interno del dataset.
CLC_A_DATASET = {
    "nivel_1": {
        1: 4, 2: 1, 3: 2, 4: 5, 5: 3,
    },
    "nivel_2": {
        11: 6, 12: 9, 13: 7, 14: 10,
        21: 11, 22: 12, 23: 4, 24: 1,
        31: 3, 32: 2, 33: 13, 41: 8, 51: 5,
    },
}

MAPAS_REFERENCIA_A_DATASET = {
    nivel: {
        0: 0,
        **{
            valor: CLC_A_DATASET[nivel][clc]
            for valor, clc in correspondencia.items()
        },
    }
    for nivel, correspondencia in VALUE_A_CLC.items()
}

PADRE_DATASET_N2_N1 = {
    1: 1, 2: 2, 3: 2, 4: 1, 5: 3,
    6: 4, 7: 4, 8: 5, 9: 4, 10: 4,
    11: 1, 12: 1, 13: 2,
}

CONTEOS_COMUNES_HOMOLOGADOS = {}
registros_correspondencia = {}

for nivel, cantidad_clases in (("nivel_1", 5), ("nivel_2", 13)):

    mapa = MAPAS_REFERENCIA_A_DATASET[nivel]
    desconocidos = set(conteos_comunes[nivel]) - set(mapa)

    if desconocidos:
        raise RuntimeError(
            f"{nivel}: códigos sin correspondencia: {desconocidos}"
        )

    # Evita fusionar accidentalmente dos clases de la referencia.
    destinos = [
        destino for origen, destino in mapa.items() if origen != 0
    ]
    if len(destinos) != len(set(destinos)):
        raise RuntimeError(f"{nivel}: la correspondencia fusionaría clases.")

    contador = Counter({
        codigo: 0 for codigo in range(1, cantidad_clases + 1)
    })

    for origen, cantidad in conteos_comunes[nivel].items():
        contador[mapa[int(origen)]] += int(cantidad)

    CONTEOS_COMUNES_HOMOLOGADOS[nivel] = dict(sorted(contador.items()))

    registros_correspondencia[nivel] = [
        {
            "value_referencia": valor,
            "codigo_clc": clc,
            "codigo_dataset": mapa[valor],
        }
        for valor, clc in VALUE_A_CLC[nivel].items()
    ]

    print("\n" + "=" * 75)
    print(nivel)
    print("=" * 75)
    print("Value de referencia → código del dataset:")
    print(mapa)
    print("\nSoporte en la superficie común preliminar:")

    for codigo, cantidad in sorted(contador.items()):
        print(f"  Código del dataset {codigo}: {cantidad:,}")

    print(
        "Clases sin referencia:",
        [codigo for codigo, cantidad in sorted(contador.items())
         if cantidad == 0]
    )


total_comparado = 0
discrepancias = 0
detalle_discrepancias = []

for (valor1, valor2), cantidad in sorted(cruce_referencias.items()):

    codigo1 = MAPAS_REFERENCIA_A_DATASET["nivel_1"][int(valor1)]
    codigo2 = MAPAS_REFERENCIA_A_DATASET["nivel_2"][int(valor2)]
    padre = PADRE_DATASET_N2_N1[codigo2]

    total_comparado += int(cantidad)

    if codigo1 != padre:
        discrepancias += int(cantidad)
        detalle_discrepancias.append({
            "value_nivel_1": int(valor1),
            "value_nivel_2": int(valor2),
            "codigo_dataset_nivel_1": codigo1,
            "codigo_dataset_nivel_2": codigo2,
            "pixeles": int(cantidad),
        })

for nivel, contador in CONTEOS_COMUNES_HOMOLOGADOS.items():
    if sum(contador.values()) != total_comparado:
        raise RuntimeError(f"{nivel}: no se conservaron los conteos.")

print("\nVALIDACIÓN DE JERARQUÍA CON CÓDIGOS HOMOLOGADOS")
print("Píxeles comparados:", total_comparado)
print("Píxeles discrepantes:", discrepancias)
print("Conteos conservados: True")


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_12_homologacion_documentada_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 12,
            "fecha": datetime.now().astimezone().isoformat(),
            "procedencia_declarada": (
                "Mapa oficial de coberturas IDEAM 2022, "
                "recortado a la zona de estudio, según el usuario."
            ),
            "evidencia": (
                "Capturas image(4).png e image(5).png: "
                "campos Value y nivel_1/nivel_2."
            ),
            "correspondencias": registros_correspondencia,
            "mapas_referencia_a_dataset": MAPAS_REFERENCIA_A_DATASET,
            "conteos_comunes_homologados": CONTEOS_COMUNES_HOMOLOGADOS,
            "pixeles_comparados": total_comparado,
            "pixeles_discrepantes": discrepancias,
            "detalle_discrepancias": detalle_discrepancias,
            "estado": (
                "Correspondencia registrada y validada sobre el cruce "
                "de referencias alineadas; todavía no escrita en TIFF."
            ),
            "limitaciones": [
                "La superficie común sigue siendo preliminar.",
                "La coherencia jerárquica no demuestra exactitud temática.",
                "La referencia de 2022 no acredita por sí sola coberturas de 2025.",
                "Sigue pendiente la correspondencia del entrenamiento de U-Net.",
            ],
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:")
print(ruta_registro)

if discrepancias:
    print("\nHay discrepancias pendientes. No generar referencias finales.")
else:
    print("\nJerarquía coherente en todos los píxeles comunes revisados.")


nivel_1
Value de referencia → código del dataset:
{0: 0, 1: 2, 2: 1, 3: 4, 4: 5, 5: 3}

Soporte en la superficie común preliminar:
  Código del dataset 1: 8,674,615
  Código del dataset 2: 3,374,666
  Código del dataset 3: 174,659
  Código del dataset 4: 87,395
  Código del dataset 5: 8,586
Clases sin referencia: []

nivel_2
Value de referencia → código del dataset:
{0: 0, 1: 2, 2: 1, 3: 4, 4: 6, 5: 8, 6: 11, 7: 3, 8: 9, 9: 7, 10: 5, 11: 12, 12: 13}

Soporte en la superficie común preliminar:
  Código del dataset 1: 5,339,526
  Código del dataset 2: 2,509,303
  Código del dataset 3: 838,692
  Código del dataset 4: 3,128,559
  Código del dataset 5: 174,659
  Código del dataset 6: 64,148
  Código del dataset 7: 4,817
  Código del dataset 8: 8,586
  Código del dataset 9: 18,430
  Código del dataset 10: 0
  Código del dataset 11: 145,835
  Código del dataset 12: 60,695
  Código del dataset 13: 26,671
Clases sin referencia: [10]

VALIDACIÓN DE JERARQUÍA CON CÓDIGOS HOMOLOGADOS
Píxeles comp

In [17]:
# ============================================================
# CELDA 13 — GUARDAR Y VERIFICAR REFERENCIAS HOMOLOGADAS
# Conserva los originales. No modifica predicciones ni pesos.
# ============================================================

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import rasterio

for variable in (
    "MAPAS_REFERENCIA_A_DATASET",
    "CARPETA_AUDITORIA",
    "discrepancias",
):
    if variable not in globals():
        raise RuntimeError(f"Falta {variable}. Ejecuta la celda 12.")

if discrepancias != 0:
    raise RuntimeError("La validación jerárquica tiene discrepancias.")


CLASES_COMUNES = {
    "nivel_1": {
        0: "NoData",
        1: "2. Territorios agrícolas",
        2: "3. Bosques y áreas seminaturales",
        3: "5. Superficies de agua",
        4: "1. Territorios artificializados",
        5: "4. Áreas húmedas",
    },
    "nivel_2": {
        0: "NoData",
        1: "2.4 Áreas agrícolas heterogéneas",
        2: "3.2 Áreas con vegetación herbácea o arbustiva",
        3: "3.1 Bosques",
        4: "2.3 Pastos",
        5: "5.1 Aguas continentales",
        6: "1.1 Zonas urbanizadas",
        7: "1.3 Zonas de extracción minera y escombreras",
        8: "4.1 Áreas húmedas continentales",
        9: "1.2 Zonas industriales o comerciales y redes de comunicación",
        10: "1.4 Zonas verdes artificializadas no agrícolas",
        11: "2.1 Cultivos transitorios",
        12: "2.2 Cultivos permanentes",
        13: "3.3 Áreas abiertas sin o con poca vegetación",
    },
}

BASE = Path(r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC")

ORIGINALES = {
    "nivel_1": BASE / "Raster_Almeidas_NIVEL_1_ALINEADO.tif",
    "nivel_2": BASE / "Raster_Almeidas_NIVEL_2_ALINEADO.tif",
}

# Validar ambas referencias antes de crear los archivos.
preparados = {}

for nivel, ruta in ORIGINALES.items():
    with rasterio.open(ruta) as src:
        if src.count != 1 or src.nodata != 0:
            raise RuntimeError(f"{nivel}: banda o NoData inesperados.")

        original = src.read(1)
        mascara = src.read_masks(1)
        perfil = src.profile.copy()

    mapa = MAPAS_REFERENCIA_A_DATASET[nivel]
    desconocidos = set(np.unique(original).tolist()) - set(mapa)

    if desconocidos:
        raise RuntimeError(
            f"{nivel}: códigos sin correspondencia: {desconocidos}"
        )

    if len(set(mapa.values())) != len(mapa):
        raise RuntimeError(f"{nivel}: el mapa fusionaría clases.")

    if not set(mapa.values()).issubset(CLASES_COMUNES[nivel]):
        raise RuntimeError(f"{nivel}: destinos fuera del catálogo.")

    # Tabla de consulta: evita remapeos encadenados sobre el mismo array.
    tabla = np.zeros(256, dtype=np.uint8)
    for origen, destino in mapa.items():
        tabla[origen] = destino

    homologada = tabla[original]

    # Comprobar reversibilidad: cada píxel conserva su clase de origen.
    inversa = np.zeros(256, dtype=np.uint8)
    for origen, destino in mapa.items():
        inversa[destino] = origen

    if not np.array_equal(inversa[homologada], original):
        raise RuntimeError(f"{nivel}: falló la verificación inversa.")

    preparados[nivel] = {
        "datos": homologada,
        "mascara": mascara,
        "perfil": perfil,
    }


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
CARPETA_REFERENCIAS_HOMOLOGADAS = (
    Path(CARPETA_AUDITORIA)
    / f"Referencias_homologadas_{marca}"
)
CARPETA_REFERENCIAS_HOMOLOGADAS.mkdir(exist_ok=False)

REFERENCIAS_HOMOLOGADAS = {}
VERIFICACION_REFERENCIAS = {}

for nivel, preparado in preparados.items():

    salida = (
        CARPETA_REFERENCIAS_HOMOLOGADAS
        / f"Referencia_IDEAM_2022_{nivel}_codigos_dataset.tif"
    )

    perfil = preparado["perfil"].copy()
    perfil.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=0,
        compress="LZW",
    )

    with rasterio.open(salida, "w", **perfil) as dst:
        dst.write(preparado["datos"], 1)
        dst.write_mask(preparado["mascara"])
        dst.set_band_description(1, f"Clases del dataset — {nivel}")
        dst.update_tags(
            fuente_raster=str(ORIGINALES[nivel]),
            fuente_temporal="IDEAM 2022, según procedencia declarada",
            metodo="Homologación por tabla de atributos; sin remuestreo",
        )
        dst.update_tags(
            1,
            clases_json=json.dumps(
                CLASES_COMUNES[nivel], ensure_ascii=False
            ),
            correspondencia_json=json.dumps(
                MAPAS_REFERENCIA_A_DATASET[nivel]
            ),
        )

    with rasterio.open(salida) as comprobacion:
        verificaciones = {
            "valores_correctos": np.array_equal(
                comprobacion.read(1), preparado["datos"]
            ),
            "mascara_conservada": np.array_equal(
                comprobacion.read_masks(1), preparado["mascara"]
            ),
            "transformacion_conservada": (
                comprobacion.transform == preparado["perfil"]["transform"]
            ),
            "crs_conservado": (
                comprobacion.crs == preparado["perfil"]["crs"]
            ),
            "dimensiones_conservadas": (
                comprobacion.width == preparado["perfil"]["width"]
                and comprobacion.height == preparado["perfil"]["height"]
            ),
            "nodata_conservado": comprobacion.nodata == 0,
        }

    VERIFICACION_REFERENCIAS[nivel] = verificaciones

    if not all(verificaciones.values()):
        raise RuntimeError(
            f"{nivel}: falló la verificación: {verificaciones}. "
            "No usar los archivos de esta ejecución."
        )

    REFERENCIAS_HOMOLOGADAS[nivel] = str(salida)

    print("\n" + "=" * 75)
    print(nivel)
    print("=" * 75)
    for nombre, resultado in verificaciones.items():
        print(f"{nombre}: {resultado}")
    print("Archivo generado:", salida)


registro = {
    "celda": 13,
    "fecha": datetime.now().astimezone().isoformat(),
    "referencias_originales": {
        nivel: str(ruta) for nivel, ruta in ORIGINALES.items()
    },
    "referencias_homologadas": REFERENCIAS_HOMOLOGADAS,
    "catalogo": CLASES_COMUNES,
    "correspondencias": MAPAS_REFERENCIA_A_DATASET,
    "verificacion": VERIFICACION_REFERENCIAS,
    "correccion_ejecutada": (
        "Creación de copias de las referencias con códigos del dataset "
        "y catálogo de nombres, mediante correspondencia documental."
    ),
    "alcance": (
        "Referencias completas homologadas. "
        "Todavía no constituyen una selección independiente de evaluación."
    ),
}

ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_13_referencias_homologadas_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(registro, archivo, ensure_ascii=False, indent=2)

# Libera los arrays; conserva rutas y resultados para las próximas celdas.
del preparados, preparado, original, homologada, mascara

print("\nRegistro guardado en:", ruta_registro)
print("\nCELDA 13 TERMINADA. Referencias nuevas verificadas.")
print("Los originales, las predicciones y los pesos se conservaron.")


nivel_1
valores_correctos: True
mascara_conservada: True
transformacion_conservada: True
crs_conservado: True
dimensiones_conservadas: True
nodata_conservado: True
Archivo generado: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Referencias_homologadas_20261007_220856_013511\Referencia_IDEAM_2022_nivel_1_codigos_dataset.tif

nivel_2
valores_correctos: True
mascara_conservada: True
transformacion_conservada: True
crs_conservado: True
dimensiones_conservadas: True
nodata_conservado: True
Archivo generado: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Referencias_homologadas_20261007_220856_013511\Referencia_IDEAM_2022_nivel_2_codigos_dataset.tif

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_13_referencias_homologadas_20261007_220856_013511.json

CELDA 13 TERMINADA. Referencias nuevas verificadas.
Los originales, las predicciones y los pesos se conservaron.


In [18]:
# ============================================================
# CELDA 14 — PÍXELES ÚNICOS FUERA DE ENTRENAMIENTO/VALIDACIÓN
# Diagnóstico espacial. No calcula métricas de los modelos.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import math

import numpy as np
import rasterio
from rasterio.crs import CRS as RasterioCRS

for variable in (
    "REFERENCIAS_HOMOLOGADAS",
    "REVISION_RASTERS",
    "CLASES_COMUNES",
    "CARPETA_AUDITORIA",
    "crs_equivalentes_xy",
):
    if variable not in globals():
        raise RuntimeError(f"Falta {variable}. Ejecuta las celdas anteriores.")

RUTA_IMAGEN = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\zona_estudio.tif"
)

PARTICIONES_EXCLUIDAS = [
    "nivel_1_train", "nivel_1_val",
    "nivel_2_train", "nivel_2_val",
]

with rasterio.open(RUTA_IMAGEN) as src:
    alto, ancho = src.height, src.width
    transformacion = src.transform
    crs_base = src.crs

    if (
        transformacion.b != 0 or transformacion.d != 0
        or transformacion.a <= 0 or transformacion.e >= 0
    ):
        raise RuntimeError("La cuadrícula no tiene la orientación esperada.")

    # Validez de todas las bandas, sin cargar la imagen completa a la vez.
    comun = np.ones((alto, ancho), dtype=bool)

    for banda in src.indexes:
        datos = src.read(banda)
        valida = (src.read_masks(banda) > 0) & np.isfinite(datos)

        nodata = src.nodatavals[banda - 1]
        if nodata is not None:
            valida &= datos != nodata

        comun &= valida

    del datos, valida


referencias = {}

for nivel, ruta in REFERENCIAS_HOMOLOGADAS.items():
    with rasterio.open(ruta) as src:
        if not (
            src.width == ancho
            and src.height == alto
            and src.transform == transformacion
            and crs_equivalentes_xy(src.crs, crs_base)
        ):
            raise RuntimeError(f"{nivel}: geometría incompatible.")

        datos = src.read(1)
        valida = (src.read_masks(1) > 0) & (datos != 0)

        desconocidos = (
            set(np.unique(datos[valida]).tolist())
            - set(CLASES_COMUNES[nivel])
        )
        if desconocidos:
            raise RuntimeError(f"{nivel}: códigos desconocidos {desconocidos}")

        referencias[nivel] = datos
        comun &= valida

if int(comun.sum()) != 12319921:
    raise RuntimeError(
        f"La superficie común cambió: {int(comun.sum())} píxeles. "
        "Revisar antes de continuar."
    )


# Unión de las huellas: cada píxel se cuenta como máximo una vez.
exclusion = np.zeros((alto, ancho), dtype=bool)
resumen_huellas = {}
cache_crs = {}

for particion in PARTICIONES_EXCLUIDAS:
    dentro = 0
    fuera = 0

    for registro in REVISION_RASTERS[particion]:
        if registro.get("problemas"):
            raise RuntimeError(
                f"Incidencia pendiente: {registro['chip_id']}"
            )

        meta = registro["imagen"]
        texto_crs = meta["crs"]

        if texto_crs not in cache_crs:
            cache_crs[texto_crs] = crs_equivalentes_xy(
                RasterioCRS.from_user_input(texto_crs),
                crs_base
            )

        if not cache_crs[texto_crs]:
            raise RuntimeError(
                f"{particion}: CRS incompatible con la zona de estudio."
            )

        a, b, c, d, e, f = meta["transformacion"]
        if b != 0 or d != 0:
            raise RuntimeError("Se encontró un parche con rotación.")

        izquierda, abajo, derecha, arriba = meta["limites"]

        # Intersección territorial con la imagen completa.
        x0 = max(izquierda, transformacion.c)
        x1 = min(derecha, transformacion.c + ancho * transformacion.a)
        y0 = max(abajo, transformacion.f + alto * transformacion.e)
        y1 = min(arriba, transformacion.f)

        if x1 <= x0 or y1 <= y0:
            fuera += 1
            continue

        dentro += 1

        # Criterio conservador: excluye celdas con intersección positiva.
        # No requiere que el centro del píxel caiga dentro del parche.
        col0 = max(0, math.floor((x0 - transformacion.c) / transformacion.a))
        col1 = min(ancho, math.ceil((x1 - transformacion.c) / transformacion.a))
        fil0 = max(0, math.floor((y1 - transformacion.f) / transformacion.e))
        fil1 = min(alto, math.ceil((y0 - transformacion.f) / transformacion.e))

        exclusion[fil0:fil1, col0:col1] = True

    resumen_huellas[particion] = {
        "parches_intersectan_zona": dentro,
        "parches_fuera_zona": fuera,
    }

    print(particion, resumen_huellas[particion])


MASCARA_COMUN_PRELIMINAR = comun
MASCARA_CANDIDATA_ESPACIAL = comun & ~exclusion

total_comun = int(comun.sum())
total_excluido = int(np.count_nonzero(comun & exclusion))
total_candidato = int(MASCARA_CANDIDATA_ESPACIAL.sum())

if total_comun != total_excluido + total_candidato:
    raise RuntimeError("No se conserva el total de píxeles.")

print("\n" + "=" * 75)
print("PÍXELES ÚNICOS")
print("=" * 75)
print("Superficie común preliminar:", total_comun)
print("Excluidos por entrenamiento/validación:", total_excluido)
print("Candidatos fuera de esas huellas:", total_candidato)

soportes = {}

for nivel, datos in referencias.items():
    soportes[nivel] = {}

    print("\n", nivel)
    for codigo, nombre in CLASES_COMUNES[nivel].items():
        if codigo == 0:
            continue

        cantidad = int(np.count_nonzero(
            MASCARA_CANDIDATA_ESPACIAL & (datos == codigo)
        ))
        soportes[nivel][codigo] = cantidad
        print(f"  {codigo} — {nombre}: {cantidad:,}")


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
carpeta = Path(CARPETA_AUDITORIA) / f"Diagnostico_espacial_{marca}"
carpeta.mkdir(exist_ok=False)

ruta_mascara = carpeta / "Mascara_candidata_espacial_PRELIMINAR.tif"

with rasterio.open(
    ruta_mascara,
    "w",
    driver="GTiff",
    width=ancho,
    height=alto,
    count=1,
    dtype="uint8",
    crs=crs_base,
    transform=transformacion,
    compress="LZW",
    nodata=None,
) as dst:
    dst.write(MASCARA_CANDIDATA_ESPACIAL.astype("uint8"), 1)
    dst.update_tags(
        significado_0="Fuera del conjunto candidato",
        significado_1="Pixel candidato; evaluacion independiente no confirmada",
        criterio="Fuera de huellas train/val de ambos niveles",
    )

with rasterio.open(ruta_mascara) as src:
    if not np.array_equal(
        src.read(1), MASCARA_CANDIDATA_ESPACIAL.astype("uint8")
    ):
        raise RuntimeError("Falló la verificación de la máscara escrita.")

ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_14_disponibilidad_espacial_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 14,
            "fecha": datetime.now().astimezone().isoformat(),
            "huellas": resumen_huellas,
            "pixeles_comunes": total_comun,
            "pixeles_excluidos": total_excluido,
            "pixeles_candidatos": total_candidato,
            "soporte_candidato_por_clase": soportes,
            "mascara_preliminar": str(ruta_mascara),
            "criterio": (
                "Exclusión conservadora de celdas que intersectan "
                "huellas completas de entrenamiento y validación."
            ),
            "limitaciones": [
                "No demuestra independencia espacial ni temporal.",
                "No aplica distancia adicional de separación.",
                "No incluye todavía control de nubes.",
                "No exige pertenencia a los CSV de prueba.",
                "Depende de confirmar qué particiones usó cada modelo.",
                "La versión de entrenamiento de U-Net sigue pendiente.",
            ],
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nMáscara preliminar:", ruta_mascara)
print("Registro:", ruta_registro)
print("\nCELDA 14 TERMINADA. No se modificaron los insumos originales.")

nivel_1_train {'parches_intersectan_zona': 304, 'parches_fuera_zona': 713}
nivel_1_val {'parches_intersectan_zona': 55, 'parches_fuera_zona': 163}
nivel_2_train {'parches_intersectan_zona': 983, 'parches_fuera_zona': 2744}
nivel_2_val {'parches_intersectan_zona': 218, 'parches_fuera_zona': 581}

PÍXELES ÚNICOS
Superficie común preliminar: 12319921
Excluidos por entrenamiento/validación: 11987353
Candidatos fuera de esas huellas: 332568

 nivel_1
  1 — 2. Territorios agrícolas: 219,605
  2 — 3. Bosques y áreas seminaturales: 96,545
  3 — 5. Superficies de agua: 7,500
  4 — 1. Territorios artificializados: 8,918
  5 — 4. Áreas húmedas: 0

 nivel_2
  1 — 2.4 Áreas agrícolas heterogéneas: 136,609
  2 — 3.2 Áreas con vegetación herbácea o arbustiva: 68,908
  3 — 3.1 Bosques: 27,049
  4 — 2.3 Pastos: 68,584
  5 — 5.1 Aguas continentales: 7,500
  6 — 1.1 Zonas urbanizadas: 8,036
  7 — 1.3 Zonas de extracción minera y escombreras: 0
  8 — 4.1 Áreas húmedas continentales: 0
  9 — 1.2 Zonas indu

In [19]:
# ============================================================
# CELDA 15 — LOCALIZAR VERSIONES DEL CÓDIGO DE U-NET
# Solo lectura del código fuente; no ejecuta los archivos.
# ============================================================

from pathlib import Path
from datetime import datetime
import os
import json
import re

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")

RAICES_BUSQUEDA = [
    Path(r"C:\Users\Ivan sua\anaconda3"),
    Path(r"D:\PROYECTO_GRADO_1"),
]

# Se excluyen dependencias, datos masivos y la versión descartada.
CARPETAS_OMITIDAS = {
    "envs", "pkgs", "lib", "libs", "include", "dlls",
    "site-packages", "node_modules", "__pycache__", ".git",
    "dataset", "dataset_clc_final_v3",
    "auditoria_comparacion_modelos",
}

patrones = {
    "capa_final_outc": r"self\.outc\s*=",
    "bloque_net": r"self\.net\s*=",
    "ignore_index_cero": r"\bIGNORE_INDEX\s*=\s*0\b",
    "config_num_classes": r"\bNUM_CLASSES\b",
    "config_class_dict": r"\bCLASS_DICT\b",
    "muestreo_ponderado": r"\bUSE_WEIGHTED_SAMPLER\b",
    "estadisticas": r"\b(means|stds)\b",
}

CANDIDATOS_CODIGO_UNET = []
incidencias_busqueda = []
archivos_revisados = 0
archivos_grandes_omitidos = []


def registrar_error_recorrido(error):
    incidencias_busqueda.append(str(error))


for raiz in RAICES_BUSQUEDA:
    if not raiz.is_dir():
        incidencias_busqueda.append(f"Carpeta no encontrada: {raiz}")
        continue

    print("\nBuscando en:", raiz, flush=True)

    for carpeta, subcarpetas, archivos in os.walk(
        raiz,
        followlinks=False,
        onerror=registrar_error_recorrido,
    ):
        subcarpetas[:] = [
            nombre for nombre in subcarpetas
            if nombre.lower() not in CARPETAS_OMITIDAS
        ]

        for nombre in archivos:
            ruta = Path(carpeta) / nombre

            if ruta.suffix.lower() not in {".py", ".ipynb"}:
                continue

            try:
                # Evita cargar cuadernos enormes con imágenes incrustadas.
                if ruta.stat().st_size > 50_000_000:
                    archivos_grandes_omitidos.append(str(ruta))
                    continue

                texto = ruta.read_text(
                    encoding="utf-8-sig",
                    errors="replace"
                )

                if ruta.suffix.lower() == ".ipynb":
                    cuaderno = json.loads(texto)
                    fragmentos = []

                    for celda in cuaderno.get("cells", []):
                        if celda.get("cell_type") != "code":
                            continue
                        fuente = celda.get("source", [])
                        fragmentos.append(
                            fuente if isinstance(fuente, str)
                            else "".join(fuente)
                        )

                    texto = "\n\n".join(fragmentos)

                archivos_revisados += 1

                # Detecta U-Net por contenido o por nombre.
                if not re.search(
                    r"u[_-]?net",
                    texto + "\n" + ruta.name,
                    flags=re.IGNORECASE,
                ):
                    continue

                indicadores = {
                    etiqueta: bool(re.search(patron, texto))
                    for etiqueta, patron in patrones.items()
                }

                # Prioriza firmas estructurales de los pesos revisados.
                prioridad = (
                    4 * indicadores["capa_final_outc"]
                    + 3 * indicadores["bloque_net"]
                    + 2 * indicadores["ignore_index_cero"]
                    + indicadores["config_class_dict"]
                )

                lineas_relevantes = [
                    linea.strip()[:250]
                    for linea in texto.splitlines()
                    if re.search(
                        r"self\.outc\s*=|self\.net\s*=|"
                        r"IGNORE_INDEX\s*=|NUM_CLASSES\s*=|"
                        r"DATASET_ROOT\s*=|USE_WEIGHTED_SAMPLER\s*=|"
                        r"def\s+(read_reflectance|__getitem__|"
                        r"compute_image_stats)\b",
                        linea,
                    )
                ][:20]

                CANDIDATOS_CODIGO_UNET.append({
                    "ruta": str(ruta),
                    "prioridad_busqueda": int(prioridad),
                    "indicadores": indicadores,
                    "fragmentos": lineas_relevantes,
                })

            except Exception as error:
                incidencias_busqueda.append(
                    f"{ruta}: {type(error).__name__}: {error}"
                )


CANDIDATOS_CODIGO_UNET.sort(
    key=lambda item: (-item["prioridad_busqueda"], item["ruta"])
)

print("\n" + "=" * 75)
print("RESULTADOS")
print("=" * 75)
print("Archivos fuente revisados:", archivos_revisados)
print("Candidatos relacionados con U-Net:", len(CANDIDATOS_CODIGO_UNET))
print("Archivos grandes omitidos:", len(archivos_grandes_omitidos))
print("Incidencias de lectura:", len(incidencias_busqueda))

for numero, candidato in enumerate(CANDIDATOS_CODIGO_UNET[:15], start=1):
    print(f"\n{numero}. {candidato['ruta']}")
    print("Indicadores:", candidato["indicadores"])
    for linea in candidato["fragmentos"]:
        print(" ", linea)

for ruta in archivos_grandes_omitidos[:10]:
    print("\nCuaderno/archivo grande no leído:", ruta)

for error in incidencias_busqueda[:5]:
    print("\nIncidencia:", error)


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_15_busqueda_codigo_unet_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 15,
            "fecha": datetime.now().astimezone().isoformat(),
            "raices": [str(ruta) for ruta in RAICES_BUSQUEDA],
            "archivos_revisados": archivos_revisados,
            "candidatos": CANDIDATOS_CODIGO_UNET,
            "archivos_grandes_omitidos": archivos_grandes_omitidos,
            "incidencias": incidencias_busqueda,
            "alcance": (
                "Búsqueda por firmas del código. Encontrar una firma "
                "no acredita por sí solo que ese archivo generó los pesos. "
                "No se ejecutó código encontrado ni se modificaron archivos fuente."
            ),
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:", ruta_registro)
print("\nCELDA 15 TERMINADA.")


Buscando en: C:\Users\Ivan sua\anaconda3

Buscando en: D:\PROYECTO_GRADO_1

RESULTADOS
Archivos fuente revisados: 231
Candidatos relacionados con U-Net: 0
Archivos grandes omitidos: 0
Incidencias de lectura: 0

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_15_busqueda_codigo_unet_20261007_221419_786762.json

CELDA 15 TERMINADA.


In [20]:
# ============================================================
# CELDA 16 — CARGA ESTRICTA DE LOS DOS MODELOS U-NET
# Arquitectura: primera celda de PREDICCION_FINAL(1).ipynb.
# Sin entrenamiento ni modificación de los pesos.
# ============================================================

from pathlib import Path
from datetime import datetime
import gc
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")

CARPETAS_UNET = {
    "nivel_1": Path(r"C:\Users\Ivan sua\anaconda3\Modelos_UNET\Nivel 1"),
    "nivel_2": Path(r"C:\Users\Ivan sua\anaconda3\Modelos_UNET\Nivel 2"),
}


class DoubleConvAuditada(nn.Module):
    def __init__(self, entrada, salida, dropout=0.0):
        super().__init__()
        capas = [
            nn.Conv2d(entrada, salida, 3, padding=1),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            capas.append(nn.Dropout2d(dropout))
        capas.extend([
            nn.Conv2d(salida, salida, 3, padding=1),
            nn.ReLU(inplace=True),
        ])
        self.net = nn.Sequential(*capas)

    def forward(self, x):
        return self.net(x)


class DownAuditada(nn.Module):
    def __init__(self, entrada, salida, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConvAuditada(entrada, salida, dropout),
        )

    def forward(self, x):
        return self.net(x)


class UpAuditada(nn.Module):
    def __init__(self, entrada, salida):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            entrada, entrada // 2, kernel_size=2, stride=2
        )
        self.conv = DoubleConvAuditada(entrada, salida)

    def forward(self, decoder, encoder):
        decoder = self.up(decoder)

        dy = encoder.size(2) - decoder.size(2)
        dx = encoder.size(3) - decoder.size(3)

        if dx != 0 or dy != 0:
            decoder = F.pad(
                decoder,
                [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2],
            )

        return self.conv(torch.cat([encoder, decoder], dim=1))


class UNetAuditada(nn.Module):
    def __init__(self, canales, clases, base, dropout_bottleneck):
        super().__init__()
        b = base

        self.inc = DoubleConvAuditada(canales, b)
        self.down1 = DownAuditada(b, b * 2)
        self.down2 = DownAuditada(b * 2, b * 4)
        self.down3 = DownAuditada(b * 4, b * 8)
        self.down4 = DownAuditada(b * 8, b * 16, dropout_bottleneck)

        self.up1 = UpAuditada(b * 16, b * 8)
        self.up2 = UpAuditada(b * 8, b * 4)
        self.up3 = UpAuditada(b * 4, b * 2)
        self.up4 = UpAuditada(b * 2, b)

        self.outc = nn.Conv2d(b, clases, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        return self.outc(x)


REVISION_CARGA_UNET = {}

for nivel, carpeta in CARPETAS_UNET.items():

    print("\n" + "=" * 75)
    print(nivel)
    print("=" * 75)

    ruta_pesos = carpeta / "best_model.pt"
    ruta_config = carpeta / "config.json"

    resultado = {
        "pesos": str(ruta_pesos),
        "configuracion": str(ruta_config),
        "estado": "pendiente",
    }

    checkpoint = estado = modelo = entrada = salida = None

    try:
        with ruta_config.open("r", encoding="utf-8-sig") as archivo:
            config = json.load(archivo)

        checkpoint = torch.load(
            ruta_pesos,
            map_location="cpu",
            weights_only=True,
        )

        if "model_state_dict" not in checkpoint:
            raise RuntimeError(
                "El checkpoint no contiene model_state_dict."
            )

        estado = checkpoint["model_state_dict"]

        # No renombramos ni descartamos claves del checkpoint.
        canales = int(estado["inc.net.0.weight"].shape[1])
        base = int(estado["inc.net.0.weight"].shape[0])
        clases = int(estado["outc.weight"].shape[0])

        clases_esperadas = 6 if nivel == "nivel_1" else 14

        if canales != 4 or clases != clases_esperadas:
            raise RuntimeError(
                f"Dimensiones inesperadas: canales={canales}, clases={clases}."
            )

        if int(config["NUM_CLASSES"]) != clases:
            raise RuntimeError("NUM_CLASSES no coincide con los pesos.")

        if int(config["BASE_CHANNELS"]) != base:
            raise RuntimeError("BASE_CHANNELS no coincide con los pesos.")

        if bool(config["BILINEAR"]):
            raise RuntimeError("Esta arquitectura requiere BILINEAR=False.")

        if float(config["DROPOUT"]) != 0:
            raise RuntimeError(
                "El dropout general difiere del cuaderno recuperado."
            )

        dropout_bottleneck = (
            float(config["BOTTLENECK_DROPOUT"])
            if bool(config["USE_DROPOUT_BOTTLENECK"])
            else 0.0
        )

        # Conserva el estado del generador aleatorio del cuaderno.
        with torch.random.fork_rng(devices=[]):
            modelo = UNetAuditada(
                canales, clases, base, dropout_bottleneck
            ).cpu()

        esperado = modelo.state_dict()

        faltantes = sorted(set(esperado) - set(estado))
        inesperadas = sorted(set(estado) - set(esperado))
        formas_distintas = {
            clave: {
                "arquitectura": list(esperado[clave].shape),
                "checkpoint": list(estado[clave].shape),
            }
            for clave in set(esperado) & set(estado)
            if esperado[clave].shape != estado[clave].shape
        }

        resultado.update({
            "canales_entrada": canales,
            "canales_salida": clases,
            "base_channels": base,
            "claves_faltantes": faltantes,
            "claves_inesperadas": inesperadas,
            "formas_distintas": formas_distintas,
        })

        print("Canales de entrada:", canales)
        print("Canales de salida:", clases)
        print("Claves faltantes:", faltantes)
        print("Claves inesperadas:", inesperadas)
        print("Formas incompatibles:", formas_distintas)

        if faltantes or inesperadas or formas_distintas:
            raise RuntimeError("La arquitectura no coincide con los pesos.")

        modelo.load_state_dict(estado, strict=True)
        modelo.eval()

        # Comprueba que la copia de todos los tensores fue exacta.
        copia_exacta = all(
            torch.equal(tensor, estado[clave])
            for clave, tensor in modelo.state_dict().items()
        )

        if not copia_exacta:
            raise RuntimeError("La copia de los pesos no fue exacta.")

        # Entrada sintética: no utiliza imágenes ni etiquetas de prueba.
        entrada = torch.zeros((1, canales, 32, 32), dtype=torch.float32)

        with torch.inference_mode():
            salida = modelo(entrada)

        forma_correcta = tuple(salida.shape) == (1, clases, 32, 32)
        valores_finitos = bool(torch.isfinite(salida).all().item())

        if not forma_correcta or not valores_finitos:
            raise RuntimeError("Falló la prueba de ejecución sintética.")

        resultado.update({
            "estado": "carga_estricta_y_prueba_correctas",
            "copia_exacta_pesos": copia_exacta,
            "forma_salida_sintetica": list(salida.shape),
            "salida_finita": valores_finitos,
            "numero_parametros": sum(
                parametro.numel() for parametro in modelo.parameters()
            ),
        })

        print("Carga estricta: CORRECTA")
        print("Copia exacta de pesos:", copia_exacta)
        print("Salida sintética:", list(salida.shape))
        print("Valores finitos:", valores_finitos)

    except Exception as error:
        resultado["estado"] = "error"
        resultado["error"] = f"{type(error).__name__}: {error}"
        print("ERROR:", resultado["error"])

    finally:
        REVISION_CARGA_UNET[nivel] = resultado
        checkpoint = estado = modelo = entrada = salida = None
        # Libera referencias a los tensores del modelo.
        if "esperado" in globals():
            del esperado
        gc.collect()


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_16_carga_estricta_unet_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 16,
            "fecha": datetime.now().astimezone().isoformat(),
            "fuente_arquitectura": (
                "PREDICCION_FINAL(1).ipynb, primera celda de código."
            ),
            "resultados": REVISION_CARGA_UNET,
            "alcance": (
                "Compatibilidad estructural y ejecución sintética en CPU. "
                "No acredita el preprocesamiento original, la semántica "
                "de los canales ni la identidad del dataset entrenado."
            ),
            "pesos_modificados": False,
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:", ruta_registro)

CARGA_UNET_VERIFICADA = all(
    resultado["estado"] == "carga_estricta_y_prueba_correctas"
    for resultado in REVISION_CARGA_UNET.values()
)

print("CARGA_UNET_VERIFICADA:", CARGA_UNET_VERIFICADA)
print("\nCELDA 16 TERMINADA.")


nivel_1
Canales de entrada: 4
Canales de salida: 6
Claves faltantes: []
Claves inesperadas: []
Formas incompatibles: {}
Carga estricta: CORRECTA
Copia exacta de pesos: True
Salida sintética: [1, 6, 32, 32]
Valores finitos: True

nivel_2
Canales de entrada: 4
Canales de salida: 14
Claves faltantes: []
Claves inesperadas: []
Formas incompatibles: {}
Carga estricta: CORRECTA
Copia exacta de pesos: True
Salida sintética: [1, 14, 32, 32]
Valores finitos: True

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_16_carga_estricta_unet_20261007_232850_645247.json
CARGA_UNET_VERIFICADA: True

CELDA 16 TERMINADA.


In [21]:
# ============================================================
# CELDA 17 — DIAGNÓSTICO RADIOMÉTRICO
# Muestras de entrenamiento y ventanas de la imagen completa.
# Sin inferencia, sin etiquetas de prueba y sin modificar datos.
# ============================================================

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import rasterio
from rasterio.windows import Window

for variable in (
    "FILAS_LOCALES", "CARPETAS_UNET",
    "CARPETA_AUDITORIA", "CARGA_UNET_VERIFICADA",
):
    if variable not in globals():
        raise RuntimeError(f"Falta {variable}.")

if not CARGA_UNET_VERIFICADA:
    raise RuntimeError("Primero debe verificarse la carga de U-Net.")

RUTA_IMAGEN = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\zona_estudio.tif"
)

rng = np.random.default_rng(42)
RESUMEN_RADIOMETRICO = {}


def leer_muestra(ruta, ventana=None):
    with rasterio.open(ruta) as src:
        if src.count != 4:
            raise RuntimeError(f"Se esperaban cuatro bandas: {ruta}")

        datos = src.read(window=ventana).astype(np.float32)
        validos = (
            (src.read_masks(window=ventana) > 0)
            & np.isfinite(datos)
        )

        for banda, nodata in enumerate(src.nodatavals):
            if nodata is not None:
                validos[banda] &= datos[banda] != nodata

        metadatos = {
            "ruta": str(ruta),
            "tipos": list(src.dtypes),
            "nodata": str(src.nodata),
            "descripciones": list(src.descriptions),
            "escalas": list(src.scales),
            "offsets": list(src.offsets),
        }

    # La función antigua decidía con el máximo de todo el array,
    # incluyendo NoData. Registramos también ese comportamiento.
    finitos = datos[np.isfinite(datos)]
    maximo_almacenado = float(finitos.max()) if finitos.size else None

    validos_comunes = validos.all(axis=0)
    valores = datos[:, validos_comunes]

    metadatos["pixeles_validos_comunes"] = int(validos_comunes.sum())
    metadatos["maximo_almacenado"] = maximo_almacenado
    metadatos["division_condicional_se_activaria"] = (
        maximo_almacenado > 2 if maximo_almacenado is not None else None
    )

    return valores, metadatos


def resumir(grupo, muestras, detalles):
    utilizables = [m for m in muestras if m.shape[1] > 0]

    if not utilizables:
        raise RuntimeError(f"{grupo}: no hay píxeles válidos en la muestra.")

    valores = np.concatenate(utilizables, axis=1)
    bandas = {}

    print("\n" + "=" * 75)
    print(grupo)
    print("=" * 75)
    print("Archivos/ventanas revisados:", len(detalles))
    print("Píxeles válidos sumados en la muestra:", valores.shape[1])

    for b in range(4):
        v = valores[b]
        cuantiles = np.percentile(v, [0, 1, 50, 99, 100])

        bandas[b + 1] = {
            "minimo": float(cuantiles[0]),
            "p01": float(cuantiles[1]),
            "mediana": float(cuantiles[2]),
            "p99": float(cuantiles[3]),
            "maximo": float(cuantiles[4]),
            "media_sin_escalar": float(np.mean(v, dtype=np.float64)),
            "media_dividida_10000": float(
                np.mean(v, dtype=np.float64) / 10000
            ),
            "porcentaje_fuera_0_1_sin_escalar": float(
                100 * np.mean((v < 0) | (v > 1))
            ),
            "porcentaje_afectado_por_clip_tras_dividir_10000": float(
                100 * np.mean((v < 0) | (v > 10000))
            ),
        }

        print(f"\nBanda {b + 1}")
        for clave, valor in bandas[b + 1].items():
            print(f"  {clave}: {valor:.8f}")

    RESUMEN_RADIOMETRICO[grupo] = {
        "bandas": bandas,
        "detalle_muestras": detalles,
        "pixeles_validos_sumados": int(valores.shape[1]),
    }


# Solo entrenamiento: selección reproducible, sin tocar val/test.
for nivel in ("nivel_1", "nivel_2"):
    filas = FILAS_LOCALES[f"{nivel}_train"]
    indices = rng.choice(
        len(filas), size=min(24, len(filas)), replace=False
    )

    muestras, detalles = [], []

    for indice in sorted(indices.tolist()):
        fila = filas[indice]
        valores, meta = leer_muestra(fila["image_path_local"])
        meta["chip_id"] = fila["chip_id"]
        muestras.append(valores)
        detalles.append(meta)

    resumir(f"{nivel}_train", muestras, detalles)

    with (CARPETAS_UNET[nivel] / "config.json").open(
        "r", encoding="utf-8-sig"
    ) as archivo:
        cfg = json.load(archivo)

    RESUMEN_RADIOMETRICO[f"{nivel}_train"]["estadisticas_guardadas"] = {
        "means": cfg["means"],
        "stds": cfg["stds"],
    }

    print("\nMedias guardadas en el modelo:", cfg["means"])
    print("Desviaciones guardadas en el modelo:", cfg["stds"])


# 25 ventanas distribuidas sobre la imagen, sin remuestreo.
muestras, detalles = [], []

with rasterio.open(RUTA_IMAGEN) as src:
    tamano = min(128, src.width, src.height)
    filas_ventanas = np.unique(
        np.linspace(0, src.height - tamano, 5).astype(int)
    )
    columnas_ventanas = np.unique(
        np.linspace(0, src.width - tamano, 5).astype(int)
    )

for fila in filas_ventanas:
    for columna in columnas_ventanas:
        ventana = Window(int(columna), int(fila), tamano, tamano)
        valores, meta = leer_muestra(RUTA_IMAGEN, ventana)
        meta["ventana"] = [int(columna), int(fila), tamano, tamano]
        muestras.append(valores)
        detalles.append(meta)

resumir("imagen_completa_muestreada", muestras, detalles)


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_17_diagnostico_radiometrico_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 17,
            "fecha": datetime.now().astimezone().isoformat(),
            "resultados": RESUMEN_RADIOMETRICO,
            "alcance": [
                "Diagnóstico muestral de valores almacenados.",
                "No se aplicaron escalas ni offsets de metadatos.",
                "No se recalcularon estadísticas para normalizar modelos.",
                "No se utilizaron predicciones ni métricas para elegir variantes.",
                "Las muestras pueden repetir territorio entre parches.",
                "Los valores radiométricos no prueban el orden espectral.",
                "No acredita por sí solo el preprocesamiento original de U-Net.",
            ],
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:", ruta_registro)
print("\nCELDA 17 TERMINADA. No se modificaron archivos originales.")


nivel_1_train
Archivos/ventanas revisados: 24
Píxeles válidos sumados en la muestra: 1531956

Banda 1
  minimo: 1.00000000
  p01: 1099.00000000
  mediana: 1281.00000000
  p99: 2171.00000000
  maximo: 7414.00000000
  media_sin_escalar: 1319.71113400
  media_dividida_10000: 0.13197111
  porcentaje_fuera_0_1_sin_escalar: 99.99993472
  porcentaje_afectado_por_clip_tras_dividir_10000: 0.00000000

Banda 2
  minimo: 5.00000000
  p01: 1168.00000000
  mediana: 1464.00000000
  p99: 2394.00000000
  maximo: 8867.00000000
  media_sin_escalar: 1513.08975780
  media_dividida_10000: 0.15130898
  porcentaje_fuera_0_1_sin_escalar: 100.00000000
  porcentaje_afectado_por_clip_tras_dividir_10000: 0.00000000

Banda 3
  minimo: 5.00000000
  p01: 1102.00000000
  mediana: 1394.00000000
  p99: 2588.00000000
  maximo: 8656.00000000
  media_sin_escalar: 1469.71232790
  media_dividida_10000: 0.14697123
  porcentaje_fuera_0_1_sin_escalar: 100.00000000
  porcentaje_afectado_por_clip_tras_dividir_10000: 0.00000000



In [22]:
# ============================================================
# CELDA 18 — DISPOSITIVO Y REGISTRO DEL PREPROCESAMIENTO
# Sin carga de modelos, entrenamiento ni predicción de imágenes.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import torch

if "CARPETA_AUDITORIA" not in globals():
    raise RuntimeError("No se encontró CARPETA_AUDITORIA.")

if not globals().get("CARGA_UNET_VERIFICADA", False):
    raise RuntimeError("Primero debe terminar correctamente la celda 16.")


REVISION_DISPOSITIVO = {
    "torch": str(torch.__version__),
    "cuda_compilacion": torch.version.cuda,
    "hip_compilacion": getattr(torch.version, "hip", None),
    "cuda_disponible": bool(torch.cuda.is_available()),
    "hilos_cpu": torch.get_num_threads(),
    "dispositivos": [],
    "prueba_gpu_correcta": False,
}

DISPOSITIVO_INFERENCIA = torch.device("cpu")

if REVISION_DISPOSITIVO["cuda_disponible"]:

    for indice in range(torch.cuda.device_count()):
        propiedades = torch.cuda.get_device_properties(indice)

        REVISION_DISPOSITIVO["dispositivos"].append({
            "indice": indice,
            "nombre": propiedades.name,
            "memoria_total_GiB": (
                propiedades.total_memory / (1024 ** 3)
            ),
        })

    a = b = None

    try:
        # Prueba mínima de cálculo: no es una prueba de memoria de U-Net.
        a = torch.ones((32, 32), device="cuda:0")
        b = a @ a
        torch.cuda.synchronize()

        correcto = bool(torch.all(b == 32).item())

        if not correcto:
            raise RuntimeError("La prueba numérica de GPU no fue correcta.")

        REVISION_DISPOSITIVO["prueba_gpu_correcta"] = True
        DISPOSITIVO_INFERENCIA = torch.device("cuda:0")

    except Exception as error:
        REVISION_DISPOSITIVO["error_gpu"] = (
            f"{type(error).__name__}: {error}"
        )

    finally:
        a = b = None
        torch.cuda.empty_cache()


REVISION_DISPOSITIVO["dispositivo_seleccionado"] = str(
    DISPOSITIVO_INFERENCIA
)

ESTADO_PREPROCESAMIENTO_UNET = {
    "fuente": "PREDICCION_FINAL(1).ipynb, primera celda de código",
    "estado": "Implementación de predicción recuperada; provisional",
    "divisor": 10000.0,
    "recorte_0_1": False,
    "normalizacion": "(x - means) / (stds + 1e-6)",
    "estadisticas": "Las guardadas en cada checkpoint/configuración",
    "recalcular_estadisticas": False,
    "tratamiento_nodata": (
        "Detectar antes del escalado. Definir explícitamente "
        "su tratamiento en la entrada del modelo y en la salida."
    ),
    "evidencia_celda_17": (
        "La escala DN/10000 es compatible con las medias guardadas "
        "en la muestra revisada. El clip modifica algunos valores "
        "válidos de la imagen de predicción."
    ),
    "pendientes": [
        "Confirmar el preprocesamiento original del entrenamiento de U-Net.",
        "Confirmar documentalmente el orden B2, B3, B4, B8 de zona_estudio.tif.",
        "Confirmar procedencia radiométrica y posibles offsets del producto.",
        "Completar el tratamiento común de calidad y nubes.",
    ],
}

print("=" * 75)
print("DISPOSITIVO")
print("=" * 75)
print(json.dumps(
    REVISION_DISPOSITIVO,
    ensure_ascii=False,
    indent=2
))

print("\n" + "=" * 75)
print("ESTADO DEL PREPROCESAMIENTO DE U-NET")
print("=" * 75)
print(json.dumps(
    ESTADO_PREPROCESAMIENTO_UNET,
    ensure_ascii=False,
    indent=2
))

marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_18_dispositivo_y_preprocesamiento_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 18,
            "fecha": datetime.now().astimezone().isoformat(),
            "dispositivo": REVISION_DISPOSITIVO,
            "preprocesamiento_unet": ESTADO_PREPROCESAMIENTO_UNET,
            "alcance": (
                "La prueba mínima de GPU no garantiza memoria suficiente "
                "para los modelos completos. No se generaron predicciones."
            ),
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:", ruta_registro)
print("\nCELDA 18 TERMINADA.")

DISPOSITIVO
{
  "torch": "2.8.0+cu126",
  "cuda_compilacion": "12.6",
  "hip_compilacion": null,
  "cuda_disponible": false,
  "hilos_cpu": 6,
  "dispositivos": [],
  "prueba_gpu_correcta": false,
  "dispositivo_seleccionado": "cpu"
}

ESTADO DEL PREPROCESAMIENTO DE U-NET
{
  "fuente": "PREDICCION_FINAL(1).ipynb, primera celda de código",
  "estado": "Implementación de predicción recuperada; provisional",
  "divisor": 10000.0,
  "recorte_0_1": false,
  "normalizacion": "(x - means) / (stds + 1e-6)",
  "estadisticas": "Las guardadas en cada checkpoint/configuración",
  "recalcular_estadisticas": false,
  "tratamiento_nodata": "Detectar antes del escalado. Definir explícitamente su tratamiento en la entrada del modelo y en la salida.",
  "evidencia_celda_17": "La escala DN/10000 es compatible con las medias guardadas en la muestra revisada. El clip modifica algunos valores válidos de la imagen de predicción.",
  "pendientes": [
    "Confirmar el preprocesamiento original del entrenamient

In [23]:
# ============================================================
# CELDA 19 — PRUEBA TÉCNICA DE U-NET CON UNA VENTANA REAL
# Sin métricas ni mapas finales. Preprocesamiento provisional.
# ============================================================

from pathlib import Path
from datetime import datetime
from time import perf_counter
import gc
import json

import numpy as np
import rasterio
from rasterio.windows import Window
import torch

for variable in (
    "UNetAuditada", "CARPETAS_UNET", "CARPETA_AUDITORIA",
    "CARGA_UNET_VERIFICADA",
):
    if variable not in globals():
        raise RuntimeError(f"Falta {variable}.")

if not CARGA_UNET_VERIFICADA:
    raise RuntimeError("La carga estricta debe estar verificada.")

RUTA_IMAGEN = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\zona_estudio.tif"
)

TAMANO_PRUEBA = 128
mejor_fraccion = -1.0
datos_prueba = validez_prueba = ventana_prueba = None

# Selección determinista por validez de imagen.
# No utiliza etiquetas, predicciones ni desempeño.
with rasterio.open(RUTA_IMAGEN) as src:
    if src.count != 4:
        raise RuntimeError("Se esperaban cuatro bandas.")

    for fila in range(0, src.height - TAMANO_PRUEBA + 1, TAMANO_PRUEBA):
        for columna in range(
            0, src.width - TAMANO_PRUEBA + 1, TAMANO_PRUEBA
        ):
            ventana = Window(
                columna, fila, TAMANO_PRUEBA, TAMANO_PRUEBA
            )
            datos = src.read(window=ventana).astype(np.float32)
            validos = (
                (src.read_masks(window=ventana) > 0)
                & np.isfinite(datos)
            )

            for banda, nodata in enumerate(src.nodatavals):
                if nodata is not None:
                    validos[banda] &= datos[banda] != nodata

            validos = validos.all(axis=0)
            fraccion = float(validos.mean())

            if fraccion > mejor_fraccion:
                mejor_fraccion = fraccion
                datos_prueba = datos.copy()
                validez_prueba = validos.copy()
                ventana_prueba = [
                    columna, fila, TAMANO_PRUEBA, TAMANO_PRUEBA
                ]

            if mejor_fraccion >= 0.99:
                break

        if mejor_fraccion >= 0.99:
            break

if datos_prueba is None or mejor_fraccion <= 0:
    raise RuntimeError("No se encontró una ventana con píxeles válidos.")

print("Ventana [columna, fila, ancho, alto]:", ventana_prueba)
print("Fracción válida:", mejor_fraccion)

PRUEBA_REAL_UNET = {}

for nivel, carpeta in CARPETAS_UNET.items():
    print("\nProcesando prueba:", nivel, flush=True)

    with (carpeta / "config.json").open(
        "r", encoding="utf-8-sig"
    ) as archivo:
        cfg = json.load(archivo)

    medias = np.asarray(cfg["means"], dtype=np.float32)
    desv = np.asarray(cfg["stds"], dtype=np.float32)

    if (
        medias.shape != (4,) or desv.shape != (4,)
        or not np.isfinite(medias).all()
        or not np.isfinite(desv).all()
        or np.any(desv <= 0)
    ):
        raise RuntimeError(f"{nivel}: estadísticas inválidas.")

    # Fórmula recuperada del predictor; no se aplica clip.
    entrada_np = datos_prueba / 10000.0
    entrada_np = (
        entrada_np - medias[:, None, None]
    ) / (desv[:, None, None] + 1e-6)

    # Política explícita para esta prueba:
    # cero normalizado en píxeles sin información válida.
    entrada_np[:, ~validez_prueba] = 0.0

    if not np.isfinite(entrada_np).all():
        raise RuntimeError(f"{nivel}: entrada normalizada no finita.")

    checkpoint = torch.load(
        carpeta / "best_model.pt",
        map_location="cpu",
        weights_only=True,
    )

    dropout = (
        float(cfg["BOTTLENECK_DROPOUT"])
        if cfg["USE_DROPOUT_BOTTLENECK"] else 0.0
    )

    with torch.random.fork_rng(devices=[]):
        modelo = UNetAuditada(
            4, int(cfg["NUM_CLASSES"]),
            int(cfg["BASE_CHANNELS"]), dropout
        ).cpu()

    modelo.load_state_dict(checkpoint["model_state_dict"], strict=True)
    del checkpoint
    modelo.eval()

    entrada = torch.from_numpy(
        np.ascontiguousarray(entrada_np)
    ).unsqueeze(0)

    with torch.inference_mode():
        inicio = perf_counter()
        salida = modelo(entrada)
        segundos = perf_counter() - inicio

        if not torch.isfinite(salida).all().item():
            raise RuntimeError(f"{nivel}: salida no finita.")

        forma_esperada = (
            1, int(cfg["NUM_CLASSES"]), TAMANO_PRUEBA, TAMANO_PRUEBA
        )
        if tuple(salida.shape) != forma_esperada:
            raise RuntimeError(f"{nivel}: dimensiones de salida incorrectas.")

        indices = salida.argmax(dim=1)[0].cpu().numpy()

    valores, cantidades = np.unique(
        indices[validez_prueba], return_counts=True
    )

    resultado = {
        "forma_salida": list(salida.shape),
        "segundos_primera_ejecucion_cpu": segundos,
        "indices_salida_en_pixeles_validos": {
            int(valor): int(cantidad)
            for valor, cantidad in zip(valores, cantidades)
        },
        "salida_finita": True,
    }
    PRUEBA_REAL_UNET[nivel] = resultado

    print(json.dumps(resultado, ensure_ascii=False, indent=2))

    del modelo, entrada, salida, indices
    gc.collect()


marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
ruta_registro = (
    Path(CARPETA_AUDITORIA)
    / f"celda_19_prueba_real_unet_{marca}.json"
)

with ruta_registro.open("x", encoding="utf-8") as archivo:
    json.dump(
        {
            "celda": 19,
            "fecha": datetime.now().astimezone().isoformat(),
            "imagen": str(RUTA_IMAGEN),
            "ventana": ventana_prueba,
            "fraccion_valida": mejor_fraccion,
            "resultados": PRUEBA_REAL_UNET,
            "procedencia_declarada": (
                "Descarga desde plataforma Sentinel y recorte. "
                "El usuario solo conserva el TIFF recortado."
            ),
            "supuestos": {
                "orden_bandas_provisional": ["B2", "B3", "B4", "B8"],
                "preprocesamiento": "DN/10000; z-score con std+1e-6; sin clip",
                "relleno_invalido": "Cero después de normalizar",
            },
            "limitaciones": [
                "Prueba de funcionamiento, no de exactitud.",
                "Orden espectral y offsets originales sin verificar.",
                "Preprocesamiento original de entrenamiento pendiente.",
                "Tiempo de primera ejecución; no extrapolar directamente al mapa.",
                "Tamaño de ventana exclusivo de esta prueba.",
                "No se asignan nombres semánticos a los índices en este diagnóstico.",
            ],
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )

print("\nRegistro guardado en:", ruta_registro)
print("\nCELDA 19 TERMINADA. No se generaron mapas finales.")

Ventana [columna, fila, ancho, alto]: [2816, 640, 128, 128]
Fracción válida: 0.9935302734375

Procesando prueba: nivel_1
{
  "forma_salida": [
    1,
    6,
    128,
    128
  ],
  "segundos_primera_ejecucion_cpu": 0.11898120000114432,
  "indices_salida_en_pixeles_validos": {
    "1": 16250,
    "4": 28
  },
  "salida_finita": true
}

Procesando prueba: nivel_2
{
  "forma_salida": [
    1,
    14,
    128,
    128
  ],
  "segundos_primera_ejecucion_cpu": 0.08686649999799556,
  "indices_salida_en_pixeles_validos": {
    "1": 16223,
    "4": 12,
    "6": 32,
    "7": 1,
    "11": 10
  },
  "salida_finita": true
}

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_19_prueba_real_unet_20261007_234213_619292.json

CELDA 19 TERMINADA. No se generaron mapas finales.


In [24]:
# ============================================================
# CELDA 20 — RECONSTRUCCIÓN Y CARGA ESTRICTA DE RESNET50
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


CARPETA_AUDITORIA = Path(
    r"D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos"
    r"\20261007_191328_200990"
)

CARPETAS_RESNET = {
    "nivel_1": Path(
        r"C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion"
        r"\ResNet50Seg_nivel_1_20260730_191354"
    ),
    "nivel_2": Path(
        r"C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion"
        r"\ResNet50Seg_nivel_2_20260803_201945"
    ),
}

if not CARPETA_AUDITORIA.is_dir():
    raise FileNotFoundError(CARPETA_AUDITORIA)

CARGA_RESNET_VERIFICADA = False
CONFIGS_RESNET_AUDITADAS = {}


# ------------------------------------------------------------
# Arquitectura reproducida del código de entrenamiento
# ------------------------------------------------------------

class ConvNormActResAuditada(nn.Module):
    def __init__(self, entrada, salida, dropout=0.0):
        super().__init__()

        grupos = min(32, salida)
        while salida % grupos != 0:
            grupos -= 1

        capas = [
            nn.Conv2d(entrada, salida, 3, padding=1, bias=False),
            nn.GroupNorm(grupos, salida),
            nn.ReLU(inplace=True),

            nn.Conv2d(salida, salida, 3, padding=1, bias=False),
            nn.GroupNorm(grupos, salida),
            nn.ReLU(inplace=True),
        ]

        if dropout > 0:
            capas.insert(3, nn.Dropout2d(dropout))

        self.block = nn.Sequential(*capas)

    def forward(self, x):
        return self.block(x)


class DecoderResAuditado(nn.Module):
    def __init__(self, entrada, salto, salida, dropout=0.0):
        super().__init__()
        self.conv = ConvNormActResAuditada(
            entrada + salto, salida, dropout
        )

    def forward(self, x, salto):
        x = F.interpolate(
            x,
            size=salto.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([x, salto], dim=1))


def reemplazar_bn_res_auditada(modulo):
    """Reproduce la conversión a GroupNorm del entrenamiento."""
    for nombre, hijo in list(modulo.named_children()):
        if isinstance(hijo, nn.BatchNorm2d):
            grupos = next(
                g for g in (32, 16, 8, 4, 2, 1)
                if g <= hijo.num_features and hijo.num_features % g == 0
            )

            nuevo = nn.GroupNorm(
                grupos,
                hijo.num_features,
                eps=hijo.eps,
                affine=hijo.affine,
            )

            if hijo.affine:
                with torch.no_grad():
                    nuevo.weight.copy_(hijo.weight)
                    nuevo.bias.copy_(hijo.bias)

            setattr(modulo, nombre, nuevo)
        else:
            reemplazar_bn_res_auditada(hijo)


class ResNet50Auditada(nn.Module):
    def __init__(self, entrada, clases, dropout):
        super().__init__()

        # Todos los parámetros se cargarán desde tu checkpoint.
        encoder = models.resnet50(weights=None)

        anterior = encoder.conv1
        encoder.conv1 = nn.Conv2d(
            entrada,
            anterior.out_channels,
            kernel_size=anterior.kernel_size,
            stride=anterior.stride,
            padding=anterior.padding,
            bias=False,
        )

        reemplazar_bn_res_auditada(encoder)

        self.conv1 = encoder.conv1
        self.bn1 = encoder.bn1
        self.relu = encoder.relu
        self.maxpool = encoder.maxpool

        self.layer1 = encoder.layer1
        self.layer2 = encoder.layer2
        self.layer3 = encoder.layer3
        self.layer4 = encoder.layer4

        self.decoder4 = DecoderResAuditado(2048, 1024, 512, dropout)
        self.decoder3 = DecoderResAuditado(512, 512, 256, dropout)
        self.decoder2 = DecoderResAuditado(256, 256, 128, dropout)
        self.decoder1 = DecoderResAuditado(128, 64, 64, dropout)

        self.final_refine = ConvNormActResAuditada(64, 32, dropout)
        self.segmentation_head = nn.Conv2d(32, clases, 1)

    def forward(self, x):
        dimensiones = x.shape[-2:]

        salto0 = self.relu(self.bn1(self.conv1(x)))
        salto1 = self.layer1(self.maxpool(salto0))
        salto2 = self.layer2(salto1)
        salto3 = self.layer3(salto2)
        x = self.layer4(salto3)

        x = self.decoder4(x, salto3)
        x = self.decoder3(x, salto2)
        x = self.decoder2(x, salto1)
        x = self.decoder1(x, salto0)

        x = F.interpolate(
            x,
            size=dimensiones,
            mode="bilinear",
            align_corners=False,
        )

        return self.segmentation_head(self.final_refine(x))


# ------------------------------------------------------------
# Verificación de los dos checkpoints
# ------------------------------------------------------------

registro_20 = {
    "celda": 20,
    "fecha": datetime.now().isoformat(),
    "fuente_arquitectura": "PREDICCION_RESNET50(1).ipynb",
    "dispositivo": "cpu",
    "niveles": {},
}

ruta_registro_20 = CARPETA_AUDITORIA / (
    "celda_20_carga_estricta_resnet50_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)

try:
    for nivel, carpeta in CARPETAS_RESNET.items():
        print("\n" + "=" * 75)
        print(nivel)
        print("=" * 75)

        detalle = {}
        registro_20["niveles"][nivel] = detalle

        with (carpeta / "config.json").open(
            encoding="utf-8-sig"
        ) as archivo:
            config = json.load(archivo)

        checkpoint = torch.load(
            carpeta / "best_model.pt",
            map_location="cpu",
            weights_only=True,
        )

        estado = checkpoint["model_state_dict"]

        entrada = int(estado["conv1.weight"].shape[1])
        clases = int(estado["segmentation_head.weight"].shape[0])
        esperadas = 5 if nivel == "nivel_1" else 13

        if entrada != 4 or clases != esperadas:
            raise RuntimeError(
                f"{nivel}: canales inesperados: {entrada} → {clases}"
            )

        if int(config["num_classes"]) != clases:
            raise RuntimeError("La configuración y el checkpoint discrepan.")

        idx_to_code = {
            int(indice): int(codigo)
            for indice, codigo in config["idx_to_code"].items()
        }

        if (
            set(idx_to_code) != set(range(clases))
            or set(idx_to_code.values()) != set(range(1, clases + 1))
        ):
            raise RuntimeError(
                f"Correspondencia de clases incompleta: {idx_to_code}"
            )

        modelo = ResNet50Auditada(
            entrada=entrada,
            clases=clases,
            dropout=float(config["decoder_dropout"]),
        ).cpu()

        esperado = modelo.state_dict()

        faltantes = sorted(set(esperado) - set(estado))
        inesperadas = sorted(set(estado) - set(esperado))
        incompatibles = {
            clave: {
                "modelo": list(esperado[clave].shape),
                "checkpoint": list(estado[clave].shape),
            }
            for clave in set(esperado) & set(estado)
            if esperado[clave].shape != estado[clave].shape
        }

        detalle.update({
            "canales_entrada": entrada,
            "canales_salida": clases,
            "claves_faltantes": faltantes,
            "claves_inesperadas": inesperadas,
            "formas_incompatibles": incompatibles,
            "idx_to_code": idx_to_code,
        })

        print("Canales de entrada:", entrada)
        print("Canales de salida:", clases)
        print("Claves faltantes:", faltantes)
        print("Claves inesperadas:", inesperadas)
        print("Formas incompatibles:", incompatibles)

        if faltantes or inesperadas or incompatibles:
            raise RuntimeError("La arquitectura no coincide con los pesos.")

        modelo.load_state_dict(estado, strict=True)
        modelo.eval()

        copia_exacta = all(
            torch.equal(valor, estado[clave])
            for clave, valor in modelo.state_dict().items()
        )

        if not copia_exacta:
            raise RuntimeError("La copia de los pesos no es exacta.")

        with torch.inference_mode():
            salida = modelo(torch.zeros(1, entrada, 32, 32))

        finita = bool(torch.isfinite(salida).all().item())

        if list(salida.shape) != [1, clases, 32, 32] or not finita:
            raise RuntimeError("La prueba sintética no es correcta.")

        detalle.update({
            "carga_estricta": True,
            "copia_exacta_pesos": copia_exacta,
            "salida_sintetica": list(salida.shape),
            "valores_finitos": finita,
        })

        CONFIGS_RESNET_AUDITADAS[nivel] = config

        print("Carga estricta: CORRECTA")
        print("Copia exacta de pesos:", copia_exacta)
        print("Salida sintética:", list(salida.shape))
        print("Valores finitos:", finita)
        print("Índice de salida → código del dataset:", idx_to_code)

        del modelo, esperado, estado, checkpoint, salida
        gc.collect()

    CARGA_RESNET_VERIFICADA = True

except Exception as error:
    registro_20["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    registro_20["carga_resnet_verificada"] = CARGA_RESNET_VERIFICADA

    with ruta_registro_20.open("x", encoding="utf-8") as archivo:
        json.dump(registro_20, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro guardado en:", ruta_registro_20)

print("CARGA_RESNET_VERIFICADA:", CARGA_RESNET_VERIFICADA)
print("\nCELDA 20 TERMINADA. No se generaron mapas finales.")


nivel_1
Canales de entrada: 4
Canales de salida: 5
Claves faltantes: []
Claves inesperadas: []
Formas incompatibles: {}
Carga estricta: CORRECTA
Copia exacta de pesos: True
Salida sintética: [1, 5, 32, 32]
Valores finitos: True
Índice de salida → código del dataset: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}

nivel_2
Canales de entrada: 4
Canales de salida: 13
Claves faltantes: []
Claves inesperadas: []
Formas incompatibles: {}
Carga estricta: CORRECTA
Copia exacta de pesos: True
Salida sintética: [1, 13, 32, 32]
Valores finitos: True
Índice de salida → código del dataset: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10, 10: 11, 11: 12, 12: 13}

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_20_carga_estricta_resnet50_20261007_234755_541515.json
CARGA_RESNET_VERIFICADA: True

CELDA 20 TERMINADA. No se generaron mapas finales.


In [25]:
# ============================================================
# CELDA 21 — PRUEBA REAL DE RESNET50 EN LA VENTANA DE LA CELDA 19
# ============================================================

import json
import gc
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import Window
import torch


if not globals().get("CARGA_RESNET_VERIFICADA", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 20.")

RUTA_IMAGEN_PRUEBA = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO"
    r"\Raster_CLC\zona_estudio.tif"
)

# Misma ventana reportada en la celda 19:
VENTANA_RESNET = Window(
    col_off=2816,
    row_off=640,
    width=128,
    height=128,
)

registro_21 = {
    "celda": 21,
    "fecha": datetime.now().isoformat(),
    "imagen": str(RUTA_IMAGEN_PRUEBA),
    "ventana_col_fila_ancho_alto": [2816, 640, 128, 128],
    "dispositivo": "cpu",
    "finalidad": "Prueba técnica; no evaluación de exactitud",
    "orden_espectral_tiff": "Pendiente de confirmación documental",
    "tratamiento_invalido": "Entrada normalizada igual a cero",
    "niveles": {},
}

ruta_registro_21 = CARPETA_AUDITORIA / (
    "celda_21_prueba_real_resnet50_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)

PRUEBA_REAL_RESNET_VERIFICADA = False
validez_comparacion = None


def preparar_entrada_resnet(config):
    """Reproduce el escalado y la normalización del entrenamiento."""
    bandas = [int(b) for b in config["band_indexes"]]

    if bandas != [1, 2, 3, 4]:
        raise RuntimeError(f"Orden de lectura inesperado: {bandas}")

    with rasterio.open(RUTA_IMAGEN_PRUEBA) as src:
        datos = src.read(
            indexes=bandas,
            window=VENTANA_RESNET,
        ).astype(np.float32)

        mascaras = src.read_masks(
            indexes=bandas,
            window=VENTANA_RESNET,
        ) > 0

        nodata = src.nodata
        escalas = np.asarray(
            [src.scales[b - 1] for b in bandas],
            dtype=np.float32,
        )[:, None, None]
        offsets = np.asarray(
            [src.offsets[b - 1] for b in bandas],
            dtype=np.float32,
        )[:, None, None]

    if datos.shape != (4, 128, 128):
        raise RuntimeError(f"Ventana inesperada: {datos.shape}")

    validos_banda = np.isfinite(datos)
    if nodata is not None and np.isfinite(nodata):
        validos_banda &= datos != nodata

    # Comprueba que la máscara TIFF no exige un tratamiento adicional
    # al empleado por el lector del entrenamiento.
    validez_original = validos_banda.all(axis=0)
    validez_tiff = mascaras.all(axis=0)

    if np.any(validez_original & ~validez_tiff):
        raise RuntimeError(
            "Existen píxeles excluidos por la máscara TIFF que el lector "
            "original consideraría válidos. Debemos revisar su tratamiento."
        )

    datos = np.where(validos_banda, datos, np.nan)
    valores = datos[np.isfinite(datos)]

    if valores.size == 0:
        raise RuntimeError("La ventana no contiene datos válidos.")

    usar_transformacion = bool(config["use_raster_scale_offset"]) and (
        not np.allclose(escalas, 1.0)
        or not np.allclose(offsets, 0.0)
    )

    if usar_transformacion:
        datos = datos * escalas + offsets
        escalado_aplicado = "Escalas y offsets del TIFF"

    elif (
        bool(config["auto_scale_reflectance"])
        and float(np.percentile(valores, 99.5)) > 2.0
    ):
        divisor = float(config["reflectance_scale"])
        offset = float(config["reflectance_add_offset"])

        if divisor <= 0:
            raise RuntimeError("Divisor radiométrico no válido.")

        datos = (datos + offset) / divisor
        escalado_aplicado = f"(DN + {offset}) / {divisor}"

    else:
        escalado_aplicado = "Sin escalado adicional"

    afectados_clip = (
        ((datos < 0) | (datos > 1)).any(axis=0)
        & validez_original
    )

    datos = np.clip(datos, 0.0, 1.0)

    medias = np.asarray(config["means"], dtype=np.float32)
    desv = np.asarray(config["stds"], dtype=np.float32)

    if (
        medias.shape != (4,)
        or desv.shape != (4,)
        or not np.isfinite(medias).all()
        or not np.isfinite(desv).all()
        or np.any(desv <= 0)
    ):
        raise RuntimeError("Estadísticas de normalización no válidas.")

    datos = np.nan_to_num(datos, nan=0.0)
    datos = (
        datos - medias[:, None, None]
    ) / np.maximum(desv[:, None, None], 1e-6)

    # Sin referencia temática: la validez depende de la imagen.
    datos[:, ~validez_original] = 0.0

    if not np.isfinite(datos).all():
        raise RuntimeError("La entrada normalizada contiene valores no finitos.")

    tensor = torch.from_numpy(
        np.ascontiguousarray(datos)
    ).unsqueeze(0)

    detalle = {
        "escalado_aplicado": escalado_aplicado,
        "recorte_0_1": True,
        "pixeles_validos_afectados_por_clip": int(afectados_clip.sum()),
        "medias": medias.tolist(),
        "desviaciones": desv.tolist(),
        "normalizacion": "(x - means) / maximum(stds, 1e-6)",
    }

    return tensor, validez_original, detalle


try:
    for nivel, carpeta in CARPETAS_RESNET.items():
        print("\n" + "=" * 75)
        print("Procesando prueba:", nivel)
        print("=" * 75)

        config = CONFIGS_RESNET_AUDITADAS[nivel]
        entrada, validos, preprocesamiento = preparar_entrada_resnet(config)

        # La celda 19 reportó exactamente 16.278 píxeles válidos.
        if int(validos.sum()) != 16278:
            raise RuntimeError(
                f"Validez distinta de la celda 19: {int(validos.sum())}"
            )

        if validez_comparacion is None:
            validez_comparacion = validos.copy()
        elif not np.array_equal(validos, validez_comparacion):
            raise RuntimeError("Los dos niveles no usan la misma máscara.")

        checkpoint = torch.load(
            carpeta / "best_model.pt",
            map_location="cpu",
            weights_only=True,
        )

        modelo = ResNet50Auditada(
            entrada=4,
            clases=int(config["num_classes"]),
            dropout=float(config["decoder_dropout"]),
        ).cpu()

        modelo.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True,
        )
        modelo.eval()

        with torch.inference_mode():
            inicio = time.perf_counter()
            salida = modelo(entrada)
            segundos = time.perf_counter() - inicio

        forma_esperada = [1, int(config["num_classes"]), 128, 128]
        finita = bool(torch.isfinite(salida).all().item())

        if list(salida.shape) != forma_esperada or not finita:
            raise RuntimeError("Salida con forma o valores incorrectos.")

        indices = salida.argmax(dim=1)[0].cpu().numpy()

        correspondencia = {
            int(i): int(c) for i, c in config["idx_to_code"].items()
        }
        tabla_codigos = np.asarray(
            [correspondencia[i] for i in range(int(config["num_classes"]))],
            dtype=np.uint8,
        )
        codigos = tabla_codigos[indices]

        def contar(array):
            valores, cantidades = np.unique(
                array[validos], return_counts=True
            )
            return {
                int(v): int(n)
                for v, n in zip(valores, cantidades)
            }

        resultado = {
            "forma_salida": list(salida.shape),
            "pixeles_validos": int(validos.sum()),
            "fraccion_valida": float(validos.mean()),
            "segundos_primera_ejecucion_cpu": segundos,
            "indices_salida_en_pixeles_validos": contar(indices),
            "codigos_dataset_en_pixeles_validos": contar(codigos),
            "salida_finita": finita,
            "preprocesamiento": preprocesamiento,
        }

        registro_21["niveles"][nivel] = resultado
        print(json.dumps(resultado, ensure_ascii=False, indent=2))

        del modelo, checkpoint, entrada, salida, indices, codigos
        gc.collect()

    PRUEBA_REAL_RESNET_VERIFICADA = True

except Exception as error:
    registro_21["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    registro_21["prueba_correcta"] = PRUEBA_REAL_RESNET_VERIFICADA

    with ruta_registro_21.open("x", encoding="utf-8") as archivo:
        json.dump(registro_21, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro guardado en:", ruta_registro_21)

print("\nCELDA 21 TERMINADA. No se generaron mapas finales.")


Procesando prueba: nivel_1
{
  "forma_salida": [
    1,
    5,
    128,
    128
  ],
  "pixeles_validos": 16278,
  "fraccion_valida": 0.9935302734375,
  "segundos_primera_ejecucion_cpu": 0.0744914000024437,
  "indices_salida_en_pixeles_validos": {
    "0": 15923,
    "1": 328,
    "3": 27
  },
  "codigos_dataset_en_pixeles_validos": {
    "1": 15923,
    "2": 328,
    "4": 27
  },
  "salida_finita": true,
  "preprocesamiento": {
    "escalado_aplicado": "(DN + 0.0) / 10000.0",
    "recorte_0_1": true,
    "pixeles_validos_afectados_por_clip": 0,
    "medias": [
      0.13287627696990967,
      0.15295307338237762,
      0.14892812073230743,
      0.3427632451057434
    ],
    "desviaciones": [
      0.024314777925610542,
      0.028529347851872444,
      0.035429444164037704,
      0.07958653569221497
    ],
    "normalizacion": "(x - means) / maximum(stds, 1e-6)"
  }
}

Procesando prueba: nivel_2
{
  "forma_salida": [
    1,
    13,
    128,
    128
  ],
  "pixeles_validos": 16278,
 

In [26]:
# ============================================================
# CELDA 22 — RECONSTRUCCIÓN Y CARGA ESTRICTA DE DENSENET121
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


# Componentes ya comprobados contra ambos códigos fuente.
requisitos = [
    "ConvNormActResAuditada",
    "DecoderResAuditado",
    "reemplazar_bn_res_auditada",
    "CARPETA_AUDITORIA",
]

faltan = [nombre for nombre in requisitos if nombre not in globals()]
if faltan:
    raise RuntimeError(
        f"Faltan componentes de la celda 20: {faltan}. "
        "Ejecuta esa celda antes de continuar."
    )

CARPETAS_DENSENET = {
    "nivel_1": Path(
        r"C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion"
        r"\DenseNet121Seg_nivel_1_20260815_122755"
    ),
    "nivel_2": Path(
        r"C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion"
        r"\DenseNet121Seg_nivel_2_20260815_172335"
    ),
}

CARGA_DENSENET_VERIFICADA = False
CONFIGS_DENSENET_AUDITADAS = {}


# ------------------------------------------------------------
# Arquitectura del código de entrenamiento de DenseNet121
# ------------------------------------------------------------

class DenseNet121Auditada(nn.Module):
    def __init__(self, entrada, clases, dropout, memory_efficient):
        super().__init__()

        encoder = models.densenet121(
            weights=None,
            memory_efficient=memory_efficient,
        )

        # Reproduce GroupNorm, utilizado durante el entrenamiento.
        reemplazar_bn_res_auditada(encoder)

        features = encoder.features
        anterior = features.conv0

        features.conv0 = nn.Conv2d(
            entrada,
            anterior.out_channels,
            kernel_size=anterior.kernel_size,
            stride=anterior.stride,
            padding=anterior.padding,
            bias=False,
        )

        self.conv0 = features.conv0
        self.norm0 = features.norm0
        self.relu0 = features.relu0
        self.pool0 = features.pool0

        self.denseblock1 = features.denseblock1
        self.transition1 = features.transition1
        self.denseblock2 = features.denseblock2
        self.transition2 = features.transition2
        self.denseblock3 = features.denseblock3
        self.transition3 = features.transition3
        self.denseblock4 = features.denseblock4
        self.norm5 = features.norm5

        self.decoder4 = DecoderResAuditado(1024, 1024, 512, dropout)
        self.decoder3 = DecoderResAuditado(512, 512, 256, dropout)
        self.decoder2 = DecoderResAuditado(256, 256, 128, dropout)
        self.decoder1 = DecoderResAuditado(128, 64, 64, dropout)

        self.final_refine = ConvNormActResAuditada(64, 32, dropout)
        self.segmentation_head = nn.Conv2d(32, clases, 1)

    def forward(self, x):
        dimensiones = x.shape[-2:]

        salto0 = self.relu0(self.norm0(self.conv0(x)))
        x = self.pool0(salto0)

        salto1 = self.denseblock1(x)
        x = self.transition1(salto1)

        salto2 = self.denseblock2(x)
        x = self.transition2(salto2)

        salto3 = self.denseblock3(x)
        x = self.transition3(salto3)

        x = self.norm5(self.denseblock4(x))
        x = F.relu(x, inplace=True)

        x = self.decoder4(x, salto3)
        x = self.decoder3(x, salto2)
        x = self.decoder2(x, salto1)
        x = self.decoder1(x, salto0)

        x = F.interpolate(
            x,
            size=dimensiones,
            mode="bilinear",
            align_corners=False,
        )

        return self.segmentation_head(self.final_refine(x))


# ------------------------------------------------------------
# Carga y comprobación de los dos niveles
# ------------------------------------------------------------

registro_22 = {
    "celda": 22,
    "fecha": datetime.now().isoformat(),
    "fuente_arquitectura": "PREDICCION_DENSET121(1).ipynb",
    "dispositivo": "cpu",
    "niveles": {},
}

ruta_registro_22 = Path(CARPETA_AUDITORIA) / (
    "celda_22_carga_estricta_densenet121_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)

try:
    for nivel, carpeta in CARPETAS_DENSENET.items():
        print("\n" + "=" * 75)
        print(nivel)
        print("=" * 75)

        detalle = {}
        registro_22["niveles"][nivel] = detalle

        with (carpeta / "config.json").open(
            encoding="utf-8-sig"
        ) as archivo:
            config = json.load(archivo)

        checkpoint = torch.load(
            carpeta / "best_model.pt",
            map_location="cpu",
            weights_only=True,
        )

        estado = checkpoint["model_state_dict"]

        entrada = int(estado["conv0.weight"].shape[1])
        clases = int(estado["segmentation_head.weight"].shape[0])
        esperadas = 5 if nivel == "nivel_1" else 13

        if entrada != 4 or clases != esperadas:
            raise RuntimeError(
                f"{nivel}: canales inesperados: {entrada} → {clases}"
            )

        if int(config["num_classes"]) != clases:
            raise RuntimeError(
                "La configuración y el checkpoint discrepan."
            )

        correspondencia = {
            int(indice): int(codigo)
            for indice, codigo in config["idx_to_code"].items()
        }

        if (
            set(correspondencia) != set(range(clases))
            or set(correspondencia.values()) != set(range(1, clases + 1))
        ):
            raise RuntimeError(
                f"Correspondencia de clases incompleta: {correspondencia}"
            )

        modelo = DenseNet121Auditada(
            entrada=entrada,
            clases=clases,
            dropout=float(config["decoder_dropout"]),
            memory_efficient=bool(config["densenet_memory_efficient"]),
        ).cpu()

        esperado = modelo.state_dict()

        faltantes = sorted(set(esperado) - set(estado))
        inesperadas = sorted(set(estado) - set(esperado))
        incompatibles = {
            clave: {
                "modelo": list(esperado[clave].shape),
                "checkpoint": list(estado[clave].shape),
            }
            for clave in set(esperado) & set(estado)
            if esperado[clave].shape != estado[clave].shape
        }

        detalle.update({
            "canales_entrada": entrada,
            "canales_salida": clases,
            "claves_faltantes": faltantes,
            "claves_inesperadas": inesperadas,
            "formas_incompatibles": incompatibles,
            "idx_to_code": correspondencia,
            "memory_efficient": bool(config["densenet_memory_efficient"]),
        })

        print("Canales de entrada:", entrada)
        print("Canales de salida:", clases)
        print("Claves faltantes:", faltantes)
        print("Claves inesperadas:", inesperadas)
        print("Formas incompatibles:", incompatibles)

        if faltantes or inesperadas or incompatibles:
            raise RuntimeError(
                "La arquitectura no coincide con los pesos."
            )

        modelo.load_state_dict(estado, strict=True)
        modelo.eval()

        copia_exacta = all(
            torch.equal(valor, estado[clave])
            for clave, valor in modelo.state_dict().items()
        )

        if not copia_exacta:
            raise RuntimeError("La copia de los pesos no es exacta.")

        with torch.inference_mode():
            salida = modelo(torch.zeros(1, entrada, 32, 32))

        finita = bool(torch.isfinite(salida).all().item())

        if list(salida.shape) != [1, clases, 32, 32] or not finita:
            raise RuntimeError("La prueba sintética no es correcta.")

        detalle.update({
            "carga_estricta": True,
            "copia_exacta_pesos": copia_exacta,
            "salida_sintetica": list(salida.shape),
            "valores_finitos": finita,
        })

        CONFIGS_DENSENET_AUDITADAS[nivel] = config

        print("Carga estricta: CORRECTA")
        print("Copia exacta de pesos:", copia_exacta)
        print("Salida sintética:", list(salida.shape))
        print("Valores finitos:", finita)
        print("Índice de salida → código del dataset:", correspondencia)

        del modelo, esperado, estado, checkpoint, salida
        gc.collect()

    CARGA_DENSENET_VERIFICADA = True

except Exception as error:
    registro_22["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    registro_22["carga_densenet_verificada"] = CARGA_DENSENET_VERIFICADA

    with ruta_registro_22.open("x", encoding="utf-8") as archivo:
        json.dump(registro_22, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro guardado en:", ruta_registro_22)

print("CARGA_DENSENET_VERIFICADA:", CARGA_DENSENET_VERIFICADA)
print("\nCELDA 22 TERMINADA. No se generaron mapas finales.")


nivel_1
Canales de entrada: 4
Canales de salida: 5
Claves faltantes: []
Claves inesperadas: []
Formas incompatibles: {}
Carga estricta: CORRECTA
Copia exacta de pesos: True
Salida sintética: [1, 5, 32, 32]
Valores finitos: True
Índice de salida → código del dataset: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}

nivel_2
Canales de entrada: 4
Canales de salida: 13
Claves faltantes: []
Claves inesperadas: []
Formas incompatibles: {}
Carga estricta: CORRECTA
Copia exacta de pesos: True
Salida sintética: [1, 13, 32, 32]
Valores finitos: True
Índice de salida → código del dataset: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10, 10: 11, 11: 12, 12: 13}

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_22_carga_estricta_densenet121_20261007_235133_419958.json
CARGA_DENSENET_VERIFICADA: True

CELDA 22 TERMINADA. No se generaron mapas finales.


In [27]:
# ============================================================
# CELDA 23 — PRUEBA REAL DE DENSENET121
# ============================================================

import json
import gc
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import torch


requisitos = [
    "DenseNet121Auditada",
    "CARPETAS_DENSENET",
    "CONFIGS_DENSENET_AUDITADAS",
    "preparar_entrada_resnet",
    "validez_comparacion",
    "RUTA_IMAGEN_PRUEBA",
    "VENTANA_RESNET",
    "CARPETA_AUDITORIA",
]

faltan = [nombre for nombre in requisitos if nombre not in globals()]
if faltan:
    raise RuntimeError(f"Faltan variables de las celdas anteriores: {faltan}")

if not globals().get("CARGA_DENSENET_VERIFICADA", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 22.")

if not globals().get("PRUEBA_REAL_RESNET_VERIFICADA", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 21.")

ventana_actual = [
    int(VENTANA_RESNET.col_off),
    int(VENTANA_RESNET.row_off),
    int(VENTANA_RESNET.width),
    int(VENTANA_RESNET.height),
]

if ventana_actual != [2816, 640, 128, 128]:
    raise RuntimeError("La ventana cambió respecto a las pruebas anteriores.")

# Las operaciones coinciden en ambos códigos de entrenamiento.
# La configuración que recibirá la función será la de DenseNet121.
preparar_entrada_densenet = preparar_entrada_resnet

PRUEBA_REAL_DENSENET_VERIFICADA = False

registro_23 = {
    "celda": 23,
    "fecha": datetime.now().isoformat(),
    "fuente_preprocesamiento": "PREDICCION_DENSET121(1).ipynb",
    "imagen": str(RUTA_IMAGEN_PRUEBA),
    "ventana_col_fila_ancho_alto": ventana_actual,
    "dispositivo": "cpu",
    "finalidad": "Prueba técnica; no evaluación de exactitud",
    "orden_espectral_tiff": "Pendiente de confirmación documental",
    "tratamiento_invalido": "Entrada normalizada igual a cero",
    "niveles": {},
}

ruta_registro_23 = Path(CARPETA_AUDITORIA) / (
    "celda_23_prueba_real_densenet121_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)


def contar_validos_23(array, mascara):
    valores, cantidades = np.unique(
        array[mascara], return_counts=True
    )
    return {
        int(valor): int(cantidad)
        for valor, cantidad in zip(valores, cantidades)
    }


try:
    for nivel, carpeta in CARPETAS_DENSENET.items():
        print("\n" + "=" * 75)
        print("Procesando prueba:", nivel)
        print("=" * 75)

        config = CONFIGS_DENSENET_AUDITADAS[nivel]

        entrada, validos, preprocesamiento = (
            preparar_entrada_densenet(config)
        )

        misma_mascara = np.array_equal(
            validos, validez_comparacion
        )

        if not misma_mascara or int(validos.sum()) != 16278:
            raise RuntimeError(
                "La máscara o el número de píxeles difiere "
                "de la prueba de ResNet50."
            )

        checkpoint = torch.load(
            carpeta / "best_model.pt",
            map_location="cpu",
            weights_only=True,
        )

        modelo = DenseNet121Auditada(
            entrada=4,
            clases=int(config["num_classes"]),
            dropout=float(config["decoder_dropout"]),
            memory_efficient=bool(config["densenet_memory_efficient"]),
        ).cpu()

        modelo.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True,
        )
        modelo.eval()

        with torch.inference_mode():
            inicio = time.perf_counter()
            salida = modelo(entrada)
            segundos = time.perf_counter() - inicio

        forma_esperada = [1, int(config["num_classes"]), 128, 128]
        finita = bool(torch.isfinite(salida).all().item())

        if list(salida.shape) != forma_esperada or not finita:
            raise RuntimeError(
                "La salida tiene dimensiones o valores incorrectos."
            )

        indices = salida.argmax(dim=1)[0].cpu().numpy()

        correspondencia = {
            int(indice): int(codigo)
            for indice, codigo in config["idx_to_code"].items()
        }

        tabla_codigos = np.asarray(
            [
                correspondencia[indice]
                for indice in range(int(config["num_classes"]))
            ],
            dtype=np.uint8,
        )

        codigos = tabla_codigos[indices]

        resultado = {
            "forma_salida": list(salida.shape),
            "pixeles_validos": int(validos.sum()),
            "fraccion_valida": float(validos.mean()),
            "mascara_identica_a_resnet": bool(misma_mascara),
            "segundos_primera_ejecucion_cpu": segundos,
            "indices_salida_en_pixeles_validos": contar_validos_23(
                indices, validos
            ),
            "codigos_dataset_en_pixeles_validos": contar_validos_23(
                codigos, validos
            ),
            "salida_finita": finita,
            "preprocesamiento": preprocesamiento,
        }

        registro_23["niveles"][nivel] = resultado

        print(json.dumps(
            resultado,
            ensure_ascii=False,
            indent=2,
        ))

        del modelo, checkpoint, entrada, salida, indices, codigos
        gc.collect()

    PRUEBA_REAL_DENSENET_VERIFICADA = True

except Exception as error:
    registro_23["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    registro_23["prueba_correcta"] = PRUEBA_REAL_DENSENET_VERIFICADA

    with ruta_registro_23.open("x", encoding="utf-8") as archivo:
        json.dump(
            registro_23,
            archivo,
            ensure_ascii=False,
            indent=2,
        )

    print("\nRegistro guardado en:", ruta_registro_23)

print(
    "PRUEBA_REAL_DENSENET_VERIFICADA:",
    PRUEBA_REAL_DENSENET_VERIFICADA,
)
print("\nCELDA 23 TERMINADA. No se generaron mapas finales.")


Procesando prueba: nivel_1
{
  "forma_salida": [
    1,
    5,
    128,
    128
  ],
  "pixeles_validos": 16278,
  "fraccion_valida": 0.9935302734375,
  "mascara_identica_a_resnet": true,
  "segundos_primera_ejecucion_cpu": 0.06902120000086143,
  "indices_salida_en_pixeles_validos": {
    "0": 16125,
    "1": 153
  },
  "codigos_dataset_en_pixeles_validos": {
    "1": 16125,
    "2": 153
  },
  "salida_finita": true,
  "preprocesamiento": {
    "escalado_aplicado": "(DN + 0.0) / 10000.0",
    "recorte_0_1": true,
    "pixeles_validos_afectados_por_clip": 0,
    "medias": [
      0.13287627696990967,
      0.15295307338237762,
      0.14892812073230743,
      0.3427632451057434
    ],
    "desviaciones": [
      0.024314777925610542,
      0.028529347851872444,
      0.035429444164037704,
      0.07958653569221497
    ],
    "normalizacion": "(x - means) / maximum(stds, 1e-6)"
  }
}

Procesando prueba: nivel_2
{
  "forma_salida": [
    1,
    13,
    128,
    128
  ],
  "pixeles_valido

In [28]:
# ============================================================
# CELDA 24 — PLAN COMÚN DE VENTANAS PARA LA IMAGEN COMPLETA
# ============================================================

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import rasterio


if not (
    globals().get("PRUEBA_REAL_RESNET_VERIFICADA", False)
    and globals().get("PRUEBA_REAL_DENSENET_VERIFICADA", False)
    and globals().get("CARGA_UNET_VERIFICADA", False)
):
    raise RuntimeError(
        "Deben estar completas las comprobaciones anteriores."
    )

RUTA_IMAGEN_INFERENCIA = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO"
    r"\Raster_CLC\zona_estudio.tif"
)

PLAN_VENTANAS_VERIFICADO = False
VENTANAS_COMUNES = {}

PLAN_INFERENCIA = {
    "celda": 24,
    "fecha": datetime.now().isoformat(),
    "imagen": str(RUTA_IMAGEN_INFERENCIA),
    "modelos": ["UNET", "RESNET50", "DENSENET121"],
    "dispositivo": "cpu",
    "batch_size": 1,
    "combinacion": "Promedio uniforme de probabilidades por modelo",
    "decision": "Argmax después de combinar las ventanas",
    "mezclar_modelos": False,
    "remapeo_optimizado_con_referencia": False,
    "postprocesamiento_tematico": False,
    "nodata_salida": 255,
    "salida_unet_0": (
        "Conservar como salida sin clase temática; "
        "contabilizar como error sobre referencia válida"
    ),
    "conteo_metricas": "Un conteo por píxel de la máscara de evaluación",
    "preprocesamiento": {
        "UNET": "Provisional: DN/10000, sin clip, estadísticas guardadas",
        "RESNET50": "Según código de entrenamiento y configuración propia",
        "DENSENET121": "Según código de entrenamiento y configuración propia",
    },
    "pendientes": [
        "Confirmación documental del orden espectral del TIFF",
        "Preprocesamiento original de entrenamiento de U-Net",
        "Procedencia radiométrica completa del producto",
        "Tratamiento de calidad y nubes",
        "Delimitación y alcance final de las máscaras de evaluación",
    ],
    "fuente_metodo_ventanas": (
        "https://docs.monai.io/en/1.4.0/inferers.html"
    ),
    "niveles": {},
}

ruta_registro_24 = Path(CARPETA_AUDITORIA) / (
    "celda_24_plan_ventanas_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)


def posiciones_ventanas(longitud, tamano, paso):
    """Incluye el borde final sin recortar ni rellenar la imagen."""
    if longitud < tamano:
        raise RuntimeError(
            "La dimensión de la imagen es menor que la ventana."
        )

    posiciones = list(range(0, longitud - tamano + 1, paso))
    ultima = longitud - tamano

    if posiciones[-1] != ultima:
        posiciones.append(ultima)

    return posiciones


def cobertura_eje(longitud, posiciones, tamano):
    diferencias = np.zeros(longitud + 1, dtype=np.int32)

    for inicio in posiciones:
        diferencias[inicio] += 1
        diferencias[inicio + tamano] -= 1

    return np.cumsum(diferencias[:-1])


try:
    with rasterio.open(RUTA_IMAGEN_INFERENCIA) as src:
        ancho = src.width
        alto = src.height

        if src.count != 4:
            raise RuntimeError(
                f"Se esperaban 4 bandas y se encontraron {src.count}."
            )

        PLAN_INFERENCIA["cuadricula"] = {
            "ancho": ancho,
            "alto": alto,
            "bandas": src.count,
            "crs_wkt": src.crs.to_wkt() if src.crs else None,
            "transformacion": list(src.transform)[:6],
            "nodata_entrada": src.nodata,
        }

    print("Imagen:", RUTA_IMAGEN_INFERENCIA)
    print("Dimensiones:", ancho, "x", alto)
    print("Píxeles de la cuadrícula:", ancho * alto)

    for nivel, tamano in [("nivel_1", 256), ("nivel_2", 128)]:
        paso = tamano // 2

        # Comprueba los tamaños declarados por ResNet50 y DenseNet121.
        for nombre, configuraciones in [
            ("RESNET50", CONFIGS_RESNET_AUDITADAS),
            ("DENSENET121", CONFIGS_DENSENET_AUDITADAS),
        ]:
            forma = tuple(
                int(v) for v in configuraciones[nivel]["chip_shape"]
            )
            if forma != (tamano, tamano):
                raise RuntimeError(
                    f"{nombre}, {nivel}: chip_shape={forma}; "
                    f"el plan propone {(tamano, tamano)}."
                )

        filas = posiciones_ventanas(alto, tamano, paso)
        columnas = posiciones_ventanas(ancho, tamano, paso)

        cobertura_filas = cobertura_eje(alto, filas, tamano)
        cobertura_columnas = cobertura_eje(ancho, columnas, tamano)

        completa = bool(
            np.all(cobertura_filas > 0)
            and np.all(cobertura_columnas > 0)
        )

        if not completa:
            raise RuntimeError(f"{nivel}: quedan píxeles sin cubrir.")

        # Orden: columna, fila, ancho, alto.
        ventanas = [
            (columna, fila, tamano, tamano)
            for fila in filas
            for columna in columnas
        ]

        if len(ventanas) != len(set(ventanas)):
            raise RuntimeError(f"{nivel}: existen ventanas duplicadas.")

        VENTANAS_COMUNES[nivel] = ventanas

        detalle = {
            "tamano_ventana": tamano,
            "paso": paso,
            "solapamiento_nominal": 0.5,
            "ajuste_borde": (
                "Última ventana anclada al borde; "
                "el solapamiento puede aumentar allí"
            ),
            "posiciones_fila": filas,
            "posiciones_columna": columnas,
            "ventanas_por_modelo": len(ventanas),
            "cobertura_completa": completa,
            "minimo_aportes_geometricos_por_pixel": int(
                cobertura_filas.min() * cobertura_columnas.min()
            ),
            "maximo_aportes_geometricos_por_pixel": int(
                cobertura_filas.max() * cobertura_columnas.max()
            ),
        }

        PLAN_INFERENCIA["niveles"][nivel] = detalle

        print("\n" + "=" * 75)
        print(nivel)
        print("=" * 75)
        print("Tamaño de ventana:", tamano, "x", tamano)
        print("Desplazamiento:", paso)
        print("Ventanas por modelo:", len(ventanas))
        print("Cobertura completa:", completa)
        print("Primera ventana:", ventanas[0])
        print("Última ventana:", ventanas[-1])
        print(
            "Aportes geométricos por píxel:",
            detalle["minimo_aportes_geometricos_por_pixel"],
            "a",
            detalle["maximo_aportes_geometricos_por_pixel"],
        )

    PLAN_VENTANAS_VERIFICADO = True

except Exception as error:
    PLAN_INFERENCIA["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    PLAN_INFERENCIA["plan_ventanas_verificado"] = (
        PLAN_VENTANAS_VERIFICADO
    )

    with ruta_registro_24.open("x", encoding="utf-8") as archivo:
        json.dump(
            PLAN_INFERENCIA,
            archivo,
            ensure_ascii=False,
            indent=2,
        )

    print("\nRegistro guardado en:", ruta_registro_24)

print("\nCELDA 24 TERMINADA. Plan de ventanas verificado.")
print("Todavía no se ejecutó la predicción de la imagen completa.")

Imagen: D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC\zona_estudio.tif
Dimensiones: 4376 x 4999
Píxeles de la cuadrícula: 21875624

nivel_1
Tamaño de ventana: 256 x 256
Desplazamiento: 128
Ventanas por modelo: 1326
Cobertura completa: True
Primera ventana: (0, 0, 256, 256)
Última ventana: (4120, 4743, 256, 256)
Aportes geométricos por píxel: 1 a 9

nivel_2
Tamaño de ventana: 128 x 128
Desplazamiento: 64
Ventanas por modelo: 5304
Cobertura completa: True
Primera ventana: (0, 0, 128, 128)
Última ventana: (4248, 4871, 128, 128)
Aportes geométricos por píxel: 1 a 9

Registro guardado en: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\celda_24_plan_ventanas_20261007_235458_990963.json

CELDA 24 TERMINADA. Plan de ventanas verificado.
Todavía no se ejecutó la predicción de la imagen completa.


In [29]:
# ============================================================
# CELDA 25 — PREDICCIÓN COMPLETA RESNET50, NIVEL 1
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import time
import gc
import shutil

import numpy as np
import rasterio
from rasterio.windows import Window
import torch


if not globals().get("PLAN_VENTANAS_VERIFICADO", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 24.")

if not globals().get("PRUEBA_REAL_RESNET_VERIFICADA", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 21.")


def leer_entrada_inferencia(src, ventana, config):
    """Preprocesamiento documentado para ResNet50 y DenseNet121."""
    bandas = [int(b) for b in config["band_indexes"]]
    if bandas != [1, 2, 3, 4]:
        raise RuntimeError(f"Bandas inesperadas: {bandas}")

    datos = src.read(bandas, window=ventana).astype(np.float32)
    validos_banda = np.isfinite(datos)

    if src.nodata is not None:
        validos_banda &= datos != src.nodata

    validos = validos_banda.all(axis=0)
    mascara_tiff = (src.read_masks(bandas, window=ventana) > 0).all(axis=0)

    if np.any(validos & ~mascara_tiff):
        raise RuntimeError(
            "La máscara TIFF excluye píxeles adicionales. "
            "Se requiere revisar su tratamiento."
        )

    if not validos.any():
        return None, validos

    datos = np.where(validos_banda, datos, np.nan)

    escalas = np.asarray(
        [src.scales[b - 1] for b in bandas], dtype=np.float32
    )[:, None, None]
    offsets = np.asarray(
        [src.offsets[b - 1] for b in bandas], dtype=np.float32
    )[:, None, None]

    usa_metadatos = bool(config["use_raster_scale_offset"]) and (
        not np.allclose(escalas, 1)
        or not np.allclose(offsets, 0)
    )

    if usa_metadatos:
        datos = datos * escalas + offsets
    elif (
        bool(config["auto_scale_reflectance"])
        and np.percentile(datos[np.isfinite(datos)], 99.5) > 2
    ):
        divisor = float(config["reflectance_scale"])
        if divisor <= 0:
            raise RuntimeError("Divisor radiométrico no válido.")

        datos = (
            datos + float(config["reflectance_add_offset"])
        ) / divisor

    datos = np.clip(datos, 0, 1)
    medias = np.asarray(config["means"], dtype=np.float32)[:, None, None]
    desv = np.asarray(config["stds"], dtype=np.float32)[:, None, None]

    if (
        medias.shape != (4, 1, 1)
        or desv.shape != (4, 1, 1)
        or not np.isfinite(medias).all()
        or not np.isfinite(desv).all()
        or np.any(desv <= 0)
    ):
        raise RuntimeError("Estadísticas de normalización no válidas.")

    datos = np.nan_to_num(datos, nan=0.0)
    datos = (datos - medias) / np.maximum(desv, 1e-6)
    datos[:, ~validos] = 0

    if not np.isfinite(datos).all():
        raise RuntimeError("Entrada normalizada no finita.")

    return torch.from_numpy(np.ascontiguousarray(datos)).unsqueeze(0), validos


MODELO_25 = "RESNET50"
NIVEL_25 = "nivel_1"
config_25 = CONFIGS_RESNET_AUDITADAS[NIVEL_25]
clases_25 = int(config_25["num_classes"])

carpeta_25 = Path(CARPETA_AUDITORIA) / (
    "Inferencia_RESNET50_nivel_1_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
carpeta_25.mkdir(exist_ok=False)

ruta_final_25 = carpeta_25 / "RESNET50_nivel_1_prediccion.tif"
ruta_parcial_25 = carpeta_25 / "RESNET50_nivel_1_PARCIAL.tif"
ruta_sumas_25 = carpeta_25 / "sumas_probabilidades_temporal.dat"
ruta_aportes_25 = carpeta_25 / "aportes_temporal.dat"
ruta_registro_25 = carpeta_25 / "celda_25_inferencia_y_verificacion.json"

registro_25 = {
    "celda": 25,
    "modelo": MODELO_25,
    "nivel": NIVEL_25,
    "inicio": datetime.now().isoformat(),
    "plan": PLAN_INFERENCIA,
    "configuracion": config_25,
    "completada": False,
}

sumas_25 = None
aportes_25 = None
modelo_25 = None
inicio_25 = time.perf_counter()

try:
    checkpoint_25 = torch.load(
        CARPETAS_RESNET[NIVEL_25] / "best_model.pt",
        map_location="cpu",
        weights_only=True,
    )

    modelo_25 = ResNet50Auditada(
        entrada=4,
        clases=clases_25,
        dropout=float(config_25["decoder_dropout"]),
    ).cpu()

    modelo_25.load_state_dict(
        checkpoint_25["model_state_dict"], strict=True
    )
    modelo_25.eval()
    del checkpoint_25

    mapa_indices = {
        int(i): int(c) for i, c in config_25["idx_to_code"].items()
    }
    tabla_codigos = np.asarray(
        [mapa_indices[i] for i in range(clases_25)],
        dtype=np.uint8,
    )

    with rasterio.open(RUTA_IMAGEN_INFERENCIA) as src:
        alto, ancho = src.height, src.width

        if (
            ancho != PLAN_INFERENCIA["cuadricula"]["ancho"]
            or alto != PLAN_INFERENCIA["cuadricula"]["alto"]
            or list(src.transform)[:6]
            != PLAN_INFERENCIA["cuadricula"]["transformacion"]
            or src.crs.to_wkt()
            != PLAN_INFERENCIA["cuadricula"]["crs_wkt"]
        ):
            raise RuntimeError("La cuadrícula cambió respecto al plan.")

        espacio_necesario = alto * ancho * (4 * clases_25 + 2)
        if shutil.disk_usage(carpeta_25).free < espacio_necesario + 200_000_000:
            raise RuntimeError("Espacio insuficiente para los acumuladores.")

        # Acumuladores respaldados en disco.
        sumas_25 = np.memmap(
            ruta_sumas_25, mode="w+", dtype="float32",
            shape=(clases_25, alto, ancho),
        )
        aportes_25 = np.memmap(
            ruta_aportes_25, mode="w+", dtype="uint16",
            shape=(alto, ancho),
        )
        sumas_25[:] = 0
        aportes_25[:] = 0

        ventanas = VENTANAS_COMUNES[NIVEL_25]
        ejecutadas = 0
        omitidas = 0

        with torch.inference_mode():
            for numero, (col, fila, w, h) in enumerate(ventanas, 1):
                ventana = Window(col, fila, w, h)
                entrada, validos = leer_entrada_inferencia(
                    src, ventana, config_25
                )

                if entrada is None:
                    omitidas += 1
                else:
                    logits = modelo_25(entrada)

                    if (
                        list(logits.shape) != [1, clases_25, h, w]
                        or not torch.isfinite(logits).all().item()
                    ):
                        raise RuntimeError(
                            f"Salida incorrecta en ventana {numero}."
                        )

                    probabilidades = torch.softmax(
                        logits, dim=1
                    )[0].cpu().numpy()

                    sumas_25[:, fila:fila+h, col:col+w] += (
                        probabilidades * validos[None, :, :]
                    )
                    aportes_25[fila:fila+h, col:col+w] += (
                        validos.astype(np.uint16)
                    )
                    ejecutadas += 1
                    del entrada, logits, probabilidades

                if numero % 100 == 0 or numero == len(ventanas):
                    minutos = (time.perf_counter() - inicio_25) / 60
                    print(
                        f"Ventanas: {numero}/{len(ventanas)} | "
                        f"Ejecutadas: {ejecutadas} | "
                        f"Sin datos: {omitidas} | "
                        f"Minutos: {minutos:.1f}",
                        flush=True,
                    )

        sumas_25.flush()
        aportes_25.flush()

        perfil = {
            "driver": "GTiff",
            "width": ancho,
            "height": alto,
            "count": 1,
            "dtype": "uint8",
            "crs": src.crs,
            "transform": src.transform,
            "nodata": 255,
            "compress": "LZW",
            "tiled": True,
            "blockxsize": 256,
            "blockysize": 256,
        }

        conteos = np.zeros(256, dtype=np.int64)

        with rasterio.open(ruta_parcial_25, "w", **perfil) as dst:
            for fila in range(0, alto, 256):
                h = min(256, alto - fila)
                ventana = Window(0, fila, ancho, h)

                datos = src.read([1, 2, 3, 4], window=ventana)
                validos = np.isfinite(datos).all(axis=0)
                if src.nodata is not None:
                    validos &= (datos != src.nodata).all(axis=0)
                validos &= (
                    src.read_masks([1, 2, 3, 4], window=ventana) > 0
                ).all(axis=0)

                n = np.asarray(aportes_25[fila:fila+h, :])

                if not np.array_equal(n > 0, validos):
                    raise RuntimeError(
                        "La cobertura procesada no coincide con la validez."
                    )

                promedio = np.asarray(
                    sumas_25[:, fila:fila+h, :]
                ) / np.maximum(n[None, :, :], 1)

                if validos.any() and not np.allclose(
                    promedio.sum(axis=0)[validos],
                    1.0, atol=1e-5, rtol=1e-5,
                ):
                    raise RuntimeError("Promedio de probabilidades incorrecto.")

                salida = tabla_codigos[promedio.argmax(axis=0)]
                salida[~validos] = 255
                dst.write(salida, 1, window=ventana)

                conteos += np.bincount(
                    salida.ravel(), minlength=256
                )
                del promedio, salida, datos, n

            dst.update_tags(
                modelo=MODELO_25,
                nivel=NIVEL_25,
                combinacion="Promedio uniforme de probabilidades",
                estado="Inferencia técnica; evaluación pendiente",
                clases=json.dumps(
                    CLASES_COMUNES[NIVEL_25],
                    ensure_ascii=False,
                ),
            )

    # Reabrir y verificar lo efectivamente escrito.
    conteos_leidos = np.zeros(256, dtype=np.int64)

    with rasterio.open(ruta_parcial_25) as dst:
        if (
            dst.width != ancho
            or dst.height != alto
            or dst.crs != perfil["crs"]
            or dst.transform != perfil["transform"]
            or dst.nodata != 255
        ):
            raise RuntimeError("La georreferenciación de salida no coincide.")

        for _, ventana in dst.block_windows(1):
            bloque = dst.read(1, window=ventana)
            conteos_leidos += np.bincount(
                bloque.ravel(), minlength=256
            )

    if not np.array_equal(conteos, conteos_leidos):
        raise RuntimeError("Los conteos cambiaron durante la escritura.")

    if int(conteos.sum()) != alto * ancho:
        raise RuntimeError("El total de píxeles no coincide.")

    presentes = set(np.flatnonzero(conteos).tolist())
    if not presentes.issubset(set(tabla_codigos.tolist()) | {255}):
        raise RuntimeError("La salida contiene códigos inesperados.")

    ruta_parcial_25.rename(ruta_final_25)

    registro_25.update({
        "completada": True,
        "archivo": str(ruta_final_25),
        "ventanas_ejecutadas": ejecutadas,
        "ventanas_sin_datos": omitidas,
        "pixeles_con_prediccion": int(conteos[:255].sum()),
        "pixeles_nodata": int(conteos[255]),
        "conteos": {
            int(c): int(conteos[c])
            for c in np.flatnonzero(conteos)
        },
        "geometria_verificada": True,
        "cobertura_valida_verificada": True,
        "conteos_relectura_verificados": True,
    })

except Exception as error:
    registro_25["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    # Cerrar los acumuladores antes de eliminar temporales en Windows.
    for acumulador in (sumas_25, aportes_25):
        if acumulador is not None:
            acumulador.flush()
            acumulador._mmap.close()

    modelo_25 = None
    gc.collect()

    if registro_25["completada"]:
        ruta_sumas_25.unlink()
        ruta_aportes_25.unlink()

    registro_25["segundos_totales"] = (
        time.perf_counter() - inicio_25
    )

    with ruta_registro_25.open("x", encoding="utf-8") as archivo:
        json.dump(registro_25, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro:", ruta_registro_25)

print("\nArchivo generado:", ruta_final_25)
print("Píxeles con predicción:", registro_25["pixeles_con_prediccion"])
print("Píxeles NoData:", registro_25["pixeles_nodata"])
print("Conteos por código:", registro_25["conteos"])
print("\nCELDA 25 TERMINADA. Mapa generado y verificado.")

Ventanas: 100/1326 | Ejecutadas: 10 | Sin datos: 90 | Minutos: 0.0
Ventanas: 200/1326 | Ejecutadas: 36 | Sin datos: 164 | Minutos: 0.1
Ventanas: 300/1326 | Ejecutadas: 73 | Sin datos: 227 | Minutos: 0.2
Ventanas: 400/1326 | Ejecutadas: 127 | Sin datos: 273 | Minutos: 0.4
Ventanas: 500/1326 | Ejecutadas: 198 | Sin datos: 302 | Minutos: 0.6
Ventanas: 600/1326 | Ejecutadas: 288 | Sin datos: 312 | Minutos: 0.9
Ventanas: 700/1326 | Ejecutadas: 387 | Sin datos: 313 | Minutos: 1.2
Ventanas: 800/1326 | Ejecutadas: 485 | Sin datos: 315 | Minutos: 1.5
Ventanas: 900/1326 | Ejecutadas: 585 | Sin datos: 315 | Minutos: 1.8
Ventanas: 1000/1326 | Ejecutadas: 685 | Sin datos: 315 | Minutos: 2.1
Ventanas: 1100/1326 | Ejecutadas: 778 | Sin datos: 322 | Minutos: 2.3
Ventanas: 1200/1326 | Ejecutadas: 855 | Sin datos: 345 | Minutos: 2.6
Ventanas: 1300/1326 | Ejecutadas: 903 | Sin datos: 397 | Minutos: 2.7
Ventanas: 1326/1326 | Ejecutadas: 914 | Sin datos: 412 | Minutos: 2.7

Registro: D:\PROYECTO_GRADO_1\Au

RuntimeError: La georreferenciación de salida no coincide.

In [30]:
# ============================================================
# CELDA 25A — DIAGNÓSTICO DEL TIFF GENERADO
# No repite la inferencia ni modifica los rásteres.
# ============================================================

from pathlib import Path
from datetime import datetime
import json

import rasterio
from pyproj import CRS


carpeta_revision_25A = Path(
    r"D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos"
    r"\20261007_191328_200990"
    r"\Inferencia_RESNET50_nivel_1_20261007_235816_276690"
)

imagen_revision_25A = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO"
    r"\Raster_CLC\zona_estudio.tif"
)

prediccion_revision_25A = (
    carpeta_revision_25A / "RESNET50_nivel_1_PARCIAL.tif"
)

for ruta in [imagen_revision_25A, prediccion_revision_25A]:
    if not ruta.is_file():
        raise FileNotFoundError(ruta)


def describir_raster_25A(src):
    crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None

    return {
        "ruta": src.name,
        "ancho": src.width,
        "alto": src.height,
        "bandas": src.count,
        "tipos": list(src.dtypes),
        "transformacion": list(src.transform)[:6],
        "limites": list(src.bounds),
        "nodata": src.nodata,
        "crs_wkt": src.crs.to_wkt() if src.crs else None,
        "epsg_identificado": crs.to_epsg() if crs else None,
        "ejes": [
            {
                "nombre": eje.name,
                "direccion": eje.direction,
                "unidad": eje.unit_name,
            }
            for eje in crs.axis_info
        ] if crs else [],
    }


registro_25A = {
    "celda": "25A",
    "fecha": datetime.now().isoformat(),
}

with rasterio.open(imagen_revision_25A) as origen, \
     rasterio.open(prediccion_revision_25A) as salida:

    comprobaciones = {
        "dimensiones_iguales": (
            origen.width == salida.width
            and origen.height == salida.height
        ),
        "transformacion_exactamente_igual": (
            origen.transform == salida.transform
        ),
        "limites_iguales": origen.bounds == salida.bounds,
        "crs_igual_segun_rasterio": origen.crs == salida.crs,
        "nodata_salida_es_255": salida.nodata == 255,
        "una_banda_de_salida": salida.count == 1,
        "tipo_salida_uint8": salida.dtypes == ("uint8",),
    }

    comprobaciones["diferencia_transformacion"] = [
        float(b - a)
        for a, b in zip(
            list(origen.transform)[:6],
            list(salida.transform)[:6],
        )
    ]

    if origen.crs is not None and salida.crs is not None:
        crs_origen = CRS.from_wkt(origen.crs.to_wkt())
        crs_salida = CRS.from_wkt(salida.crs.to_wkt())

        comprobaciones["equivalencia_pyproj"] = (
            crs_origen.equals(crs_salida)
        )
        comprobaciones["equivalencia_ignorando_orden_ejes"] = (
            crs_origen.equals(crs_salida, ignore_axis_order=True)
        )

        # Reutiliza la comprobación específica auditada anteriormente,
        # si permanece disponible en el cuaderno.
        if callable(globals().get("crs_equivalentes_xy")):
            try:
                comprobaciones["equivalencia_xy_auditada"] = bool(
                    crs_equivalentes_xy(origen.crs, salida.crs)
                )
            except Exception as error:
                comprobaciones["error_comprobacion_xy"] = str(error)

    registro_25A["comprobaciones"] = comprobaciones
    registro_25A["imagen_original"] = describir_raster_25A(origen)
    registro_25A["prediccion_parcial"] = describir_raster_25A(salida)

print("=" * 75)
print("COMPROBACIONES INDIVIDUALES")
print("=" * 75)
print(json.dumps(comprobaciones, ensure_ascii=False, indent=2))

for nombre in ["imagen_original", "prediccion_parcial"]:
    print("\n" + "=" * 75)
    print(nombre)
    print("=" * 75)
    print(json.dumps(
        registro_25A[nombre],
        ensure_ascii=False,
        indent=2,
    ))

registro_25A["temporales_disponibles"] = {
    nombre: (carpeta_revision_25A / nombre).is_file()
    for nombre in [
        "sumas_probabilidades_temporal.dat",
        "aportes_temporal.dat",
    ]
}

print("\nTemporales disponibles:")
print(registro_25A["temporales_disponibles"])

ruta_registro_25A = carpeta_revision_25A / (
    "celda_25A_diagnostico_georreferenciacion_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)

with ruta_registro_25A.open("x", encoding="utf-8") as archivo:
    json.dump(registro_25A, archivo, ensure_ascii=False, indent=2)

print("\nRegistro:", ruta_registro_25A)
print("\nCELDA 25A TERMINADA. No se modificaron los rásteres.")

COMPROBACIONES INDIVIDUALES
{
  "dimensiones_iguales": true,
  "transformacion_exactamente_igual": true,
  "limites_iguales": true,
  "crs_igual_segun_rasterio": false,
  "nodata_salida_es_255": true,
  "una_banda_de_salida": true,
  "tipo_salida_uint8": true,
  "diferencia_transformacion": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0
  ],
  "equivalencia_pyproj": false,
  "equivalencia_ignorando_orden_ejes": false,
  "equivalencia_xy_auditada": true
}

imagen_original
{
  "ruta": "D:\\PROYECTO_GRADO_1\\PROYECTO_GRADO\\Raster_CLC\\zona_estudio.tif",
  "ancho": 4376,
  "alto": 4999,
  "bandas": 4,
  "tipos": [
    "uint16",
    "uint16",
    "uint16",
    "uint16"
  ],
  "transformacion": [
    10.0,
    0.0,
    4903449.579587339,
    0.0,
    -10.0,
    2150915.3094111346
  ],
  "limites": [
    4903449.579587339,
    2100925.3094111346,
    4947209.579587339,
    2150915.3094111346
  ],
  "nodata": 65535.0,
  "crs_wkt": "PROJCS[\"MAGNA-SIRGAS 2018 / Origen-Nacional\",GEOGCS

In [31]:
# ============================================================
# CELDA 25B — VERIFICAR Y RECUPERAR LA PREDICCIÓN GENERADA
# No ejecuta nuevamente el modelo.
# ============================================================

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import rasterio
from rasterio.windows import Window


if not callable(globals().get("crs_equivalentes_xy")):
    raise RuntimeError(
        "Falta la función crs_equivalentes_xy auditada anteriormente."
    )

carpeta_25B = Path(
    r"D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos"
    r"\20261007_191328_200990"
    r"\Inferencia_RESNET50_nivel_1_20261007_235816_276690"
)

imagen_25B = Path(
    r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO"
    r"\Raster_CLC\zona_estudio.tif"
)

parcial_25B = carpeta_25B / "RESNET50_nivel_1_PARCIAL.tif"
final_25B = carpeta_25B / "RESNET50_nivel_1_prediccion.tif"
sumas_path_25B = carpeta_25B / "sumas_probabilidades_temporal.dat"
aportes_path_25B = carpeta_25B / "aportes_temporal.dat"

for ruta in [
    imagen_25B,
    parcial_25B,
    sumas_path_25B,
    aportes_path_25B,
]:
    if not ruta.is_file():
        raise FileNotFoundError(ruta)

if final_25B.exists():
    raise FileExistsError(
        f"Ya existe el archivo final; no se sobrescribirá: {final_25B}"
    )

# Recuperar la configuración exacta registrada en la ejecución.
with (carpeta_25B / "celda_25_inferencia_y_verificacion.json").open(
    encoding="utf-8"
) as archivo:
    ejecucion_original = json.load(archivo)

config = ejecucion_original["configuracion"]
numero_clases = int(config["num_classes"])

correspondencia = {
    int(i): int(c)
    for i, c in config["idx_to_code"].items()
}
tabla_codigos = np.asarray(
    [correspondencia[i] for i in range(numero_clases)],
    dtype=np.uint8,
)

if numero_clases != 5 or set(tabla_codigos.tolist()) != set(range(1, 6)):
    raise RuntimeError("La configuración no corresponde al nivel 1 esperado.")

registro_25B = {
    "celda": "25B",
    "fecha": datetime.now().isoformat(),
    "modelo": "RESNET50",
    "nivel": "nivel_1",
    "registro_original": str(
        carpeta_25B / "celda_25_inferencia_y_verificacion.json"
    ),
    "correccion": (
        "Verificación mediante equivalencia XY auditada, "
        "con transformación y dimensiones exactamente iguales"
    ),
    "inferencia_repetida": False,
    "valores_modificados": False,
    "recuperacion_completada": False,
}

ruta_registro_25B = carpeta_25B / (
    "celda_25B_recuperacion_verificada_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)

sumas = None
aportes = None
RESNET_NIVEL1_COMPLETO_VERIFICADO = False

try:
    with rasterio.open(imagen_25B) as origen, \
         rasterio.open(parcial_25B) as pred:

        comprobaciones = {
            "dimensiones_identicas": (
                origen.width == pred.width
                and origen.height == pred.height
            ),
            "transformacion_identica": origen.transform == pred.transform,
            "limites_identicos": origen.bounds == pred.bounds,
            "crs_equivalente_xy": bool(
                crs_equivalentes_xy(origen.crs, pred.crs)
            ),
            "nodata_correcto": pred.nodata == 255,
            "una_banda_uint8": (
                pred.count == 1 and pred.dtypes == ("uint8",)
            ),
        }

        registro_25B["geometria_y_formato"] = comprobaciones

        if not all(comprobaciones.values()):
            raise RuntimeError(
                f"Falló la comprobación espacial: {comprobaciones}"
            )

        alto, ancho = origen.height, origen.width

        tamanos_esperados = {
            sumas_path_25B: numero_clases * alto * ancho * 4,
            aportes_path_25B: alto * ancho * 2,
        }

        for ruta, esperado in tamanos_esperados.items():
            if ruta.stat().st_size != esperado:
                raise RuntimeError(
                    f"Tamaño inesperado del acumulador: {ruta.name}"
                )

        sumas = np.memmap(
            sumas_path_25B,
            mode="r",
            dtype="float32",
            shape=(numero_clases, alto, ancho),
        )
        aportes = np.memmap(
            aportes_path_25B,
            mode="r",
            dtype="uint16",
            shape=(alto, ancho),
        )

        conteos = np.zeros(256, dtype=np.int64)
        pixeles_verificados = 0

        for fila in range(0, alto, 256):
            h = min(256, alto - fila)
            ventana = Window(0, fila, ancho, h)

            datos = origen.read([1, 2, 3, 4], window=ventana)
            validos = np.isfinite(datos).all(axis=0)

            if origen.nodata is not None:
                validos &= (datos != origen.nodata).all(axis=0)

            validos &= (
                origen.read_masks([1, 2, 3, 4], window=ventana) > 0
            ).all(axis=0)

            # Copias pequeñas por franja, sin referencias abiertas al memmap.
            n = np.array(aportes[fila:fila+h, :], copy=True)
            acumulado = np.array(
                sumas[:, fila:fila+h, :], copy=True
            )

            if not np.array_equal(n > 0, validos):
                raise RuntimeError(
                    f"Cobertura distinta de la validez en la fila {fila}."
                )

            if not np.isfinite(acumulado).all() or np.any(acumulado < 0):
                raise RuntimeError("Acumulador con valores incorrectos.")

            promedio = acumulado / np.maximum(n[None, :, :], 1)

            if validos.any() and not np.allclose(
                promedio.sum(axis=0)[validos],
                1.0,
                atol=1e-5,
                rtol=1e-5,
            ):
                raise RuntimeError("Probabilidades acumuladas incoherentes.")

            esperado = tabla_codigos[promedio.argmax(axis=0)]
            esperado[~validos] = 255

            almacenado = pred.read(1, window=ventana)

            if not np.array_equal(almacenado, esperado):
                diferencias = int(np.count_nonzero(almacenado != esperado))
                raise RuntimeError(
                    f"El TIFF difiere del acumulador en {diferencias} "
                    f"píxeles de la franja iniciada en {fila}."
                )

            if not np.array_equal(
                pred.read_masks(1, window=ventana) > 0,
                validos,
            ):
                raise RuntimeError("La máscara del TIFF no coincide.")

            conteos += np.bincount(almacenado.ravel(), minlength=256)
            pixeles_verificados += almacenado.size

        if pixeles_verificados != alto * ancho:
            raise RuntimeError("La verificación no cubrió toda la imagen.")

        registro_25B.update({
            "pixeles_verificados": int(pixeles_verificados),
            "pixeles_con_prediccion": int(conteos[:255].sum()),
            "pixeles_nodata": int(conteos[255]),
            "coincidencia_pixel_a_pixel_con_acumuladores": True,
            "mascara_verificada": True,
            "conteos_por_codigo": {
                int(c): int(conteos[c])
                for c in np.flatnonzero(conteos)
            },
        })

    # Los rásteres ya están cerrados: cambiar únicamente el nombre.
    parcial_25B.rename(final_25B)

    registro_25B["archivo_final"] = str(final_25B)
    registro_25B["recuperacion_completada"] = True
    RESNET_NIVEL1_COMPLETO_VERIFICADO = True
    RUTA_RESNET_NIVEL1_VERIFICADA = final_25B

except Exception as error:
    registro_25B["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    for acumulador in (sumas, aportes):
        if acumulador is not None:
            acumulador._mmap.close()

    with ruta_registro_25B.open("x", encoding="utf-8") as archivo:
        json.dump(registro_25B, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro:", ruta_registro_25B)

print("\nArchivo verificado:", final_25B)
print("Píxeles verificados:", registro_25B["pixeles_verificados"])
print("Píxeles con predicción:", registro_25B["pixeles_con_prediccion"])
print("Píxeles NoData:", registro_25B["pixeles_nodata"])
print("Conteos por código:", registro_25B["conteos_por_codigo"])
print(
    "Coincidencia píxel a píxel con acumuladores:",
    registro_25B["coincidencia_pixel_a_pixel_con_acumuladores"],
)
print("\nCELDA 25B TERMINADA. Predicción recuperada y verificada.")


Registro: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_RESNET50_nivel_1_20261007_235816_276690\celda_25B_recuperacion_verificada_20261008_000410_356984.json

Archivo verificado: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_RESNET50_nivel_1_20261007_235816_276690\RESNET50_nivel_1_prediccion.tif
Píxeles verificados: 21875624
Píxeles con predicción: 12323120
Píxeles NoData: 9552504
Conteos por código: {1: 9140859, 2: 2821071, 3: 187009, 4: 149118, 5: 25063, 255: 9552504}
Coincidencia píxel a píxel con acumuladores: True

CELDA 25B TERMINADA. Predicción recuperada y verificada.


In [32]:
# ============================================================
# CELDA 26 — INFERENCIA VERIFICADA Y RESNET50 NIVEL 2
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import time
import gc
import shutil

import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.crs import CRS as RasterioCRS
import torch


if not globals().get("RESNET_NIVEL1_COMPLETO_VERIFICADO", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 25B.")

if not globals().get("PLAN_VENTANAS_VERIFICADO", False):
    raise RuntimeError("Falta el plan verificado de la celda 24.")


def ejecutar_inferencia_verificada(nombre_modelo, nivel, numero_celda):
    if nombre_modelo == "RESNET50":
        config = CONFIGS_RESNET_AUDITADAS[nivel]
        carpeta_pesos = CARPETAS_RESNET[nivel]
    elif nombre_modelo == "DENSENET121":
        config = CONFIGS_DENSENET_AUDITADAS[nivel]
        carpeta_pesos = CARPETAS_DENSENET[nivel]
    else:
        raise ValueError("Esta función admite ResNet50 y DenseNet121.")

    clases = int(config["num_classes"])
    correspondencia = {
        int(i): int(c) for i, c in config["idx_to_code"].items()
    }
    if (
        set(correspondencia) != set(range(clases))
        or set(correspondencia.values()) != set(range(1, clases + 1))
    ):
        raise RuntimeError("Correspondencia de clases incorrecta.")

    tabla = np.asarray(
        [correspondencia[i] for i in range(clases)],
        dtype=np.uint8,
    )

    carpeta = Path(CARPETA_AUDITORIA) / (
        f"Inferencia_{nombre_modelo}_{nivel}_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    carpeta.mkdir(exist_ok=False)

    parcial = carpeta / f"{nombre_modelo}_{nivel}_PARCIAL.tif"
    final = carpeta / f"{nombre_modelo}_{nivel}_prediccion.tif"
    ruta_sumas = carpeta / "sumas_probabilidades_temporal.dat"
    ruta_aportes = carpeta / "aportes_temporal.dat"
    ruta_registro = carpeta / f"celda_{numero_celda}_inferencia.json"

    registro = {
        "celda": numero_celda,
        "inicio": datetime.now().isoformat(),
        "modelo": nombre_modelo,
        "nivel": nivel,
        "pesos": str(carpeta_pesos / "best_model.pt"),
        "configuracion": config,
        "plan": PLAN_INFERENCIA,
        "control_mascara": str(RUTA_RESNET_NIVEL1_VERIFICADA),
        "completada": False,
    }

    sumas = None
    aportes = None
    modelo = None
    inicio = time.perf_counter()

    try:
        argumentos = {
            "entrada": 4,
            "clases": clases,
            "dropout": float(config["decoder_dropout"]),
        }

        if nombre_modelo == "RESNET50":
            modelo = ResNet50Auditada(**argumentos).cpu()
        else:
            modelo = DenseNet121Auditada(
                **argumentos,
                memory_efficient=bool(config["densenet_memory_efficient"]),
            ).cpu()

        checkpoint = torch.load(
            carpeta_pesos / "best_model.pt",
            map_location="cpu",
            weights_only=True,
        )
        modelo.load_state_dict(checkpoint["model_state_dict"], strict=True)
        modelo.eval()
        del checkpoint

        with rasterio.open(RUTA_IMAGEN_INFERENCIA) as src:
            alto, ancho = src.height, src.width
            cuadricula = PLAN_INFERENCIA["cuadricula"]

            if (
                ancho != cuadricula["ancho"]
                or alto != cuadricula["alto"]
                or list(src.transform)[:6] != cuadricula["transformacion"]
                or not crs_equivalentes_xy(
                    src.crs,
                    RasterioCRS.from_wkt(cuadricula["crs_wkt"]),
                )
            ):
                raise RuntimeError("La cuadrícula no coincide con el plan.")

            bytes_necesarios = alto * ancho * (clases * 4 + 2)
            if shutil.disk_usage(carpeta).free < bytes_necesarios + 300_000_000:
                raise RuntimeError("Espacio insuficiente para los temporales.")

            sumas = np.memmap(
                ruta_sumas, mode="w+", dtype="float32",
                shape=(clases, alto, ancho),
            )
            aportes = np.memmap(
                ruta_aportes, mode="w+", dtype="uint16",
                shape=(alto, ancho),
            )
            sumas[:] = 0
            aportes[:] = 0

            ventanas = VENTANAS_COMUNES[nivel]
            ejecutadas = 0
            omitidas = 0

            with torch.inference_mode():
                for numero, (col, fila, w, h) in enumerate(ventanas, 1):
                    entrada, validos = leer_entrada_inferencia(
                        src, Window(col, fila, w, h), config
                    )

                    if entrada is None:
                        omitidas += 1
                    else:
                        logits = modelo(entrada)
                        if (
                            list(logits.shape) != [1, clases, h, w]
                            or not torch.isfinite(logits).all().item()
                        ):
                            raise RuntimeError(
                                f"Salida incorrecta en ventana {numero}."
                            )

                        probabilidades = torch.softmax(
                            logits, dim=1
                        )[0].cpu().numpy()

                        sumas[:, fila:fila+h, col:col+w] += (
                            probabilidades * validos[None, :, :]
                        )
                        aportes[fila:fila+h, col:col+w] += (
                            validos.astype(np.uint16)
                        )
                        ejecutadas += 1
                        del entrada, logits, probabilidades

                    if numero % 250 == 0 or numero == len(ventanas):
                        print(
                            f"{nombre_modelo} {nivel} | "
                            f"{numero}/{len(ventanas)} ventanas | "
                            f"Ejecutadas: {ejecutadas} | "
                            f"Sin datos: {omitidas} | "
                            f"Minutos: {(time.perf_counter()-inicio)/60:.1f}",
                            flush=True,
                        )

            sumas.flush()
            aportes.flush()

            # Reconstrucción por franjas desde las probabilidades.
            def reconstruir_franja(fila, h):
                ventana = Window(0, fila, ancho, h)
                datos = src.read([1, 2, 3, 4], window=ventana)
                validos = np.isfinite(datos).all(axis=0)

                if src.nodata is not None:
                    validos &= (datos != src.nodata).all(axis=0)

                validos &= (
                    src.read_masks([1, 2, 3, 4], window=ventana) > 0
                ).all(axis=0)

                n = np.array(aportes[fila:fila+h, :], copy=True)
                p = np.array(sumas[:, fila:fila+h, :], copy=True)

                if not np.array_equal(n > 0, validos):
                    raise RuntimeError("La cobertura difiere de la validez.")

                if not np.isfinite(p).all() or np.any(p < 0):
                    raise RuntimeError("Probabilidades acumuladas incorrectas.")

                p /= np.maximum(n[None, :, :], 1)

                if validos.any() and not np.allclose(
                    p.sum(axis=0)[validos], 1.0,
                    atol=1e-5, rtol=1e-5,
                ):
                    raise RuntimeError("Promedio de probabilidades incorrecto.")

                salida = tabla[p.argmax(axis=0)]
                salida[~validos] = 255
                return ventana, salida, validos

            perfil = {
                "driver": "GTiff",
                "width": ancho,
                "height": alto,
                "count": 1,
                "dtype": "uint8",
                "crs": src.crs,
                "transform": src.transform,
                "nodata": 255,
                "compress": "LZW",
                "tiled": True,
                "blockxsize": 256,
                "blockysize": 256,
            }

            with rasterio.open(parcial, "w", **perfil) as dst:
                for fila in range(0, alto, 256):
                    h = min(256, alto - fila)
                    ventana, salida, validos = reconstruir_franja(fila, h)
                    dst.write(salida, 1, window=ventana)

                leyenda = {
                    int(c): CLASES_COMUNES[nivel][int(c)]
                    for c in tabla
                }
                leyenda[255] = "NoData de entrada"

                dst.update_tags(
                    modelo=nombre_modelo,
                    nivel=nivel,
                    combinacion="Promedio uniforme de probabilidades",
                    estado="Inferencia técnica; evaluación pendiente",
                    clases=json.dumps(leyenda, ensure_ascii=False),
                )

            print("Verificando TIFF, máscara y valores píxel por píxel...")

            conteos = np.zeros(256, dtype=np.int64)

            with rasterio.open(parcial) as pred, \
                 rasterio.open(RUTA_RESNET_NIVEL1_VERIFICADA) as control:

                for archivo in (pred, control):
                    if (
                        archivo.width != ancho
                        or archivo.height != alto
                        or archivo.transform != src.transform
                        or not crs_equivalentes_xy(src.crs, archivo.crs)
                        or archivo.count != 1
                        or archivo.dtypes != ("uint8",)
                        or archivo.nodata != 255
                    ):
                        raise RuntimeError(
                            f"Geometría o formato incorrectos: {archivo.name}"
                        )

                for fila in range(0, alto, 256):
                    h = min(256, alto - fila)
                    ventana, esperado, validos = reconstruir_franja(fila, h)
                    almacenado = pred.read(1, window=ventana)

                    if not np.array_equal(almacenado, esperado):
                        raise RuntimeError("El TIFF difiere del acumulador.")

                    if not np.array_equal(
                        pred.read_masks(1, window=ventana) > 0, validos
                    ):
                        raise RuntimeError("La máscara de salida no coincide.")

                    if not np.array_equal(
                        control.read_masks(1, window=ventana) > 0, validos
                    ):
                        raise RuntimeError(
                            "La máscara difiere de ResNet50, nivel 1."
                        )

                    conteos += np.bincount(
                        almacenado.ravel(), minlength=256
                    )

            if int(conteos.sum()) != alto * ancho:
                raise RuntimeError("Total de píxeles incorrecto.")

            if int(conteos[:255].sum()) != 12323120:
                raise RuntimeError("Número de píxeles válidos inesperado.")

        parcial.rename(final)

        registro.update({
            "completada": True,
            "archivo_final": str(final),
            "ventanas_ejecutadas": ejecutadas,
            "ventanas_sin_datos": omitidas,
            "pixeles_verificados": int(conteos.sum()),
            "pixeles_con_prediccion": int(conteos[:255].sum()),
            "pixeles_nodata": int(conteos[255]),
            "geometria_verificada_xy": True,
            "mascara_identica_a_resnet_nivel1": True,
            "coincidencia_pixel_a_pixel_con_acumuladores": True,
            "conteos_por_codigo": {
                int(c): int(conteos[c])
                for c in np.flatnonzero(conteos)
            },
        })

    except Exception as error:
        registro["error"] = f"{type(error).__name__}: {error}"
        raise

    finally:
        for acumulador in (sumas, aportes):
            if acumulador is not None:
                acumulador.flush()
                acumulador._mmap.close()

        modelo = None
        gc.collect()

        # Conservar temporales ante fallos para permitir recuperación.
        if registro["completada"]:
            for temporal in (ruta_sumas, ruta_aportes):
                try:
                    temporal.unlink()
                except OSError as error:
                    registro.setdefault(
                        "temporales_no_eliminados", []
                    ).append(f"{temporal}: {error}")

        registro["segundos_totales"] = time.perf_counter() - inicio

        with ruta_registro.open("x", encoding="utf-8") as archivo:
            json.dump(registro, archivo, ensure_ascii=False, indent=2)

        print("\nRegistro:", ruta_registro)

    print("\nArchivo verificado:", final)
    print("Píxeles con predicción:", registro["pixeles_con_prediccion"])
    print("Píxeles NoData:", registro["pixeles_nodata"])
    print("Conteos por código:", registro["conteos_por_codigo"])
    print("Máscara idéntica a ResNet50 nivel 1: True")
    print("Coincidencia píxel a píxel con acumuladores: True")

    return final, registro


RUTA_RESNET_NIVEL2_VERIFICADA, RESULTADO_RESNET_NIVEL2 = (
    ejecutar_inferencia_verificada(
        nombre_modelo="RESNET50",
        nivel="nivel_2",
        numero_celda=26,
    )
)

print("\nCELDA 26 TERMINADA. ResNet50 nivel 2 generado y verificado.")

RESNET50 nivel_2 | 250/5304 ventanas | Ejecutadas: 9 | Sin datos: 241 | Minutos: 0.0
RESNET50 nivel_2 | 500/5304 ventanas | Ejecutadas: 26 | Sin datos: 474 | Minutos: 0.0
RESNET50 nivel_2 | 750/5304 ventanas | Ejecutadas: 67 | Sin datos: 683 | Minutos: 0.1
RESNET50 nivel_2 | 1000/5304 ventanas | Ejecutadas: 139 | Sin datos: 861 | Minutos: 0.2
RESNET50 nivel_2 | 1250/5304 ventanas | Ejecutadas: 224 | Sin datos: 1026 | Minutos: 0.3
RESNET50 nivel_2 | 1500/5304 ventanas | Ejecutadas: 334 | Sin datos: 1166 | Minutos: 0.4
RESNET50 nivel_2 | 1750/5304 ventanas | Ejecutadas: 489 | Sin datos: 1261 | Minutos: 0.5
RESNET50 nivel_2 | 2000/5304 ventanas | Ejecutadas: 649 | Sin datos: 1351 | Minutos: 0.7
RESNET50 nivel_2 | 2250/5304 ventanas | Ejecutadas: 852 | Sin datos: 1398 | Minutos: 0.9
RESNET50 nivel_2 | 2500/5304 ventanas | Ejecutadas: 1083 | Sin datos: 1417 | Minutos: 1.2
RESNET50 nivel_2 | 2750/5304 ventanas | Ejecutadas: 1323 | Sin datos: 1427 | Minutos: 1.4
RESNET50 nivel_2 | 3000/5304 v

In [33]:
# ============================================================
# CELDA 27 — PREDICCIÓN COMPLETA DENSENET121, NIVEL 1
# ============================================================

if not globals().get("PRUEBA_REAL_DENSENET_VERIFICADA", False):
    raise RuntimeError(
        "Primero debe finalizar correctamente la celda 23."
    )

if not callable(globals().get("ejecutar_inferencia_verificada")):
    raise RuntimeError(
        "Falta la función definida en la celda 26."
    )

RUTA_DENSENET_NIVEL1_VERIFICADA, RESULTADO_DENSENET_NIVEL1 = (
    ejecutar_inferencia_verificada(
        nombre_modelo="DENSENET121",
        nivel="nivel_1",
        numero_celda=27,
    )
)

print("\nCELDA 27 TERMINADA.")
print("DenseNet121 nivel 1 generado y verificado.")

DENSENET121 nivel_1 | 250/1326 ventanas | Ejecutadas: 49 | Sin datos: 201 | Minutos: 0.1
DENSENET121 nivel_1 | 500/1326 ventanas | Ejecutadas: 198 | Sin datos: 302 | Minutos: 0.6
DENSENET121 nivel_1 | 750/1326 ventanas | Ejecutadas: 436 | Sin datos: 314 | Minutos: 1.2
DENSENET121 nivel_1 | 1000/1326 ventanas | Ejecutadas: 685 | Sin datos: 315 | Minutos: 1.9
DENSENET121 nivel_1 | 1250/1326 ventanas | Ejecutadas: 889 | Sin datos: 361 | Minutos: 2.5
DENSENET121 nivel_1 | 1326/1326 ventanas | Ejecutadas: 914 | Sin datos: 412 | Minutos: 2.6
Verificando TIFF, máscara y valores píxel por píxel...

Registro: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_DENSENET121_nivel_1_20261008_001317_776842\celda_27_inferencia.json

Archivo verificado: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_DENSENET121_nivel_1_20261008_001317_776842\DENSENET121_nivel_1_prediccion.tif
Píxeles con predicción: 12323120
Píxeles NoData: 9552504


In [34]:
# ============================================================
# CELDA 28 — PREDICCIÓN COMPLETA DENSENET121, NIVEL 2
# ============================================================

if not globals().get("PRUEBA_REAL_DENSENET_VERIFICADA", False):
    raise RuntimeError(
        "Primero debe finalizar correctamente la celda 23."
    )

if not callable(globals().get("ejecutar_inferencia_verificada")):
    raise RuntimeError(
        "Falta la función definida en la celda 26."
    )

RUTA_DENSENET_NIVEL2_VERIFICADA, RESULTADO_DENSENET_NIVEL2 = (
    ejecutar_inferencia_verificada(
        nombre_modelo="DENSENET121",
        nivel="nivel_2",
        numero_celda=28,
    )
)

print("\nCELDA 28 TERMINADA.")
print("DenseNet121 nivel 2 generado y verificado.")

DENSENET121 nivel_2 | 250/5304 ventanas | Ejecutadas: 9 | Sin datos: 241 | Minutos: 0.0
DENSENET121 nivel_2 | 500/5304 ventanas | Ejecutadas: 26 | Sin datos: 474 | Minutos: 0.0
DENSENET121 nivel_2 | 750/5304 ventanas | Ejecutadas: 67 | Sin datos: 683 | Minutos: 0.1
DENSENET121 nivel_2 | 1000/5304 ventanas | Ejecutadas: 139 | Sin datos: 861 | Minutos: 0.2
DENSENET121 nivel_2 | 1250/5304 ventanas | Ejecutadas: 224 | Sin datos: 1026 | Minutos: 0.2
DENSENET121 nivel_2 | 1500/5304 ventanas | Ejecutadas: 334 | Sin datos: 1166 | Minutos: 0.4
DENSENET121 nivel_2 | 1750/5304 ventanas | Ejecutadas: 489 | Sin datos: 1261 | Minutos: 0.5
DENSENET121 nivel_2 | 2000/5304 ventanas | Ejecutadas: 649 | Sin datos: 1351 | Minutos: 0.7
DENSENET121 nivel_2 | 2250/5304 ventanas | Ejecutadas: 852 | Sin datos: 1398 | Minutos: 0.9
DENSENET121 nivel_2 | 2500/5304 ventanas | Ejecutadas: 1083 | Sin datos: 1417 | Minutos: 1.1
DENSENET121 nivel_2 | 2750/5304 ventanas | Ejecutadas: 1323 | Sin datos: 1427 | Minutos: 1

In [36]:
# ============================================================
# CELDA 29 — U-NET NIVEL 1 CON BITÁCORA DE DECISIONES
# Reemplaza la versión anterior de la celda 29.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
import hashlib
import json
import time
import gc
import shutil

import numpy as np
import rasterio
from rasterio.windows import Window
import torch
import torch.nn as nn
import torch.nn.functional as F


# Colombia: UTC-05:00.
ZONA_COLOMBIA_29 = timezone(timedelta(hours=-5))


def fecha_colombia_29():
    return datetime.now(ZONA_COLOMBIA_29).isoformat(timespec="seconds")


def sha256_archivo_29(ruta):
    resumen = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            resumen.update(bloque)
    return resumen.hexdigest()


if not globals().get("CARGA_UNET_VERIFICADA", False):
    raise RuntimeError("Primero debe completarse la celda 16.")

if not globals().get("PLAN_VENTANAS_VERIFICADO", False):
    raise RuntimeError("Primero debe completarse la celda 24.")

requisitos = [
    "CARPETA_AUDITORIA",
    "RUTA_IMAGEN_INFERENCIA",
    "RUTA_RESNET_NIVEL1_VERIFICADA",
    "PLAN_INFERENCIA",
    "VENTANAS_COMUNES",
    "CLASES_COMUNES",
    "crs_equivalentes_xy",
]

faltan = [nombre for nombre in requisitos if nombre not in globals()]
if faltan:
    raise RuntimeError(f"Faltan componentes anteriores: {faltan}")


CARPETAS_UNET_INFERENCIA = {
    "nivel_1": Path(r"C:\Users\Ivan sua\anaconda3\Modelos_UNET\Nivel 1"),
    "nivel_2": Path(r"C:\Users\Ivan sua\anaconda3\Modelos_UNET\Nivel 2"),
}


# ------------------------------------------------------------
# Arquitectura del predictor recuperado.
# Los nombres internos preservan las claves del checkpoint.
# ------------------------------------------------------------

class ConvUNet29(nn.Module):
    def __init__(self, entrada, salida, dropout=0.0):
        super().__init__()
        capas = [
            nn.Conv2d(entrada, salida, 3, padding=1),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            capas.append(nn.Dropout2d(dropout))
        capas += [
            nn.Conv2d(salida, salida, 3, padding=1),
            nn.ReLU(inplace=True),
        ]
        self.net = nn.Sequential(*capas)

    def forward(self, x):
        return self.net(x)


class DownUNet29(nn.Module):
    def __init__(self, entrada, salida, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            ConvUNet29(entrada, salida, dropout),
        )

    def forward(self, x):
        return self.net(x)


class UpUNet29(nn.Module):
    def __init__(self, entrada, salida):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            entrada, entrada // 2, kernel_size=2, stride=2
        )
        self.conv = ConvUNet29(entrada, salida)

    def forward(self, x, salto):
        x = self.up(x)
        dy = salto.size(2) - x.size(2)
        dx = salto.size(3) - x.size(3)

        if dx != 0 or dy != 0:
            x = F.pad(
                x,
                [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2],
            )

        return self.conv(torch.cat([salto, x], dim=1))


class UNetCompleta29(nn.Module):
    def __init__(self, config):
        super().__init__()

        if int(config["num_channels"]) != 4:
            raise RuntimeError("U-Net debe tener cuatro canales de entrada.")
        if bool(config["BILINEAR"]):
            raise RuntimeError("El modelo recuperado utiliza BILINEAR=False.")
        if float(config["DROPOUT"]) != 0.0:
            raise RuntimeError("Dropout general diferente del modelo auditado.")

        b = int(config["BASE_CHANNELS"])
        bd = (
            float(config["BOTTLENECK_DROPOUT"])
            if bool(config["USE_DROPOUT_BOTTLENECK"])
            else 0.0
        )

        self.inc = ConvUNet29(4, b)
        self.down1 = DownUNet29(b, b * 2)
        self.down2 = DownUNet29(b * 2, b * 4)
        self.down3 = DownUNet29(b * 4, b * 8)
        self.down4 = DownUNet29(b * 8, b * 16, bd)

        self.up1 = UpUNet29(b * 16, b * 8)
        self.up2 = UpUNet29(b * 8, b * 4)
        self.up3 = UpUNet29(b * 4, b * 2)
        self.up4 = UpUNet29(b * 2, b)

        self.outc = nn.Conv2d(b, int(config["NUM_CLASSES"]), 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        return self.outc(self.up4(x, x1))


def validez_imagen_unet29(src, ventana):
    datos = src.read([1, 2, 3, 4], window=ventana).astype(np.float32)
    validos = np.isfinite(datos).all(axis=0)

    if src.nodata is not None:
        validos &= (datos != src.nodata).all(axis=0)

    validos &= (
        src.read_masks([1, 2, 3, 4], window=ventana) > 0
    ).all(axis=0)

    return datos, validos


# ------------------------------------------------------------
# Decisiones fechadas: evidencia y limitaciones explícitas.
# ------------------------------------------------------------

def decisiones_unet29(nivel):
    fecha = fecha_colombia_29()
    decisiones = [
        {
            "id": "U01",
            "decision": "Conservar la arquitectura y los pesos recuperados",
            "justificacion": (
                "Utilizar el modelo guardado sin modificar sus parámetros "
                "ni el número de canales para adaptarlo a otros modelos."
            ),
            "evidencia": (
                "Predictor PREDICCION_FINAL(1).ipynb; configuración; "
                "carga estricta previamente verificada en la celda 16."
            ),
            "estado": "Se verifica nuevamente en esta ejecución",
        },
        {
            "id": "U02",
            "decision": "DN/10000, sin clip; (x-means)/(stds+1e-6)",
            "justificacion": (
                "Reproducir el predictor recuperado y la prueba real de "
                "la celda 19, conservando las estadísticas guardadas."
            ),
            "evidencia": (
                "PREDICCION_FINAL(1).ipynb; configuración del modelo; "
                "diagnóstico radiométrico de la celda 17."
            ),
            "estado": "Provisional",
            "limitacion": (
                "La compatibilidad de escala observada no demuestra "
                "que este fuera el preprocesamiento del entrenamiento."
            ),
        },
        {
            "id": "U03",
            "decision": "Leer las bandas físicas 1, 2, 3 y 4 en ese orden",
            "justificacion": (
                "Mantener el orden utilizado por los predictores y "
                "evitar una permutación espectral no documentada."
            ),
            "estado": "Orden físico definido; identidad espectral pendiente",
            "limitacion": (
                "El TIFF completo no identifica sus bandas; no se considera "
                "confirmado documentalmente el orden B2, B3, B4, B8."
            ),
        },
        {
            "id": "U04",
            "decision": "Conservar todos los canales, incluido el índice 0",
            "justificacion": (
                "No forzar una clase alternativa mediante eliminación "
                "de canales o remapeo optimizado contra la referencia."
            ),
            "evidencia": (
                "Checkpoint con 6/14 canales y configuración IGNORE_INDEX=0. "
                "PyTorch documenta que ignore_index afecta las etiquetas "
                "de la pérdida, no elimina canales de inferencia."
            ),
            "estado": "Regla operativa explícita",
        },
        {
            "id": "U05",
            "decision": "Índices 1..K como códigos del dataset; 0 sin clasificación",
            "justificacion": (
                "Mantener la codificación adoptada en el predictor recuperado "
                "y el catálogo común, sin desplazamientos automáticos +1."
            ),
            "estado": "Correspondencia operativa; trazabilidad de entrenamiento pendiente",
            "limitacion": (
                "El número de canales por sí solo no demuestra "
                "la semántica de cada canal."
            ),
        },
        {
            "id": "U06",
            "decision": "Entrada inválida igual a cero después de normalizar",
            "justificacion": (
                "Evitar que el valor NoData 65535 entre al modelo como señal "
                "radiométrica; aplicar la misma política operativa de relleno."
            ),
            "estado": "Decisión operativa aplicada también en la prueba real",
            "limitacion": (
                "El relleno puede influir en predicciones próximas al borde; "
                "no equivale a reconstruir datos ausentes ni a detectar nubes."
            ),
        },
        {
            "id": "U07",
            "decision": "Ventanas comunes y promedio uniforme de probabilidades",
            "justificacion": (
                "Mantener el protocolo de la celda 24 para las tres redes, "
                "sin elegir parámetros a partir de las métricas de prueba."
            ),
            "evidencia": "Celda 24; documentación de inferencia por ventanas de MONAI",
            "estado": "Decisión metodológica común",
            "limitacion": (
                "El solapamiento del 50% no se presenta como óptimo demostrado. "
                "El promedio define este protocolo y puede diferir de mapas anteriores."
            ),
        },
        {
            "id": "U08",
            "decision": "NoData=255; conservar 0 sobre entradas válidas",
            "justificacion": (
                "Distinguir ausencia de datos de entrada de una salida "
                "sin clase temática y conservar el universo de evaluación."
            ),
            "estado": "Implementado en el TIFF",
            "evaluacion_pendiente": (
                "Contar 0 como error sobre referencia temática válida; "
                "no incluirlo como clase temática en promedios macro. "
                "Usar una columna Sin clasificación común a los tres modelos."
            ),
        },
        {
            "id": "U09",
            "decision": "Verificar CRS por equivalencia XY y transformación exacta",
            "justificacion": (
                "Resolver la diferencia de representación de ejes documentada "
                "en las celdas 25A y 25B sin reproyectar ni desplazar píxeles."
            ),
            "estado": "Comprobación técnica",
        },
    ]

    for decision in decisiones:
        decision["fecha_registro_colombia"] = fecha
        decision["nivel"] = nivel

    return decisiones


# ------------------------------------------------------------
# Inferencia completa y verificación.
# Esta función también se utilizará en la celda 30.
# ------------------------------------------------------------

def ejecutar_unet_completa(nivel, numero_celda):
    carpeta_pesos = CARPETAS_UNET_INFERENCIA[nivel]
    ruta_config = carpeta_pesos / "config.json"
    ruta_pesos = carpeta_pesos / "best_model.pt"

    with ruta_config.open(encoding="utf-8-sig") as archivo:
        config = json.load(archivo)

    canales = int(config["NUM_CLASSES"])
    if canales != (6 if nivel == "nivel_1" else 14):
        raise RuntimeError("Número de canales inesperado.")
    if int(config["IGNORE_INDEX"]) != 0:
        raise RuntimeError("IGNORE_INDEX difiere de la configuración auditada.")

    medias = np.asarray(config["means"], dtype=np.float32)[:, None, None]
    desv = np.asarray(config["stds"], dtype=np.float32)[:, None, None]

    if (
        medias.shape != (4, 1, 1)
        or desv.shape != (4, 1, 1)
        or not np.isfinite(medias).all()
        or not np.isfinite(desv).all()
        or np.any(desv <= 0)
    ):
        raise RuntimeError("Estadísticas de normalización incorrectas.")

    carpeta = Path(CARPETA_AUDITORIA) / (
        f"Inferencia_UNET_{nivel}_documentada_"
        + datetime.now(ZONA_COLOMBIA_29).strftime("%Y%m%d_%H%M%S_%f")
    )
    carpeta.mkdir(exist_ok=False)

    parcial = carpeta / f"UNET_{nivel}_PARCIAL.tif"
    final = carpeta / f"UNET_{nivel}_prediccion.tif"
    ruta_sumas = carpeta / "sumas_probabilidades_temporal.dat"
    ruta_aportes = carpeta / "aportes_temporal.dat"
    ruta_decisiones = carpeta / "decisiones_metodologicas.json"
    ruta_registro = carpeta / f"celda_{numero_celda}_inferencia.json"

    decisiones = decisiones_unet29(nivel)

    # Registrar las decisiones antes de generar resultados.
    with ruta_decisiones.open("x", encoding="utf-8") as archivo:
        json.dump(decisiones, archivo, ensure_ascii=False, indent=2)

    registro = {
        "celda": numero_celda,
        "inicio_colombia": fecha_colombia_29(),
        "modelo": "UNET",
        "nivel": nivel,
        "imagen": str(RUTA_IMAGEN_INFERENCIA),
        "pesos": str(ruta_pesos),
        "configuracion": config,
        "plan": PLAN_INFERENCIA,
        "decisiones": decisiones,
        "bibliografia_metodologica": [
            {
                "autor_institucional": "PyTorch",
                "titulo": "CrossEntropyLoss",
                "url": (
                    "https://docs.pytorch.org/docs/stable/generated/"
                    "torch.nn.CrossEntropyLoss.html"
                ),
                "alcance": "Significado de ignore_index; no acredita el entrenamiento",
            },
            {
                "autor_institucional": "MONAI",
                "titulo": "Inference methods",
                "url": "https://docs.monai.io/en/1.4.0/inferers.html",
                "alcance": "Ventanas solapadas y combinación con pesos uniformes",
            },
        ],
        "limitaciones_pendientes": [
            "Preprocesamiento original del entrenamiento de U-Net",
            "Trazabilidad completa entre canales y etiquetas de entrenamiento",
            "Identidad espectral y procedencia radiométrica del TIFF",
            "Calidad y nubes",
            "Referencia 2022 frente a imagen identificada como 2025",
            "Alcance de independencia espacial de la evaluación",
        ],
        "completada": False,
    }

    sumas = None
    aportes = None
    modelo = None
    inicio = time.perf_counter()

    try:
        registro["sha256_pesos"] = sha256_archivo_29(ruta_pesos)
        registro["sha256_config"] = sha256_archivo_29(ruta_config)

        modelo = UNetCompleta29(config).cpu()
        checkpoint = torch.load(
            ruta_pesos, map_location="cpu", weights_only=True
        )
        estado = checkpoint["model_state_dict"]
        modelo.load_state_dict(estado, strict=True)

        copia_exacta = all(
            torch.equal(valor, estado[clave])
            for clave, valor in modelo.state_dict().items()
        )
        if not copia_exacta:
            raise RuntimeError("La copia de pesos no es exacta.")

        registro["carga_estricta"] = True
        registro["copia_exacta_pesos"] = True
        modelo.eval()
        del checkpoint, estado

        print("Bitácora previa:", ruta_decisiones)
        print("Carga estricta y copia de pesos: CORRECTAS")
        print("Preprocesamiento: provisional, documentado.")
        print("Canales de salida:", canales)

        with rasterio.open(RUTA_IMAGEN_INFERENCIA) as src, \
             rasterio.open(RUTA_RESNET_NIVEL1_VERIFICADA) as control:

            alto, ancho = src.height, src.width
            cuadricula = PLAN_INFERENCIA["cuadricula"]

            if (
                src.count != 4
                or ancho != cuadricula["ancho"]
                or alto != cuadricula["alto"]
                or list(src.transform)[:6] != cuadricula["transformacion"]
                or not crs_equivalentes_xy(
                    src.crs,
                    rasterio.crs.CRS.from_wkt(cuadricula["crs_wkt"]),
                )
                or control.width != ancho
                or control.height != alto
                or control.transform != src.transform
                or not crs_equivalentes_xy(src.crs, control.crs)
                or control.nodata != 255
            ):
                raise RuntimeError("Cuadrícula incompatible con el protocolo.")

            bytes_necesarios = alto * ancho * (canales * 4 + 2)
            if shutil.disk_usage(carpeta).free < bytes_necesarios + 300_000_000:
                raise RuntimeError("Espacio insuficiente para los temporales.")

            sumas = np.memmap(
                ruta_sumas, mode="w+", dtype="float32",
                shape=(canales, alto, ancho),
            )
            aportes = np.memmap(
                ruta_aportes, mode="w+", dtype="uint16",
                shape=(alto, ancho),
            )
            sumas[:] = 0
            aportes[:] = 0

            ventanas = VENTANAS_COMUNES[nivel]
            ejecutadas = 0
            omitidas = 0

            with torch.inference_mode():
                for numero, (col, fila, w, h) in enumerate(ventanas, 1):
                    datos, validos = validez_imagen_unet29(
                        src, Window(col, fila, w, h)
                    )

                    if not validos.any():
                        omitidas += 1
                    else:
                        datos[:, ~validos] = 0
                        datos = (datos / 10000.0 - medias) / (desv + 1e-6)
                        datos[:, ~validos] = 0

                        if not np.isfinite(datos).all():
                            raise RuntimeError("Entrada normalizada no finita.")

                        entrada = torch.from_numpy(
                            np.ascontiguousarray(datos)
                        ).unsqueeze(0)

                        logits = modelo(entrada)
                        if (
                            list(logits.shape) != [1, canales, h, w]
                            or not torch.isfinite(logits).all().item()
                        ):
                            raise RuntimeError(
                                f"Salida incorrecta en ventana {numero}."
                            )

                        probabilidades = torch.softmax(
                            logits, dim=1
                        )[0].cpu().numpy()

                        sumas[:, fila:fila+h, col:col+w] += (
                            probabilidades * validos[None, :, :]
                        )
                        aportes[fila:fila+h, col:col+w] += (
                            validos.astype(np.uint16)
                        )

                        ejecutadas += 1
                        del entrada, logits, probabilidades

                    if numero % 250 == 0 or numero == len(ventanas):
                        print(
                            f"UNET {nivel} | {numero}/{len(ventanas)} | "
                            f"Ejecutadas: {ejecutadas} | Sin datos: {omitidas} | "
                            f"Minutos: {(time.perf_counter()-inicio)/60:.1f}",
                            flush=True,
                        )

            sumas.flush()
            aportes.flush()

            def reconstruir(fila, h):
                ventana = Window(0, fila, ancho, h)
                _, validos = validez_imagen_unet29(src, ventana)
                n = np.array(aportes[fila:fila+h, :], copy=True)
                p = np.array(sumas[:, fila:fila+h, :], copy=True)

                if not np.array_equal(n > 0, validos):
                    raise RuntimeError("Cobertura distinta de la validez.")

                if not np.array_equal(
                    control.read_masks(1, window=ventana) > 0,
                    validos,
                ):
                    raise RuntimeError("La máscara difiere de ResNet50.")

                if not np.isfinite(p).all() or np.any(p < 0):
                    raise RuntimeError("Acumulador incorrecto.")

                p /= np.maximum(n[None, :, :], 1)

                if validos.any() and not np.allclose(
                    p.sum(axis=0)[validos], 1.0,
                    atol=1e-5, rtol=1e-5,
                ):
                    raise RuntimeError("Promedio de probabilidades incorrecto.")

                # No desplazar índices ni eliminar el canal 0.
                salida = p.argmax(axis=0).astype(np.uint8)
                salida[~validos] = 255
                return ventana, salida, validos

            perfil = {
                "driver": "GTiff",
                "width": ancho, "height": alto,
                "count": 1, "dtype": "uint8", "nodata": 255,
                "crs": src.crs, "transform": src.transform,
                "compress": "LZW", "tiled": True,
                "blockxsize": 256, "blockysize": 256,
            }

            with rasterio.open(parcial, "w", **perfil) as dst:
                for fila in range(0, alto, 256):
                    ventana, salida, _ = reconstruir(
                        fila, min(256, alto - fila)
                    )
                    dst.write(salida, 1, window=ventana)

                leyenda = dict(CLASES_COMUNES[nivel])
                leyenda[0] = "Sin clasificación"
                leyenda[255] = "NoData de entrada"

                dst.update_tags(
                    modelo="UNET",
                    nivel=nivel,
                    fecha_colombia=fecha_colombia_29(),
                    estado="Preprocesamiento provisional; evaluación pendiente",
                    combinacion="Promedio uniforme de probabilidades",
                    bitacora=ruta_decisiones.name,
                    sha256_pesos=registro["sha256_pesos"],
                    clases=json.dumps(leyenda, ensure_ascii=False),
                )

            print("Verificando TIFF, máscara y valores píxel por píxel...")
            conteos = np.zeros(256, dtype=np.int64)

            with rasterio.open(parcial) as pred:
                if (
                    pred.width != ancho
                    or pred.height != alto
                    or pred.transform != src.transform
                    or not crs_equivalentes_xy(src.crs, pred.crs)
                    or pred.count != 1
                    or pred.dtypes != ("uint8",)
                    or pred.nodata != 255
                ):
                    raise RuntimeError("Geometría o formato incorrectos.")

                for fila in range(0, alto, 256):
                    ventana, esperado, validos = reconstruir(
                        fila, min(256, alto - fila)
                    )
                    almacenado = pred.read(1, window=ventana)

                    if not np.array_equal(almacenado, esperado):
                        raise RuntimeError("El TIFF difiere del acumulador.")

                    if not np.array_equal(
                        pred.read_masks(1, window=ventana) > 0,
                        validos,
                    ):
                        raise RuntimeError("Máscara del TIFF incorrecta.")

                    conteos += np.bincount(
                        almacenado.ravel(), minlength=256
                    )

            if (
                int(conteos.sum()) != alto * ancho
                or int(conteos[:255].sum()) != 12323120
            ):
                raise RuntimeError("Número de píxeles inesperado.")

        parcial.rename(final)

        registro.update({
            "completada": True,
            "archivo_final": str(final),
            "sha256_tiff": sha256_archivo_29(final),
            "ventanas_ejecutadas": ejecutadas,
            "ventanas_sin_datos": omitidas,
            "pixeles_verificados": int(conteos.sum()),
            "pixeles_entrada_valida": int(conteos[:255].sum()),
            "pixeles_sin_clasificacion": int(conteos[0]),
            "pixeles_con_clase_tematica": int(conteos[1:canales].sum()),
            "pixeles_nodata": int(conteos[255]),
            "geometria_verificada_xy": True,
            "mascara_identica_a_resnet_nivel1": True,
            "coincidencia_pixel_a_pixel_con_acumuladores": True,
            "conteos_por_codigo": {
                int(c): int(conteos[c]) for c in np.flatnonzero(conteos)
            },
            "alcance": (
                "Integridad técnica verificada. No acredita exactitud "
                "temática ni resuelve las limitaciones documentadas."
            ),
        })

    except Exception as error:
        registro["error"] = f"{type(error).__name__}: {error}"
        raise

    finally:
        for acumulador in (sumas, aportes):
            if acumulador is not None:
                acumulador.flush()
                acumulador._mmap.close()

        modelo = None
        gc.collect()

        # Ante un fallo se conservan temporales para recuperación.
        if registro["completada"]:
            for temporal in (ruta_sumas, ruta_aportes):
                try:
                    temporal.unlink()
                except OSError as error:
                    registro.setdefault(
                        "temporales_no_eliminados", []
                    ).append(f"{temporal}: {error}")

        registro["fin_colombia"] = fecha_colombia_29()
        registro["segundos_totales"] = time.perf_counter() - inicio

        with ruta_registro.open("x", encoding="utf-8") as archivo:
            json.dump(registro, archivo, ensure_ascii=False, indent=2)

        print("\nRegistro de ejecución:", ruta_registro)
        print("Bitácora de decisiones:", ruta_decisiones)

    print("\nArchivo verificado:", final)
    print("Píxeles de entrada válida:", registro["pixeles_entrada_valida"])
    print("Píxeles sin clasificación:", registro["pixeles_sin_clasificacion"])
    print("Píxeles con clase temática:", registro["pixeles_con_clase_tematica"])
    print("Píxeles NoData:", registro["pixeles_nodata"])
    print("Conteos por código:", registro["conteos_por_codigo"])
    print("Máscara idéntica a ResNet50 nivel 1: True")
    print("Coincidencia píxel a píxel con acumuladores: True")

    return final, registro


RUTA_UNET_NIVEL1_VERIFICADA, RESULTADO_UNET_NIVEL1 = (
    ejecutar_unet_completa("nivel_1", 29)
)

print("\nCELDA 29 TERMINADA.")
print("U-Net nivel 1 generado, verificado y documentado.")

Bitácora previa: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_UNET_nivel_1_documentada_20261008_003424_347570\decisiones_metodologicas.json
Carga estricta y copia de pesos: CORRECTAS
Preprocesamiento: provisional, documentado.
Canales de salida: 6
UNET nivel_1 | 250/1326 | Ejecutadas: 49 | Sin datos: 201 | Minutos: 0.3
UNET nivel_1 | 500/1326 | Ejecutadas: 198 | Sin datos: 302 | Minutos: 1.1
UNET nivel_1 | 750/1326 | Ejecutadas: 436 | Sin datos: 314 | Minutos: 2.3
UNET nivel_1 | 1000/1326 | Ejecutadas: 685 | Sin datos: 315 | Minutos: 3.6
UNET nivel_1 | 1250/1326 | Ejecutadas: 889 | Sin datos: 361 | Minutos: 4.7
UNET nivel_1 | 1326/1326 | Ejecutadas: 914 | Sin datos: 412 | Minutos: 4.8
Verificando TIFF, máscara y valores píxel por píxel...

Registro de ejecución: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_UNET_nivel_1_documentada_20261008_003424_347570\celda_29_inferencia.json
Bitácora de decisiones: D:\PROY

In [37]:
# ============================================================
# CELDA 30 — U-NET NIVEL 2 CON BITÁCORA DE DECISIONES
# ============================================================

if not callable(globals().get("ejecutar_unet_completa")):
    raise RuntimeError(
        "La función de la celda 29 no está disponible. "
        "Si reiniciaste el kernel, no repitas las predicciones: "
        "primero debemos recuperar las definiciones y los registros."
    )

resultado_nivel1 = globals().get("RESULTADO_UNET_NIVEL1", {})

if not resultado_nivel1.get("completada", False):
    raise RuntimeError(
        "No está disponible la confirmación de U-Net nivel 1 "
        "en esta sesión del cuaderno."
    )

RUTA_UNET_NIVEL2_VERIFICADA, RESULTADO_UNET_NIVEL2 = (
    ejecutar_unet_completa(
        nivel="nivel_2",
        numero_celda=30,
    )
)

print("\nCELDA 30 TERMINADA.")
print("U-Net nivel 2 generado, verificado y documentado.")

Bitácora previa: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Inferencia_UNET_nivel_2_documentada_20261008_080417_807532\decisiones_metodologicas.json
Carga estricta y copia de pesos: CORRECTAS
Preprocesamiento: provisional, documentado.
Canales de salida: 14
UNET nivel_2 | 250/5304 | Ejecutadas: 9 | Sin datos: 241 | Minutos: 0.0
UNET nivel_2 | 500/5304 | Ejecutadas: 26 | Sin datos: 474 | Minutos: 0.1
UNET nivel_2 | 750/5304 | Ejecutadas: 67 | Sin datos: 683 | Minutos: 0.1
UNET nivel_2 | 1000/5304 | Ejecutadas: 139 | Sin datos: 861 | Minutos: 0.2
UNET nivel_2 | 1250/5304 | Ejecutadas: 224 | Sin datos: 1026 | Minutos: 0.3
UNET nivel_2 | 1500/5304 | Ejecutadas: 334 | Sin datos: 1166 | Minutos: 0.5
UNET nivel_2 | 1750/5304 | Ejecutadas: 489 | Sin datos: 1261 | Minutos: 0.7
UNET nivel_2 | 2000/5304 | Ejecutadas: 649 | Sin datos: 1351 | Minutos: 0.9
UNET nivel_2 | 2250/5304 | Ejecutadas: 852 | Sin datos: 1398 | Minutos: 1.2
UNET nivel_2 | 2500/5304 | Ejecutadas: 

In [38]:
# ============================================================
# CELDA 31 — SUPERFICIES COMUNES DE EVALUACIÓN
# No modifica predicciones ni referencias.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
from contextlib import ExitStack
import json

import numpy as np
import rasterio
from rasterio.windows import Window


requisitos = [
    "RUTA_IMAGEN_INFERENCIA",
    "RUTA_UNET_NIVEL1_VERIFICADA",
    "RUTA_UNET_NIVEL2_VERIFICADA",
    "RUTA_RESNET_NIVEL1_VERIFICADA",
    "RUTA_RESNET_NIVEL2_VERIFICADA",
    "RUTA_DENSENET_NIVEL1_VERIFICADA",
    "RUTA_DENSENET_NIVEL2_VERIFICADA",
    "REFERENCIAS_HOMOLOGADAS",
    "CLASES_COMUNES",
    "CARPETA_AUDITORIA",
    "crs_equivalentes_xy",
]

faltan = [nombre for nombre in requisitos if nombre not in globals()]
if faltan:
    raise RuntimeError(
        f"Faltan variables: {faltan}. "
        "Debemos recuperar sus rutas; no repetir las predicciones."
    )

PREDICCIONES_COMPARACION = {
    "UNET": {
        "nivel_1": Path(RUTA_UNET_NIVEL1_VERIFICADA),
        "nivel_2": Path(RUTA_UNET_NIVEL2_VERIFICADA),
    },
    "RESNET50": {
        "nivel_1": Path(RUTA_RESNET_NIVEL1_VERIFICADA),
        "nivel_2": Path(RUTA_RESNET_NIVEL2_VERIFICADA),
    },
    "DENSENET121": {
        "nivel_1": Path(RUTA_DENSENET_NIVEL1_VERIFICADA),
        "nivel_2": Path(RUTA_DENSENET_NIVEL2_VERIFICADA),
    },
}

RUTA_CANDIDATA_31 = Path(
    r"D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos"
    r"\20261007_191328_200990"
    r"\Diagnostico_espacial_20261007_221046_888506"
    r"\Mascara_candidata_espacial_PRELIMINAR.tif"
)

# Correspondencia semántica del catálogo del dataset.
PADRES_31 = {
    1: 1, 2: 2, 3: 2, 4: 1, 5: 3,
    6: 4, 7: 4, 8: 5, 9: 4, 10: 4,
    11: 1, 12: 1, 13: 2,
}

tabla_padres = np.zeros(256, dtype=np.uint8)
for hijo, padre in PADRES_31.items():
    tabla_padres[hijo] = padre

zona_colombia = timezone(timedelta(hours=-5))
fecha_31 = datetime.now(zona_colombia)

CARPETA_EVALUACION_COMUN = Path(CARPETA_AUDITORIA) / (
    "Evaluacion_comun_" + fecha_31.strftime("%Y%m%d_%H%M%S_%f")
)
CARPETA_EVALUACION_COMUN.mkdir(exist_ok=False)

RUTAS_MASCARAS_EVALUACION = {
    "concordancia_comun": (
        CARPETA_EVALUACION_COMUN / "Mascara_concordancia_comun.tif"
    ),
    "candidata_fuera_huellas": (
        CARPETA_EVALUACION_COMUN / "Mascara_candidata_fuera_huellas.tif"
    ),
}

ruta_registro_31 = CARPETA_EVALUACION_COMUN / "celda_31_superficies.json"

registro_31 = {
    "celda": 31,
    "fecha_colombia": fecha_31.isoformat(),
    "imagen": str(RUTA_IMAGEN_INFERENCIA),
    "referencias": {
        nivel: str(REFERENCIAS_HOMOLOGADAS[nivel])
        for nivel in ("nivel_1", "nivel_2")
    },
    "predicciones": {
        modelo: {nivel: str(ruta) for nivel, ruta in niveles.items()}
        for modelo, niveles in PREDICCIONES_COMPARACION.items()
    },
    "fuente_mascara_candidata": str(RUTA_CANDIDATA_31),
    "regla": (
        "Validez de imagen y de ambas referencias; "
        "sin selección por resultados de los modelos."
    ),
    "alcance": {
        "concordancia_comun": (
            "Concordancia con referencia IDEAM 2022; "
            "no equivale a exactitud independiente contemporánea."
        ),
        "candidata_fuera_huellas": (
            "Área fuera de las huellas de entrenamiento/validación "
            "auditadas en la celda 14; no acredita por sí sola "
            "independencia espacial ni representación de todas las clases."
        ),
    },
    "limitaciones": [
        "No se incorpora una máscara adicional de nubes validada.",
        "Persisten las limitaciones documentadas de U-Net.",
        "La referencia y la imagen corresponden a fechas diferentes.",
        "Ausencia de ciertas clases en las superficies evaluadas.",
    ],
    "completada": False,
}

SUPERFICIES_EVALUACION_VERIFICADAS = False
totales = {ambito: 0 for ambito in RUTAS_MASCARAS_EVALUACION}
soportes = {
    ambito: {
        "nivel_1": np.zeros(6, dtype=np.int64),
        "nivel_2": np.zeros(14, dtype=np.int64),
    }
    for ambito in totales
}
pixeles_imagen = 0


def verificar_cuadricula_31(src, base):
    if (
        src.width != base.width
        or src.height != base.height
        or src.transform != base.transform
        or not crs_equivalentes_xy(src.crs, base.crs)
    ):
        raise RuntimeError(f"Cuadrícula incompatible: {src.name}")


try:
    with ExitStack() as pila:
        imagen = pila.enter_context(
            rasterio.open(RUTA_IMAGEN_INFERENCIA)
        )
        referencias = {
            nivel: pila.enter_context(
                rasterio.open(REFERENCIAS_HOMOLOGADAS[nivel])
            )
            for nivel in ("nivel_1", "nivel_2")
        }
        candidata = pila.enter_context(
            rasterio.open(RUTA_CANDIDATA_31)
        )
        predicciones = {
            (modelo, nivel): pila.enter_context(rasterio.open(ruta))
            for modelo, niveles in PREDICCIONES_COMPARACION.items()
            for nivel, ruta in niveles.items()
        }

        for src in [
            *referencias.values(), candidata, *predicciones.values()
        ]:
            verificar_cuadricula_31(src, imagen)
            if src.count != 1:
                raise RuntimeError(f"Se esperaba una banda: {src.name}")

        for src in predicciones.values():
            if src.nodata != 255:
                raise RuntimeError(f"NoData inesperado: {src.name}")

        perfil = {
            "driver": "GTiff",
            "width": imagen.width,
            "height": imagen.height,
            "count": 1,
            "dtype": "uint8",
            "crs": imagen.crs,
            "transform": imagen.transform,
            "nodata": None,  # 0=excluido; 1=incluido
            "compress": "LZW",
        }

        salidas = {
            ambito: pila.enter_context(rasterio.open(ruta, "w", **perfil))
            for ambito, ruta in RUTAS_MASCARAS_EVALUACION.items()
        }

        for fila in range(0, imagen.height, 256):
            h = min(256, imagen.height - fila)
            ventana = Window(0, fila, imagen.width, h)

            datos = imagen.read([1, 2, 3, 4], window=ventana)
            valida = np.isfinite(datos).all(axis=0)
            if imagen.nodata is not None:
                valida &= (datos != imagen.nodata).all(axis=0)
            valida &= (
                imagen.read_masks([1, 2, 3, 4], window=ventana) > 0
            ).all(axis=0)
            pixeles_imagen += int(valida.sum())

            refs = {}
            comun = valida.copy()

            for nivel, src in referencias.items():
                valores = src.read(1, window=ventana)
                validos_ref = src.read_masks(1, window=ventana) > 0
                validos_ref &= valores != 0

                codigos = np.arange(
                    1, 6 if nivel == "nivel_1" else 14
                )
                if np.any(validos_ref & ~np.isin(valores, codigos)):
                    raise RuntimeError(
                        f"Códigos de referencia inesperados: {nivel}"
                    )

                refs[nivel] = valores
                comun &= validos_ref

            # Las predicciones se comprueban, pero no seleccionan la muestra.
            for (modelo, nivel), src in predicciones.items():
                p = src.read(1, window=ventana)
                mascara_p = src.read_masks(1, window=ventana) > 0

                if (
                    not np.array_equal(mascara_p, valida)
                    or not np.array_equal(p != 255, valida)
                ):
                    raise RuntimeError(
                        f"Cobertura diferente: {modelo}, {nivel}"
                    )

                k = 5 if nivel == "nivel_1" else 13
                minimo = 0 if modelo == "UNET" else 1

                if np.any(
                    valida & ((p < minimo) | (p > k))
                ):
                    raise RuntimeError(
                        f"Códigos inesperados: {modelo}, {nivel}"
                    )

            if np.any(
                comun
                & (tabla_padres[refs["nivel_2"]] != refs["nivel_1"])
            ):
                raise RuntimeError("Las referencias discrepan en la jerarquía.")

            c = candidata.read(1, window=ventana)
            if not np.isin(c, [0, 1]).all():
                raise RuntimeError("La máscara candidata no es binaria.")

            seleccion_candidata = c == 1
            if np.any(seleccion_candidata & ~comun):
                raise RuntimeError(
                    "La candidata incluye píxeles fuera de la superficie común."
                )

            mascaras = {
                "concordancia_comun": comun,
                "candidata_fuera_huellas": seleccion_candidata,
            }

            for ambito, mascara in mascaras.items():
                salidas[ambito].write(
                    mascara.astype(np.uint8), 1, window=ventana
                )
                totales[ambito] += int(mascara.sum())

                for nivel in referencias:
                    soportes[ambito][nivel] += np.bincount(
                        refs[nivel][mascara],
                        minlength=len(soportes[ambito][nivel]),
                    )

        if pixeles_imagen != 12323120:
            raise RuntimeError("Cambió el número de píxeles válidos de entrada.")

        esperados = {
            "concordancia_comun": 12319921,
            "candidata_fuera_huellas": 332568,
        }
        if totales != esperados:
            raise RuntimeError(
                f"Los totales difieren del diagnóstico previo: {totales}"
            )

        for ambito, dst in salidas.items():
            dst.update_tags(
                significado="0=excluido; 1=incluido",
                ambito=ambito,
                fecha_colombia=fecha_31.isoformat(),
                alcance=registro_31["alcance"][ambito],
            )

    # Verificar los conteos de las máscaras efectivamente escritas.
    for ambito, ruta in RUTAS_MASCARAS_EVALUACION.items():
        total_leido = 0
        with rasterio.open(ruta) as src:
            for _, ventana in src.block_windows(1):
                valores = src.read(1, window=ventana)
                if not np.isin(valores, [0, 1]).all():
                    raise RuntimeError("Máscara escrita con valores inesperados.")
                total_leido += int(np.count_nonzero(valores == 1))

        if total_leido != totales[ambito]:
            raise RuntimeError("El conteo cambió durante la escritura.")

    registro_31["pixeles_imagen_valida"] = pixeles_imagen
    registro_31["superficies"] = {}

    for ambito in totales:
        detalle = {
            "ruta_mascara": str(RUTAS_MASCARAS_EVALUACION[ambito]),
            "pixeles": totales[ambito],
            "soporte_referencia": {},
            "clases_sin_referencia": {},
        }

        print("\n" + "=" * 75)
        print(ambito)
        print("Píxeles únicos:", totales[ambito])

        for nivel in ("nivel_1", "nivel_2"):
            conteos = {
                c: int(soportes[ambito][nivel][c])
                for c in range(1, len(soportes[ambito][nivel]))
            }
            ausentes = [c for c, n in conteos.items() if n == 0]

            detalle["soporte_referencia"][nivel] = conteos
            detalle["clases_sin_referencia"][nivel] = ausentes

            print("\n", nivel)
            for codigo, cantidad in conteos.items():
                print(
                    f"  {codigo} — {CLASES_COMUNES[nivel][codigo]}: "
                    f"{cantidad:,}"
                )
            print("Clases sin referencia:", ausentes)

        registro_31["superficies"][ambito] = detalle

    registro_31["completada"] = True
    SUPERFICIES_EVALUACION_VERIFICADAS = True

except Exception as error:
    registro_31["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    with ruta_registro_31.open("x", encoding="utf-8") as archivo:
        json.dump(registro_31, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro:", ruta_registro_31)

print("\nCELDA 31 TERMINADA.")
print("Superficies comunes guardadas y verificadas.")
print("Todavía no se calcularon métricas.")


concordancia_comun
Píxeles únicos: 12319921

 nivel_1
  1 — 2. Territorios agrícolas: 8,674,615
  2 — 3. Bosques y áreas seminaturales: 3,374,666
  3 — 5. Superficies de agua: 174,659
  4 — 1. Territorios artificializados: 87,395
  5 — 4. Áreas húmedas: 8,586
Clases sin referencia: []

 nivel_2
  1 — 2.4 Áreas agrícolas heterogéneas: 5,339,526
  2 — 3.2 Áreas con vegetación herbácea o arbustiva: 2,509,303
  3 — 3.1 Bosques: 838,692
  4 — 2.3 Pastos: 3,128,559
  5 — 5.1 Aguas continentales: 174,659
  6 — 1.1 Zonas urbanizadas: 64,148
  7 — 1.3 Zonas de extracción minera y escombreras: 4,817
  8 — 4.1 Áreas húmedas continentales: 8,586
  9 — 1.2 Zonas industriales o comerciales y redes de comunicación: 18,430
  10 — 1.4 Zonas verdes artificializadas no agrícolas: 0
  11 — 2.1 Cultivos transitorios: 145,835
  12 — 2.2 Cultivos permanentes: 60,695
  13 — 3.3 Áreas abiertas sin o con poca vegetación: 26,671
Clases sin referencia: [10]

candidata_fuera_huellas
Píxeles únicos: 332568

 nivel

In [39]:
# ============================================================
# CELDA 32 — MATRICES DE CONFUSIÓN CON NOMBRES DE CLASES
# Filas: referencia. Columnas: predicción.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
import json

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from openpyxl.styles import Alignment, Font


if not globals().get("SUPERFICIES_EVALUACION_VERIFICADAS", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 31.")

zona_colombia = timezone(timedelta(hours=-5))
fecha_32 = datetime.now(zona_colombia)

carpeta_32 = Path(CARPETA_EVALUACION_COMUN) / (
    "Matrices_" + fecha_32.strftime("%Y%m%d_%H%M%S_%f")
)
carpeta_32.mkdir(exist_ok=False)

ruta_excel_32 = carpeta_32 / "Matrices_confusion_comparables.xlsx"
ruta_npz_32 = carpeta_32 / "Matrices_confusion_conteos.npz"
ruta_registro_32 = carpeta_32 / "celda_32_matrices.json"

with (
    Path(CARPETA_EVALUACION_COMUN) / "celda_31_superficies.json"
).open(encoding="utf-8") as archivo:
    auditoria_31 = json.load(archivo)

if not auditoria_31.get("completada", False):
    raise RuntimeError("El registro de superficies no está completo.")

# Estructura interna de cada matriz:
# K filas: códigos de referencia 1..K
# K+1 columnas: códigos de predicción 0..K
# La columna 0 se mostrará al final en Excel.
MATRICES_COMPARACION = {}
matrices_npz = {}
tablas_excel = {}
resumen = []
MATRICES_COMPARACION_VERIFICADAS = False

registro_32 = {
    "celda": 32,
    "fecha_colombia": fecha_32.isoformat(),
    "registro_superficies": str(
        Path(CARPETA_EVALUACION_COMUN) / "celda_31_superficies.json"
    ),
    "orientacion": "Filas=referencia; columnas=predicción",
    "estructura_interna": {
        "filas": "Códigos 1..K",
        "columnas": "Códigos 0..K",
        "codigo_0": "Sin clasificación; no es una clase temática",
    },
    "reglas": [
        "Usar las máscaras comunes de la celda 31.",
        "Contar cada píxel seleccionado una sola vez.",
        "Conservar todas las clases del catálogo.",
        "No excluir píxeles según la clase predicha.",
        "Detenerse si aparece NoData dentro de la superficie evaluada.",
        "No remapear ni modificar las predicciones.",
    ],
    "ambitos": auditoria_31["alcance"],
    "completada": False,
}


def verificar_cuadricula_32(src, base):
    if (
        src.width != base.width
        or src.height != base.height
        or src.transform != base.transform
        or not crs_equivalentes_xy(src.crs, base.crs)
    ):
        raise RuntimeError(f"Cuadrícula incompatible: {src.name}")


try:
    for ambito, ruta_mascara in RUTAS_MASCARAS_EVALUACION.items():
        MATRICES_COMPARACION[ambito] = {}

        superficie = auditoria_31["superficies"][ambito]
        total_esperado = int(superficie["pixeles"])

        for nivel in ("nivel_1", "nivel_2"):
            MATRICES_COMPARACION[ambito][nivel] = {}

            k = 5 if nivel == "nivel_1" else 13
            soporte_esperado = np.asarray(
                [
                    superficie["soporte_referencia"][nivel][str(c)]
                    for c in range(1, k + 1)
                ],
                dtype=np.int64,
            )

            for modelo in ("UNET", "RESNET50", "DENSENET121"):
                matriz = np.zeros((k, k + 1), dtype=np.int64)

                with rasterio.open(ruta_mascara) as mascara, \
                     rasterio.open(
                         REFERENCIAS_HOMOLOGADAS[nivel]
                     ) as referencia, \
                     rasterio.open(
                         PREDICCIONES_COMPARACION[modelo][nivel]
                     ) as prediccion:

                    verificar_cuadricula_32(referencia, mascara)
                    verificar_cuadricula_32(prediccion, mascara)

                    if any(
                        src.count != 1
                        for src in (mascara, referencia, prediccion)
                    ):
                        raise RuntimeError("Se esperaba una banda por archivo.")

                    for fila in range(0, mascara.height, 256):
                        h = min(256, mascara.height - fila)
                        ventana = Window(0, fila, mascara.width, h)

                        m = mascara.read(1, window=ventana)
                        if not np.isin(m, [0, 1]).all():
                            raise RuntimeError("La máscara no es binaria.")

                        seleccion = m == 1
                        if not seleccion.any():
                            continue

                        r = referencia.read(1, window=ventana)[seleccion]
                        p = prediccion.read(1, window=ventana)[seleccion]

                        if not (
                            referencia.read_masks(1, window=ventana)[seleccion] > 0
                        ).all():
                            raise RuntimeError("Referencia inválida en la muestra.")

                        if not (
                            prediccion.read_masks(1, window=ventana)[seleccion] > 0
                        ).all():
                            raise RuntimeError(
                                f"NoData dentro de la muestra: {modelo}, {nivel}"
                            )

                        minimo_pred = 0 if modelo == "UNET" else 1

                        if not np.isin(r, np.arange(1, k + 1)).all():
                            raise RuntimeError("Código de referencia inesperado.")

                        if not np.isin(
                            p, np.arange(minimo_pred, k + 1)
                        ).all():
                            raise RuntimeError("Código de predicción inesperado.")

                        # Convertir antes de operar para evitar overflow uint8.
                        r = r.astype(np.int64)
                        p = p.astype(np.int64)

                        indices = (r - 1) * (k + 1) + p
                        matriz += np.bincount(
                            indices,
                            minlength=k * (k + 1),
                        ).reshape(k, k + 1)

                total = int(matriz.sum())
                soporte = matriz.sum(axis=1)

                if total != total_esperado:
                    raise RuntimeError(
                        f"Total incorrecto: {ambito}, {nivel}, {modelo}"
                    )

                if not np.array_equal(soporte, soporte_esperado):
                    raise RuntimeError(
                        f"Soporte distinto de la celda 31: {modelo}, {nivel}"
                    )

                MATRICES_COMPARACION[ambito][nivel][modelo] = matriz

                clave = f"{ambito}__{nivel}__{modelo}"
                matrices_npz[clave] = matriz

                # Presentación: clases temáticas primero, código 0 al final.
                orden_columnas = list(range(1, k + 1)) + [0]

                nombres_filas = [
                    f"{c} — {CLASES_COMUNES[nivel][c]}"
                    for c in range(1, k + 1)
                ]
                nombres_columnas = nombres_filas + ["0 — Sin clasificación"]

                tabla = pd.DataFrame(
                    matriz[:, orden_columnas],
                    index=nombres_filas,
                    columns=nombres_columnas,
                )
                tabla.index.name = "Referencia / Predicción"

                abreviatura = (
                    "COMUN" if ambito == "concordancia_comun" else "CAND"
                )
                numero_nivel = 1 if nivel == "nivel_1" else 2
                hoja = f"{modelo}_N{numero_nivel}_{abreviatura}"
                tablas_excel[hoja] = tabla

                ausentes = [
                    c for c in range(1, k + 1)
                    if soporte[c - 1] == 0
                ]

                resumen.append({
                    "Ambito": ambito,
                    "Nivel": nivel,
                    "Modelo": modelo,
                    "Hoja": hoja,
                    "Clave_NPZ": clave,
                    "Pixeles": total,
                    "Sin_clasificacion": int(matriz[:, 0].sum()),
                    "Clases_sin_referencia": ", ".join(map(str, ausentes)),
                    "Total_verificado": True,
                    "Soporte_verificado": True,
                })

                print(
                    f"{ambito} | {nivel} | {modelo} | "
                    f"N={total:,} | "
                    f"Sin clasificación={int(matriz[:, 0].sum()):,} | "
                    "Total y soporte: CORRECTOS"
                )

    # Guardar conteos numéricos sin depender del formato de Excel.
    np.savez_compressed(ruta_npz_32, **matrices_npz)

    with np.load(ruta_npz_32, allow_pickle=False) as guardadas:
        for clave, matriz in matrices_npz.items():
            if not np.array_equal(guardadas[clave], matriz):
                raise RuntimeError(f"Error al guardar la matriz: {clave}")

    notas = [
        (
            "Orientación",
            "Filas: referencia IDEAM homologada. Columnas: predicción.",
        ),
        (
            "Clases ausentes",
            "Se conservan filas y columnas. Una fila nula indica ausencia "
            "en la referencia; no demuestra buen o mal desempeño de esa clase.",
        ),
        (
            "Sin clasificación",
            "La columna 0 conserva posibles salidas sin clase temática. "
            "Sus píxeles forman parte del total evaluado.",
        ),
        (
            "NoData",
            "El código 255 no es una clase. Su aparición dentro de la "
            "superficie seleccionada detiene el cálculo.",
        ),
        (
            "Concordancia común",
            auditoria_31["alcance"]["concordancia_comun"],
        ),
        (
            "Candidata fuera de huellas",
            auditoria_31["alcance"]["candidata_fuera_huellas"],
        ),
        (
            "U-Net",
            "Se mantienen las limitaciones documentadas del preprocesamiento "
            "y la trazabilidad de los canales.",
        ),
        (
            "Fecha",
            fecha_32.isoformat(),
        ),
    ]

    with pd.ExcelWriter(ruta_excel_32, engine="openpyxl") as escritor:
        pd.DataFrame(
            notas, columns=["Tema", "Descripción"]
        ).to_excel(escritor, sheet_name="LEEME", index=False)

        pd.DataFrame(resumen).to_excel(
            escritor, sheet_name="Resumen", index=False
        )

        for hoja, tabla in tablas_excel.items():
            tabla.to_excel(escritor, sheet_name=hoja)

        for hoja in escritor.book.worksheets:
            hoja.freeze_panes = (
                "B2" if hoja.title in tablas_excel else "A2"
            )

            for celda in hoja[1]:
                celda.font = Font(bold=True)
                celda.alignment = Alignment(
                    wrap_text=True, vertical="center"
                )

            if hoja.title in tablas_excel:
                hoja.column_dimensions["A"].width = 58
                hoja.row_dimensions[1].height = 115

                for columna in range(2, hoja.max_column + 1):
                    letra = hoja.cell(1, columna).column_letter
                    hoja.column_dimensions[letra].width = 25

                for fila in hoja.iter_rows(min_row=2):
                    fila[0].alignment = Alignment(wrap_text=True)
                    hoja.row_dimensions[fila[0].row].height = 42
                    for celda in fila[1:]:
                        celda.number_format = "#,##0"
            else:
                for columna in hoja.columns:
                    letra = columna[0].column_letter
                    hoja.column_dimensions[letra].width = (
                        100 if hoja.title == "LEEME" and letra == "B" else 28
                    )
                for fila in hoja.iter_rows(min_row=2):
                    for celda in fila:
                        celda.alignment = Alignment(
                            wrap_text=True, vertical="top"
                        )

    registro_32.update({
        "completada": True,
        "numero_matrices": len(matrices_npz),
        "excel": str(ruta_excel_32),
        "conteos_npz": str(ruta_npz_32),
        "resumen": resumen,
        "conteos_npz_relectura_verificados": True,
    })
    MATRICES_COMPARACION_VERIFICADAS = True

except Exception as error:
    registro_32["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    with ruta_registro_32.open("x", encoding="utf-8") as archivo:
        json.dump(registro_32, archivo, ensure_ascii=False, indent=2)

    print("\nRegistro:", ruta_registro_32)

print("\nExcel:", ruta_excel_32)
print("Conteos numéricos:", ruta_npz_32)
print("Matrices generadas:", len(matrices_npz))
print("\nCELDA 32 TERMINADA. Matrices generadas y verificadas.")

concordancia_comun | nivel_1 | UNET | N=12,319,921 | Sin clasificación=0 | Total y soporte: CORRECTOS
concordancia_comun | nivel_1 | RESNET50 | N=12,319,921 | Sin clasificación=0 | Total y soporte: CORRECTOS
concordancia_comun | nivel_1 | DENSENET121 | N=12,319,921 | Sin clasificación=0 | Total y soporte: CORRECTOS
concordancia_comun | nivel_2 | UNET | N=12,319,921 | Sin clasificación=0 | Total y soporte: CORRECTOS
concordancia_comun | nivel_2 | RESNET50 | N=12,319,921 | Sin clasificación=0 | Total y soporte: CORRECTOS
concordancia_comun | nivel_2 | DENSENET121 | N=12,319,921 | Sin clasificación=0 | Total y soporte: CORRECTOS
candidata_fuera_huellas | nivel_1 | UNET | N=332,568 | Sin clasificación=0 | Total y soporte: CORRECTOS
candidata_fuera_huellas | nivel_1 | RESNET50 | N=332,568 | Sin clasificación=0 | Total y soporte: CORRECTOS
candidata_fuera_huellas | nivel_1 | DENSENET121 | N=332,568 | Sin clasificación=0 | Total y soporte: CORRECTOS
candidata_fuera_huellas | nivel_2 | UNET | 

In [40]:
# ============================================================
# CELDA 33 — MÉTRICAS COMPARABLES A PARTIR DE LAS MATRICES
# No modifica mapas, máscaras ni matrices.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
import hashlib
import json

import numpy as np
import pandas as pd
import sklearn

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    jaccard_score,
    cohen_kappa_score,
    matthews_corrcoef,
)
from openpyxl.styles import Alignment, Font


if not globals().get("MATRICES_COMPARACION_VERIFICADAS", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 32.")

fecha_33 = datetime.now(timezone(timedelta(hours=-5)))

carpeta_33 = Path(CARPETA_EVALUACION_COMUN) / (
    "Metricas_" + fecha_33.strftime("%Y%m%d_%H%M%S_%f")
)
carpeta_33.mkdir(exist_ok=False)

ruta_excel_33 = carpeta_33 / "Metricas_comparables.xlsx"
ruta_registro_33 = carpeta_33 / "celda_33_metricas.json"
ruta_politica_33 = carpeta_33 / "Politica_calculo_metricas.json"

politica_33 = {
    "fecha_colombia": fecha_33.isoformat(),
    "escala": "Proporciones, no porcentajes; Kappa y MCC pueden ser negativos.",
    "macro_referencia": (
        "Media no ponderada sobre códigos con soporte de referencia positivo. "
        "Conjunto fijo entre modelos dentro de cada nivel y ámbito. "
        "Precisión indefinida por ausencia de predicciones se computa como 0."
    ),
    "macro_catalogo_z0": (
        "Media no ponderada sobre todas las clases temáticas del catálogo. "
        "Divisiones indefinidas sustituidas por 0 exclusivamente para "
        "esta convención de resumen."
    ),
    "detalle_por_clase": (
        "Divisiones por cero: no estimable (NE en Excel; null en JSON)."
    ),
    "sin_clasificacion": (
        "Código 0 excluido del catálogo temático macro, pero sus píxeles "
        "permanecen en N y cuentan como errores y falsos negativos."
    ),
    "clases_ausentes": (
        "Se conservan. Predicciones hacia ellas siguen contando como errores. "
        "Su recall no es estimable."
    ),
    "accuracy": "Suma de verdaderos positivos / N",
    "precision": "TP / (TP + FP)",
    "recall": "TP / (TP + FN)",
    "f1": "2*TP / (2*TP + FP + FN)",
    "iou": "TP / (TP + FP + FN)",
    "omision": "1 - recall",
    "comision": "1 - precision",
    "kappa": "Cohen Kappa sin ponderación ordinal, sobre todos los píxeles.",
    "mcc": "Matthews multiclase, sobre todos los píxeles.",
    "pesos_computacionales": (
        "Cada par referencia-predicción se representa una vez y se pondera "
        "por su conteo en la matriz. Equivale a repetir cada píxel; "
        "no es una ponderación por importancia de clase ni ajuste de área."
    ),
    "interpretacion": (
        "Concordancia con la referencia disponible. "
        "La superficie candidata no se declara automáticamente independiente. "
        "Persisten las limitaciones temporales, radiométricas y de U-Net."
    ),
    "fuentes": [
        "https://scikit-learn.org/stable/modules/generated/"
        "sklearn.metrics.precision_recall_fscore_support.html",
        "https://scikit-learn.org/stable/modules/generated/"
        "sklearn.metrics.jaccard_score.html",
        "https://scikit-learn.org/stable/modules/generated/"
        "sklearn.metrics.cohen_kappa_score.html",
        "https://scikit-learn.org/stable/modules/generated/"
        "sklearn.metrics.matthews_corrcoef.html",
    ],
}

# Política registrada antes de calcular los resultados.
with ruta_politica_33.open("x", encoding="utf-8") as archivo:
    json.dump(politica_33, archivo, ensure_ascii=False, indent=2)


def dividir_o_nan(numerador, denominador):
    numerador = np.asarray(numerador, dtype=np.float64)
    denominador = np.asarray(denominador, dtype=np.float64)
    return np.divide(
        numerador,
        denominador,
        out=np.full(numerador.shape, np.nan, dtype=np.float64),
        where=denominador != 0,
    )


registro_33 = {
    "celda": 33,
    "fecha_colombia": fecha_33.isoformat(),
    "scikit_learn": sklearn.__version__,
    "matrices_origen": str(ruta_npz_32),
    "politica": politica_33,
    "completada": False,
}

filas_generales = []
filas_clases = []
soportes_control = {}
METRICAS_COMPARACION_VERIFICADAS = False

try:
    # Identifica exactamente el archivo de conteos utilizado.
    registro_33["sha256_matrices"] = hashlib.sha256(
        Path(ruta_npz_32).read_bytes()
    ).hexdigest()

    with np.load(ruta_npz_32, allow_pickle=False) as archivo_matrices:
        for ambito in (
            "concordancia_comun",
            "candidata_fuera_huellas",
        ):
            for nivel in ("nivel_1", "nivel_2"):
                k = 5 if nivel == "nivel_1" else 13
                catalogo = np.arange(1, k + 1)

                for modelo in ("UNET", "RESNET50", "DENSENET121"):
                    clave = f"{ambito}__{nivel}__{modelo}"
                    matriz = archivo_matrices[clave]

                    if (
                        matriz.shape != (k, k + 1)
                        or not np.issubdtype(matriz.dtype, np.integer)
                        or np.any(matriz < 0)
                    ):
                        raise RuntimeError(f"Matriz incorrecta: {clave}")

                    # Columnas internas: 0..K. Filas: 1..K.
                    n = int(matriz.sum())
                    soporte = matriz.sum(axis=1)
                    predichos = matriz[:, 1:].sum(axis=0)
                    tp = matriz[np.arange(k), catalogo]
                    fp = predichos - tp
                    fn = soporte - tp

                    if n <= 0:
                        raise RuntimeError(f"Matriz vacía: {clave}")

                    llave_soporte = (ambito, nivel)
                    if llave_soporte not in soportes_control:
                        soportes_control[llave_soporte] = soporte.copy()
                    elif not np.array_equal(
                        soporte, soportes_control[llave_soporte]
                    ):
                        raise RuntimeError("Los modelos tienen soportes distintos.")

                    clases_con_ref = catalogo[soporte > 0]

                    precision = dividir_o_nan(tp, predichos)
                    recall = dividir_o_nan(tp, soporte)
                    f1 = dividir_o_nan(2 * tp, 2 * tp + fp + fn)
                    iou = dividir_o_nan(tp, tp + fp + fn)

                    # Representación compacta de todos los píxeles.
                    filas, columnas = np.nonzero(matriz)
                    y_true = filas + 1
                    y_pred = columnas
                    pesos = matriz[filas, columnas].astype(np.float64)

                    # Verificación de las fórmulas por clase.
                    p_sk, r_sk, f_sk, soporte_sk = (
                        precision_recall_fscore_support(
                            y_true,
                            y_pred,
                            labels=catalogo,
                            average=None,
                            sample_weight=pesos,
                            zero_division=0,
                        )
                    )

                    j_sk = jaccard_score(
                        y_true,
                        y_pred,
                        labels=catalogo,
                        average=None,
                        sample_weight=pesos,
                        zero_division=0,
                    )

                    for calculada, comprobada in [
                        (precision, p_sk),
                        (recall, r_sk),
                        (f1, f_sk),
                        (iou, j_sk),
                    ]:
                        if not np.allclose(
                            np.nan_to_num(calculada, nan=0.0),
                            comprobada,
                            atol=1e-12,
                            rtol=1e-12,
                        ):
                            raise RuntimeError(
                                f"Discrepancia en cálculo por clase: {clave}"
                            )

                    if not np.array_equal(soporte, soporte_sk):
                        raise RuntimeError("Discrepancia de soporte.")

                    accuracy = float(tp.sum() / n)
                    if not np.isclose(
                        accuracy,
                        accuracy_score(
                            y_true, y_pred, sample_weight=pesos
                        ),
                    ):
                        raise RuntimeError("Discrepancia en Accuracy.")

                    kappa = float(cohen_kappa_score(
                        y_true,
                        y_pred,
                        labels=np.arange(k + 1),
                        sample_weight=pesos,
                    ))

                    # Si no hay variación en una marginal, MCC no es
                    # estimable como correlación: conservar NaN.
                    marginal_ref = np.r_[0.0, soporte.astype(np.float64)]
                    marginal_pred = matriz.sum(axis=0).astype(np.float64)
                    var_ref = float(n)**2 - float(marginal_ref @ marginal_ref)
                    var_pred = float(n)**2 - float(marginal_pred @ marginal_pred)

                    mcc = (
                        float(matthews_corrcoef(
                            y_true, y_pred, sample_weight=pesos
                        ))
                        if var_ref > 0 and var_pred > 0
                        else np.nan
                    )

                    elegibles = soporte > 0

                    general = {
                        "Ambito": ambito,
                        "Nivel": nivel,
                        "Modelo": modelo,
                        "N": n,
                        "Clases_catalogo": k,
                        "Clases_con_referencia": int(elegibles.sum()),
                        "Codigos_macro_referencia": ", ".join(
                            map(str, clases_con_ref.tolist())
                        ),
                        "Sin_clasificacion": int(matriz[:, 0].sum()),
                        "Accuracy": accuracy,
                        "Kappa": kappa,
                        "MCC": mcc,
                    }

                    for nombre, valores in [
                        ("Precision", p_sk),
                        ("Recall", r_sk),
                        ("F1", f_sk),
                        ("IoU", j_sk),
                    ]:
                        general[f"{nombre}_macro_referencia"] = float(
                            valores[elegibles].mean()
                        )
                        general[f"{nombre}_macro_catalogo_z0"] = float(
                            valores.mean()
                        )

                    filas_generales.append(general)

                    for i, codigo in enumerate(catalogo):
                        filas_clases.append({
                            "Ambito": ambito,
                            "Nivel": nivel,
                            "Modelo": modelo,
                            "Codigo": int(codigo),
                            "Clase": CLASES_COMUNES[nivel][int(codigo)],
                            "Soporte_referencia": int(soporte[i]),
                            "Pixeles_predichos": int(predichos[i]),
                            "TP": int(tp[i]),
                            "FP": int(fp[i]),
                            "FN": int(fn[i]),
                            "Precision": float(precision[i]),
                            "Recall": float(recall[i]),
                            "F1": float(f1[i]),
                            "IoU": float(iou[i]),
                            "Error_omision": float(1 - recall[i]),
                            "Error_comision": float(1 - precision[i]),
                            "Precision_estimable": bool(predichos[i] > 0),
                            "Recall_estimable": bool(soporte[i] > 0),
                            "F1_IoU_estimables": bool(
                                tp[i] + fp[i] + fn[i] > 0
                            ),
                            "Incluida_macro_referencia": bool(elegibles[i]),
                        })

    METRICAS_GENERALES = pd.DataFrame(filas_generales)
    METRICAS_POR_CLASE = pd.DataFrame(filas_clases)

    if len(METRICAS_GENERALES) != 12:
        raise RuntimeError("Se esperaban 12 conjuntos de métricas.")

    notas = [
        ("NE", "No estimable: denominador nulo. No equivale a cero."),
        ("Macro referencia", politica_33["macro_referencia"]),
        ("Macro catálogo z0", politica_33["macro_catalogo_z0"]),
        ("Clases ausentes", politica_33["clases_ausentes"]),
        ("Sin clasificación", politica_33["sin_clasificacion"]),
        ("Pesos de cálculo", politica_33["pesos_computacionales"]),
        ("Interpretación", politica_33["interpretacion"]),
        ("Escala", politica_33["escala"]),
        ("Fecha Colombia", fecha_33.isoformat()),
    ]

    with pd.ExcelWriter(ruta_excel_33, engine="openpyxl") as escritor:
        pd.DataFrame(
            notas, columns=["Tema", "Regla"]
        ).to_excel(escritor, sheet_name="LEEME", index=False)

        METRICAS_GENERALES.to_excel(
            escritor, sheet_name="Generales",
            index=False, na_rep="NE",
        )
        METRICAS_POR_CLASE.to_excel(
            escritor, sheet_name="Por_clase",
            index=False, na_rep="NE",
        )

        columnas_decimales = {
            "Accuracy", "Kappa", "MCC",
            "Precision", "Recall", "F1", "IoU",
            "Error_omision", "Error_comision",
        }

        for hoja in escritor.book.worksheets:
            hoja.freeze_panes = "D2" if hoja.title != "LEEME" else "A2"
            hoja.auto_filter.ref = hoja.dimensions
            hoja.row_dimensions[1].height = 45

            for columna in hoja.columns:
                encabezado = str(columna[0].value)
                letra = columna[0].column_letter
                hoja.column_dimensions[letra].width = (
                    65 if encabezado in ("Clase", "Regla") else 25
                )

                columna[0].font = Font(bold=True)
                columna[0].alignment = Alignment(
                    wrap_text=True, vertical="center"
                )

                if (
                    encabezado in columnas_decimales
                    or "_macro_" in encabezado
                ):
                    for celda in columna[1:]:
                        celda.number_format = "0.000000"

            if hoja.title == "LEEME":
                for fila in hoja.iter_rows(min_row=2):
                    hoja.row_dimensions[fila[0].row].height = 65
                    for celda in fila:
                        celda.alignment = Alignment(
                            wrap_text=True, vertical="top"
                        )

    # pandas convierte los valores no estimables a null para JSON.
    registro_33.update({
        "completada": True,
        "excel": str(ruta_excel_33),
        "politica_archivo": str(ruta_politica_33),
        "formulas_por_clase_verificadas_con_sklearn": True,
        "metricas_generales": json.loads(
            METRICAS_GENERALES.to_json(
                orient="records", force_ascii=False, double_precision=15
            )
        ),
        "metricas_por_clase": json.loads(
            METRICAS_POR_CLASE.to_json(
                orient="records", force_ascii=False, double_precision=15
            )
        ),
    })

    METRICAS_COMPARACION_VERIFICADAS = True

except Exception as error:
    registro_33["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    with ruta_registro_33.open("x", encoding="utf-8") as archivo:
        json.dump(
            registro_33,
            archivo,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    print("\nRegistro:", ruta_registro_33)

columnas_mostrar = [
    "Ambito", "Nivel", "Modelo", "N",
    "Clases_con_referencia",
    "Accuracy",
    "Precision_macro_referencia",
    "Recall_macro_referencia",
    "F1_macro_referencia",
    "IoU_macro_referencia",
    "Kappa", "MCC",
]

print("\nRESULTADOS — MACRO SOBRE CLASES CON REFERENCIA")
print(METRICAS_GENERALES[columnas_mostrar].to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}",
))

print("\nExcel:", ruta_excel_33)
print("Política de cálculo:", ruta_politica_33)
print("\nCELDA 33 TERMINADA. Métricas calculadas y documentadas.")


Registro: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Evaluacion_comun_20261008_081815_052051\Metricas_20261008_082759_926505\celda_33_metricas.json

RESULTADOS — MACRO SOBRE CLASES CON REFERENCIA
                 Ambito   Nivel      Modelo        N  Clases_con_referencia  Accuracy  Precision_macro_referencia  Recall_macro_referencia  F1_macro_referencia  IoU_macro_referencia    Kappa      MCC
     concordancia_comun nivel_1        UNET 12319921                      5  0.868946                    0.648823                 0.767477             0.676457              0.546297 0.669653 0.685097
     concordancia_comun nivel_1    RESNET50 12319921                      5  0.897601                    0.660030                 0.806824             0.701147              0.577334 0.753052 0.756799
     concordancia_comun nivel_1 DENSENET121 12319921                      5  0.901519                    0.793124                 0.780992             0.784804              

In [41]:
# ============================================================
# CELDA 34 — CORRESPONDENCIA JERÁRQUICA ENTRE NIVELES
# Conserva las predicciones originales.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
from contextlib import ExitStack
import json

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window


if not globals().get("METRICAS_COMPARACION_VERIFICADAS", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 33.")

fecha_34 = datetime.now(timezone(timedelta(hours=-5)))
carpeta_34 = Path(CARPETA_EVALUACION_COMUN) / (
    "Jerarquia_" + fecha_34.strftime("%Y%m%d_%H%M%S_%f")
)
carpeta_34.mkdir(exist_ok=False)

ruta_excel_34 = carpeta_34 / "Correspondencia_entre_niveles.xlsx"
ruta_npz_34 = carpeta_34 / "Matrices_jerarquicas.npz"
ruta_registro_34 = carpeta_34 / "celda_34_jerarquia.json"

# Códigos del DATASET; no confundir con los Value originales del IDEAM.
PADRES_JERARQUIA = {
    0: 0,   # Sin clasificación se conserva.
    1: 1,   # Áreas agrícolas heterogéneas -> Territorios agrícolas
    2: 2,   # Vegetación herbácea/arbustiva -> Bosques y áreas seminaturales
    3: 2,   # Bosques -> Bosques y áreas seminaturales
    4: 1,   # Pastos -> Territorios agrícolas
    5: 3,   # Aguas continentales -> Superficies de agua
    6: 4,   # Zonas urbanizadas -> Territorios artificializados
    7: 4,   # Extracción minera -> Territorios artificializados
    8: 5,   # Áreas húmedas continentales -> Áreas húmedas
    9: 4,   # Zonas industriales/redes -> Territorios artificializados
    10: 4,  # Zonas verdes artificializadas -> Territorios artificializados
    11: 1,  # Cultivos transitorios -> Territorios agrícolas
    12: 1,  # Cultivos permanentes -> Territorios agrícolas
    13: 2,  # Áreas abiertas -> Bosques y áreas seminaturales
}
padres = np.asarray(
    [PADRES_JERARQUIA[c] for c in range(14)],
    dtype=np.int64,
)

registro_34 = {
    "celda": 34,
    "fecha_colombia": fecha_34.isoformat(),
    "origen_matrices": str(ruta_npz_32),
    "correspondencia": PADRES_JERARQUIA,
    "decisiones": [
        "Agrupar clases según su categoría padre documentada.",
        "No modificar las predicciones directas de ninguno de los niveles.",
        "Separar concordancia con referencia de coherencia entre predicciones.",
        "Utilizar las mismas máscaras de evaluación ya verificadas.",
        "No interpretar coincidencia en Sin clasificación como acuerdo temático.",
    ],
    "completada": False,
}

MATRICES_NIVEL2_AGREGADO = {}
CRUCES_PREDICCIONES_NIVELES = {}
archivos_npz = {}
resumen_34 = []
JERARQUIA_PREDICCIONES_VERIFICADA = False


def verificar_geometria_34(src, base):
    if (
        src.width != base.width
        or src.height != base.height
        or src.transform != base.transform
        or not crs_equivalentes_xy(src.crs, base.crs)
    ):
        raise RuntimeError(f"Cuadrícula incompatible: {src.name}")


try:
    with np.load(ruta_npz_32, allow_pickle=False) as matrices:
        for modelo in ("UNET", "RESNET50", "DENSENET121"):
            with ExitStack() as pila:
                n1 = pila.enter_context(rasterio.open(
                    PREDICCIONES_COMPARACION[modelo]["nivel_1"]
                ))
                n2 = pila.enter_context(rasterio.open(
                    PREDICCIONES_COMPARACION[modelo]["nivel_2"]
                ))
                mascaras = {
                    ambito: pila.enter_context(rasterio.open(ruta))
                    for ambito, ruta in RUTAS_MASCARAS_EVALUACION.items()
                }

                verificar_geometria_34(n2, n1)
                for src in mascaras.values():
                    verificar_geometria_34(src, n1)

                cruces = {
                    ambito: np.zeros((6, 6), dtype=np.int64)
                    for ambito in mascaras
                }

                for fila in range(0, n1.height, 256):
                    h = min(256, n1.height - fila)
                    ventana = Window(0, fila, n1.width, h)

                    p1 = n1.read(1, window=ventana)
                    p2 = n2.read(1, window=ventana)
                    valida1 = n1.read_masks(1, window=ventana) > 0
                    valida2 = n2.read_masks(1, window=ventana) > 0

                    for ambito, src_mascara in mascaras.items():
                        valores_mascara = src_mascara.read(1, window=ventana)
                        if not np.isin(valores_mascara, [0, 1]).all():
                            raise RuntimeError("Máscara no binaria.")

                        seleccion = valores_mascara == 1
                        if not seleccion.any():
                            continue

                        if not (
                            valida1[seleccion].all()
                            and valida2[seleccion].all()
                        ):
                            raise RuntimeError("NoData dentro de la evaluación.")

                        a = p1[seleccion].astype(np.int64)
                        b = p2[seleccion].astype(np.int64)

                        minimo = 0 if modelo == "UNET" else 1
                        if (
                            np.any((a < minimo) | (a > 5))
                            or np.any((b < minimo) | (b > 13))
                        ):
                            raise RuntimeError("Códigos de predicción inesperados.")

                        b = padres[b]
                        cruces[ambito] += np.bincount(
                            a * 6 + b, minlength=36
                        ).reshape(6, 6)

                for ambito in mascaras:
                    matriz_n1 = matrices[f"{ambito}__nivel_1__{modelo}"]
                    matriz_n2 = matrices[f"{ambito}__nivel_2__{modelo}"]

                    # Agrupar AMBOS ejes de la matriz de nivel 2:
                    # referencia 1..13 -> 1..5
                    # predicción 0..13 -> 0..5
                    agregada = np.zeros((5, 6), dtype=np.int64)
                    filas, columnas = np.indices(matriz_n2.shape)

                    np.add.at(
                        agregada,
                        (padres[filas + 1] - 1, padres[columnas]),
                        matriz_n2,
                    )

                    cruce = cruces[ambito]
                    total = int(matriz_n1.sum())

                    if (
                        int(agregada.sum()) != total
                        or int(cruce.sum()) != total
                    ):
                        raise RuntimeError("Los totales jerárquicos no coinciden.")

                    if not np.array_equal(
                        agregada.sum(axis=1), matriz_n1.sum(axis=1)
                    ):
                        raise RuntimeError(
                            "La referencia agregada difiere del nivel 1."
                        )

                    # Verificar los conteos predichos contra las matrices.
                    if not np.array_equal(
                        cruce.sum(axis=1), matriz_n1.sum(axis=0)
                    ):
                        raise RuntimeError("Marginal del nivel 1 inconsistente.")

                    if not np.array_equal(
                        cruce.sum(axis=0), agregada.sum(axis=0)
                    ):
                        raise RuntimeError("Marginal del nivel 2 inconsistente.")

                    aciertos_n1 = int(
                        matriz_n1[np.arange(5), np.arange(1, 6)].sum()
                    )
                    aciertos_agregado = int(
                        agregada[np.arange(5), np.arange(1, 6)].sum()
                    )

                    acuerdos_tematicos = int(np.trace(cruce[1:, 1:]))
                    ambos_sin_clase = int(cruce[0, 0])
                    desacuerdos = total - int(np.trace(cruce))

                    MATRICES_NIVEL2_AGREGADO.setdefault(
                        ambito, {}
                    )[modelo] = agregada
                    CRUCES_PREDICCIONES_NIVELES.setdefault(
                        ambito, {}
                    )[modelo] = cruce

                    archivos_npz[f"{ambito}__{modelo}__agregada"] = agregada
                    archivos_npz[f"{ambito}__{modelo}__cruce"] = cruce

                    resumen_34.append({
                        "Ambito": ambito,
                        "Modelo": modelo,
                        "N": total,
                        "Accuracy_nivel1_directo": aciertos_n1 / total,
                        "Accuracy_nivel2_agregado": aciertos_agregado / total,
                        "Acuerdo_tematico_entre_niveles": (
                            acuerdos_tematicos / total
                        ),
                        "Pixeles_desacuerdo_entre_niveles": desacuerdos,
                        "Pixeles_ambos_sin_clasificacion": ambos_sin_clase,
                        "Totales_y_marginales_verificados": True,
                    })

    np.savez_compressed(ruta_npz_34, **archivos_npz)
    RESUMEN_JERARQUIA = pd.DataFrame(resumen_34)

    nombres = [
        f"{c} — {CLASES_COMUNES['nivel_1'][c]}"
        for c in range(1, 6)
    ]
    nombres_con_cero = nombres + ["0 — Sin clasificación"]
    orden = [1, 2, 3, 4, 5, 0]

    notas = [
        ("AGREGADA", "Referencia nivel 1 frente a predicción nivel 2 agrupada."),
        ("CRUCE", "Predicción nivel 1 frente a predicción nivel 2 agrupada."),
        (
            "Interpretación",
            "El acuerdo entre predicciones mide coherencia jerárquica, "
            "no exactitud frente a una referencia.",
        ),
        (
            "Mapas",
            "Se conservan los mapas directos. No se fuerzan coincidencias.",
        ),
        (
            "Estructura NPZ",
            "Agregadas: filas 1..5, columnas 0..5. "
            "Cruces: filas y columnas 0..5.",
        ),
    ]

    with pd.ExcelWriter(ruta_excel_34, engine="openpyxl") as escritor:
        pd.DataFrame(
            notas, columns=["Tema", "Descripción"]
        ).to_excel(escritor, sheet_name="LEEME", index=False)

        RESUMEN_JERARQUIA.to_excel(
            escritor, sheet_name="Resumen", index=False
        )

        for ambito in MATRICES_NIVEL2_AGREGADO:
            sufijo = "COMUN" if ambito == "concordancia_comun" else "CAND"

            for modelo in MATRICES_NIVEL2_AGREGADO[ambito]:
                agregada = MATRICES_NIVEL2_AGREGADO[ambito][modelo]
                cruce = CRUCES_PREDICCIONES_NIVELES[ambito][modelo]

                pd.DataFrame(
                    agregada[:, orden],
                    index=nombres,
                    columns=nombres_con_cero,
                ).rename_axis("Referencia N1 / Predicción N2 agrupada").to_excel(
                    escritor, sheet_name=f"{modelo}_{sufijo}_AGREGADA"
                )

                pd.DataFrame(
                    cruce[np.ix_(orden, orden)],
                    index=nombres_con_cero,
                    columns=nombres_con_cero,
                ).rename_axis("Predicción N1 / Predicción N2 agrupada").to_excel(
                    escritor, sheet_name=f"{modelo}_{sufijo}_CRUCE"
                )

        for hoja in escritor.book.worksheets:
            hoja.freeze_panes = "B2"
            hoja.column_dimensions["A"].width = 55
            for columna in range(2, hoja.max_column + 1):
                letra = hoja.cell(1, columna).column_letter
                hoja.column_dimensions[letra].width = 28

    registro_34.update({
        "completada": True,
        "excel": str(ruta_excel_34),
        "matrices_npz": str(ruta_npz_34),
        "resumen": json.loads(
            RESUMEN_JERARQUIA.to_json(
                orient="records", force_ascii=False, double_precision=15
            )
        ),
    })
    JERARQUIA_PREDICCIONES_VERIFICADA = True

except Exception as error:
    registro_34["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    with ruta_registro_34.open("x", encoding="utf-8") as archivo:
        json.dump(
            registro_34, archivo,
            ensure_ascii=False, indent=2, allow_nan=False,
        )
    print("\nRegistro:", ruta_registro_34)

print("\nCOMPARACIÓN JERÁRQUICA")
print(RESUMEN_JERARQUIA.to_string(
    index=False,
    float_format=lambda x: f"{x:.6f}",
))
print("\nExcel:", ruta_excel_34)
print("\nCELDA 34 TERMINADA. Predicciones originales conservadas.")


Registro: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Evaluacion_comun_20261008_081815_052051\Jerarquia_20261008_083107_911018\celda_34_jerarquia.json

COMPARACIÓN JERÁRQUICA
                 Ambito      Modelo        N  Accuracy_nivel1_directo  Accuracy_nivel2_agregado  Acuerdo_tematico_entre_niveles  Pixeles_desacuerdo_entre_niveles  Pixeles_ambos_sin_clasificacion  Totales_y_marginales_verificados
     concordancia_comun        UNET 12319921                 0.868946                  0.853782                        0.914224                           1056751                                0                              True
candidata_fuera_huellas        UNET   332568                 0.778956                  0.784588                        0.875382                             41444                                0                              True
     concordancia_comun    RESNET50 12319921                 0.897601                  0.898139             

In [42]:
# ============================================================
# CELDA 35 — DESGLOSE DE ERRORES Y COMPARACIÓN POR CLASE
# No modifica mapas, matrices ni métricas anteriores.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
import json

import numpy as np
import pandas as pd


if not globals().get("JERARQUIA_PREDICCIONES_VERIFICADA", False):
    raise RuntimeError("Primero debe finalizar correctamente la celda 34.")

if not globals().get("METRICAS_COMPARACION_VERIFICADAS", False):
    raise RuntimeError("Faltan las métricas de la celda 33.")

fecha_35 = datetime.now(timezone(timedelta(hours=-5)))
carpeta_35 = Path(CARPETA_EVALUACION_COMUN) / (
    "Diagnostico_errores_" + fecha_35.strftime("%Y%m%d_%H%M%S_%f")
)
carpeta_35.mkdir(exist_ok=False)

ruta_excel_35 = carpeta_35 / "Errores_y_comparacion_por_clase.xlsx"
ruta_registro_35 = carpeta_35 / "celda_35_diagnostico.json"

registro_35 = {
    "celda": 35,
    "fecha_colombia": fecha_35.isoformat(),
    "matrices_origen": str(ruta_npz_32),
    "matrices_jerarquicas": str(ruta_npz_34),
    "reglas": [
        "Describir diferencias frente a la referencia disponible.",
        "Conservar todas las clases, incluso las ausentes en referencia.",
        "Separar errores dentro de una categoría padre y entre categorías.",
        "No interpretar el diagnóstico como prueba causal de los errores.",
        "No modificar predicciones ni seleccionar parámetros con estos resultados.",
    ],
    "completada": False,
}

padres_35 = np.asarray(
    [PADRES_JERARQUIA[c] for c in range(14)],
    dtype=np.int64,
)

desglose = []
confusiones = []
tablas_f1 = {}


def porcentaje_35(cantidad, total):
    return 100.0 * cantidad / total if total else np.nan


try:
    with np.load(ruta_npz_32, allow_pickle=False) as matrices, \
         np.load(ruta_npz_34, allow_pickle=False) as jerarquia:

        for ambito in (
            "concordancia_comun",
            "candidata_fuera_huellas",
        ):
            for modelo in ("UNET", "RESNET50", "DENSENET121"):
                m = matrices[f"{ambito}__nivel_2__{modelo}"]
                agregada = jerarquia[f"{ambito}__{modelo}__agregada"]

                if m.shape != (13, 14):
                    raise RuntimeError("Matriz de nivel 2 con forma inesperada.")

                filas, columnas = np.indices(m.shape)
                referencias = filas + 1

                es_acierto = referencias == columnas
                es_sin_clase = columnas == 0
                es_error_tematico = (
                    (columnas > 0) & (referencias != columnas)
                )
                mismo_padre = (
                    padres_35[referencias] == padres_35[columnas]
                )

                n = int(m.sum())
                aciertos = int(m[es_acierto].sum())
                errores_internos = int(
                    m[es_error_tematico & mismo_padre].sum()
                )
                errores_entre_padres = int(
                    m[es_error_tematico & ~mismo_padre].sum()
                )
                sin_clase = int(m[es_sin_clase].sum())
                errores_totales = n - aciertos

                if (
                    aciertos + errores_internos
                    + errores_entre_padres + sin_clase
                ) != n:
                    raise RuntimeError("El desglose no conserva el total.")

                aciertos_agregados = int(
                    agregada[np.arange(5), np.arange(1, 6)].sum()
                )
                if aciertos_agregados != aciertos + errores_internos:
                    raise RuntimeError(
                        "El desglose no coincide con la matriz jerárquica."
                    )

                desglose.append({
                    "Ambito": ambito,
                    "Modelo": modelo,
                    "N": n,
                    "Aciertos_nivel2": aciertos,
                    "Errores_totales_nivel2": errores_totales,
                    "Errores_dentro_mismo_padre": errores_internos,
                    "Errores_entre_padres": errores_entre_padres,
                    "Sin_clasificacion": sin_clase,
                    "Pct_errores_dentro_mismo_padre": porcentaje_35(
                        errores_internos, errores_totales
                    ),
                    "Pct_errores_entre_padres": porcentaje_35(
                        errores_entre_padres, errores_totales
                    ),
                    "Pct_errores_sin_clasificacion": porcentaje_35(
                        sin_clase, errores_totales
                    ),
                    "Accuracy_nivel2": aciertos / n,
                    "Accuracy_nivel2_agregado": aciertos_agregados / n,
                    "Identidad_jerarquica_verificada": True,
                })

                # Registrar todas las confusiones no nulas, no solo las mayores.
                soporte = m.sum(axis=1)

                for i, j in zip(*np.nonzero(m)):
                    codigo_ref = int(i + 1)
                    codigo_pred = int(j)

                    if codigo_ref == codigo_pred:
                        continue

                    cantidad = int(m[i, j])
                    if codigo_pred == 0:
                        tipo = "Sin clasificación"
                        nombre_pred = "Sin clasificación"
                    else:
                        tipo = (
                            "Dentro del mismo padre"
                            if padres_35[codigo_ref] == padres_35[codigo_pred]
                            else "Entre categorías padre"
                        )
                        nombre_pred = CLASES_COMUNES["nivel_2"][codigo_pred]

                    confusiones.append({
                        "Ambito": ambito,
                        "Modelo": modelo,
                        "Codigo_referencia": codigo_ref,
                        "Clase_referencia": (
                            CLASES_COMUNES["nivel_2"][codigo_ref]
                        ),
                        "Codigo_prediccion": codigo_pred,
                        "Clase_predicha": nombre_pred,
                        "Tipo_error": tipo,
                        "Pixeles": cantidad,
                        "Pct_del_soporte_de_esa_clase": porcentaje_35(
                            cantidad, int(soporte[i])
                        ),
                        "Pct_de_todos_los_errores": porcentaje_35(
                            cantidad, errores_totales
                        ),
                    })

    DESGLOSE_ERRORES_NIVEL2 = pd.DataFrame(desglose)

    columnas_confusiones = [
        "Ambito", "Modelo",
        "Codigo_referencia", "Clase_referencia",
        "Codigo_prediccion", "Clase_predicha",
        "Tipo_error", "Pixeles",
        "Pct_del_soporte_de_esa_clase",
        "Pct_de_todos_los_errores",
    ]
    CONFUSIONES_NIVEL2 = pd.DataFrame(
        confusiones, columns=columnas_confusiones
    ).sort_values(
        ["Ambito", "Modelo", "Pixeles"],
        ascending=[True, True, False],
    )

    # Tabla completa de F1 por clase y modelo, acompañada de soporte.
    for ambito in (
        "concordancia_comun",
        "candidata_fuera_huellas",
    ):
        for nivel in ("nivel_1", "nivel_2"):
            detalle = METRICAS_POR_CLASE.loc[
                (METRICAS_POR_CLASE["Ambito"] == ambito)
                & (METRICAS_POR_CLASE["Nivel"] == nivel)
            ].copy()

            variacion_soporte = detalle.groupby("Codigo")[
                "Soporte_referencia"
            ].nunique()

            if (variacion_soporte != 1).any():
                raise RuntimeError("El soporte difiere entre modelos.")

            comparacion = detalle.pivot(
                index=["Codigo", "Clase"],
                columns="Modelo",
                values="F1",
            ).reindex(columns=["UNET", "RESNET50", "DENSENET121"])

            soporte = detalle.groupby(["Codigo", "Clase"])[
                "Soporte_referencia"
            ].first()

            comparacion.insert(0, "Soporte_referencia", soporte)
            comparacion.insert(
                1,
                "Presente_en_referencia",
                soporte > 0,
            )
            comparacion = comparacion.reset_index()

            sufijo = "COMUN" if ambito == "concordancia_comun" else "CAND"
            numero_nivel = 1 if nivel == "nivel_1" else 2
            tablas_f1[f"F1_N{numero_nivel}_{sufijo}"] = comparacion

    notas = [
        (
            "Desglose",
            "Los porcentajes del desglose usan como denominador "
            "todos los errores del nivel 2, no todos los píxeles.",
        ),
        (
            "Errores dentro del mismo padre",
            "La subclase no coincide con la referencia, "
            "pero ambas pertenecen a la misma categoría de nivel 1.",
        ),
        (
            "Errores entre padres",
            "La predicción difiere de la referencia también "
            "en la categoría de nivel 1.",
        ),
        (
            "Clases sin referencia",
            "Consultar el soporte. Un F1 de 0 puede deberse a "
            "predicciones falsas positivas sin ejemplos de referencia. "
            "NE indica denominador nulo.",
        ),
        (
            "Interpretación",
            "Diferencias frente a la referencia disponible; "
            "no identifica por sí solo su causa ni resuelve "
            "las limitaciones temporales o de independencia espacial.",
        ),
    ]

    with pd.ExcelWriter(ruta_excel_35, engine="openpyxl") as escritor:
        pd.DataFrame(
            notas, columns=["Tema", "Descripción"]
        ).to_excel(escritor, sheet_name="LEEME", index=False)

        DESGLOSE_ERRORES_NIVEL2.to_excel(
            escritor, sheet_name="Desglose_N2",
            index=False, na_rep="NE",
        )
        CONFUSIONES_NIVEL2.to_excel(
            escritor, sheet_name="Confusiones_N2",
            index=False, na_rep="NE",
        )
        METRICAS_POR_CLASE.to_excel(
            escritor, sheet_name="Detalle_por_clase",
            index=False, na_rep="NE",
        )

        for nombre, tabla in tablas_f1.items():
            tabla.to_excel(
                escritor, sheet_name=nombre,
                index=False, na_rep="NE",
            )

        for hoja in escritor.book.worksheets:
            hoja.freeze_panes = "C2"
            hoja.auto_filter.ref = hoja.dimensions
            for columna in hoja.columns:
                encabezado = str(columna[0].value)
                letra = columna[0].column_letter
                hoja.column_dimensions[letra].width = (
                    60 if encabezado in (
                        "Clase", "Clase_referencia",
                        "Clase_predicha", "Descripción",
                    ) else 26
                )

    registro_35.update({
        "completada": True,
        "excel": str(ruta_excel_35),
        "desglose": json.loads(
            DESGLOSE_ERRORES_NIVEL2.to_json(
                orient="records", force_ascii=False, double_precision=15
            )
        ),
        "confusiones": json.loads(
            CONFUSIONES_NIVEL2.to_json(
                orient="records", force_ascii=False, double_precision=15
            )
        ),
    })

except Exception as error:
    registro_35["error"] = f"{type(error).__name__}: {error}"
    raise

finally:
    with ruta_registro_35.open("x", encoding="utf-8") as archivo:
        json.dump(
            registro_35, archivo,
            ensure_ascii=False, indent=2, allow_nan=False,
        )
    print("\nRegistro:", ruta_registro_35)

print("\nDESGLOSE DE LOS ERRORES DEL NIVEL 2")
print(DESGLOSE_ERRORES_NIVEL2[[
    "Ambito", "Modelo",
    "Errores_totales_nivel2",
    "Pct_errores_dentro_mismo_padre",
    "Pct_errores_entre_padres",
    "Pct_errores_sin_clasificacion",
]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

for nombre, tabla in tablas_f1.items():
    print("\n" + "=" * 75)
    print(nombre)
    print(tabla.to_string(
        index=False,
        na_rep="NE",
        float_format=lambda x: f"{x:.4f}",
    ))

print("\nExcel:", ruta_excel_35)
print("\nCELDA 35 TERMINADA. Diagnóstico generado sin alterar resultados.")


Registro: D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990\Evaluacion_comun_20261008_081815_052051\Diagnostico_errores_20261008_083400_820126\celda_35_diagnostico.json

DESGLOSE DE LOS ERRORES DEL NIVEL 2
                 Ambito      Modelo  Errores_totales_nivel2  Pct_errores_dentro_mismo_padre  Pct_errores_entre_padres  Pct_errores_sin_clasificacion
     concordancia_comun        UNET                 4654323                         61.2964                   38.7036                         0.0000
     concordancia_comun    RESNET50                 3787842                         66.8698                   33.1302                         0.0000
     concordancia_comun DENSENET121                 3819536                         64.7968                   35.2032                         0.0000
candidata_fuera_huellas        UNET                  140654                         49.0672                   50.9328                         0.0000
candidata_fuera_huellas  

In [43]:
# CELDA 36 — BÚSQUEDA DE CÓDIGO, CONFIGURACIONES Y PESOS
# Ejecutar sin cambios en cada computador.
# Solo lectura de insumos. Crea una carpeta nueva para los informes.

from pathlib import Path
from datetime import datetime
from collections import Counter
import os
import re
import csv
import json
import socket
import hashlib
import ctypes
import time
import tokenize
import io

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

# Puedes agregar carpetas de red o ubicaciones adicionales.
RUTAS_ADICIONALES = []

# Se calcula el hash de TODOS los pesos encontrados.
CALCULAR_HASH_PESOS = True

# Los archivos de texto mayores a este tamaño se inventarían,
# pero no se leen completos. La omisión queda registrada.
MAX_TEXTO_MB = 100

EQUIPO = socket.gethostname()
FECHA = datetime.now().astimezone().isoformat()
SELLO = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
EQUIPO_ARCHIVO = re.sub(r"[^A-Za-z0-9_-]", "_", EQUIPO)

SALIDA_36 = (
    Path.home()
    / "Auditoria_busqueda_modelos"
    / f"{EQUIPO_ARCHIVO}_{SELLO}"
)
SALIDA_36.mkdir(parents=True, exist_ok=False)

# Se omiten directorios técnicos por nombre exacto.
# NO se excluyen anaconda3, envs, .ipynb_checkpoints,
# Documents, Downloads, Desktop, OneDrive ni carpetas del proyecto.
EXCLUIR = {
    "windows", "program files", "program files (x86)",
    "programdata", "$recycle.bin", "system volume information",
    "$windows.~bt", "$windows.~ws",
    "site-packages", "node_modules", ".git",
    "__pycache__", "pkgs", ".cache",
    "auditoria_busqueda_modelos",
}

EXT_TEXTO = {
    ".ipynb", ".py", ".pyw", ".json", ".yaml", ".yml",
    ".toml", ".ini", ".cfg", ".txt", ".md", ".log",
    ".csv", ".html", ".htm",
}
EXT_PESOS = {".pt", ".pth", ".ckpt", ".h5", ".hdf5", ".keras", ".onnx", ".safetensors"}
EXT_COMPRIMIDOS = {".zip", ".7z", ".rar", ".tar", ".gz"}

PATRONES = {
    "unet": r"\bu[_\-\s]?net\b|unet",
    "otras_redes": r"resnet|densenet",
    "entrenamiento": (
        r"loss\.backward|optimizer\.step|model\.train\("
        r"|train_loader|training_step|num_epochs"
    ),
    "dataset": r"Dataset_CLC|class\s+\w+\s*\([^)]*Dataset|__getitem__",
    "normalizacion": r"normaliz|means\b|stds\b|std\b|mean\b",
    "escalado": r"10000|10_000|reflectance|reflectancia|scale_factor|add_offset",
    "clip": r"np\.clip|numpy\.clip|torch\.clamp|\.clamp\(",
    "nodata": r"ignore_index|nodata|read_masks|dataset_mask",
    "etiquetas": r"idx_to_code|class_mapping|label_mapping|num_classes|n_classes",
    "bandas": r"\bB2\b|\bB3\b|\bB4\b|\bB8\b|in_channels|n_channels",
    "particiones": r"train_samples|val_samples|test_samples|train_test_split",
    "pesos": r"torch\.save|torch\.load|state_dict|best_model|checkpoint",
}
REGEX = {k: re.compile(v, re.IGNORECASE) for k, v in PATRONES.items()}

# ============================================================
# 2. FUNCIONES
# ============================================================

def discos_disponibles():
    if os.name != "nt":
        return [Path.home()]

    raices = []
    mascara = ctypes.windll.kernel32.GetLogicalDrives()
    for i in range(26):
        if mascara & (1 << i):
            raiz = f"{chr(65 + i)}:\\"
            tipo = ctypes.windll.kernel32.GetDriveTypeW(raiz)
            # 2 = extraíble; 3 = disco local.
            if tipo in (2, 3):
                raices.append(Path(raiz))
    return raices


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def leer_texto(ruta):
    contenido = ruta.read_bytes()
    if ruta.suffix.lower() in {".py", ".pyw"}:
        try:
            encoding, _ = tokenize.detect_encoding(
                io.BytesIO(contenido).readline
            )
            return contenido.decode(encoding), encoding
        except (SyntaxError, UnicodeError):
            pass
    try:
        return contenido.decode("utf-8-sig"), "utf-8-sig"
    except UnicodeDecodeError:
        return contenido.decode("latin-1"), "latin-1"


def es_enlace_o_junction(ruta):
    if ruta.is_symlink():
        return True
    # Compatible con Python 3.11 en Windows:
    # evita recorrer junctions y enlaces de directorio.
    atributos = getattr(ruta.lstat(), "st_file_attributes", 0)
    return bool(atributos & 0x400)


def fragmentos_y_grupos(bloques):
    grupos = set()
    ejemplos = []
    por_grupo = Counter()

    for ubicacion, texto in bloques:
        for grupo, patron in REGEX.items():
            if patron.search(texto):
                grupos.add(grupo)

        # Extrae evidencia acotada; no vuelca todo el archivo.
        for numero, linea in enumerate(texto.splitlines(), start=1):
            encontrados = [g for g, p in REGEX.items() if p.search(linea)]
            if not encontrados:
                continue
            if any(por_grupo[g] < 3 for g in encontrados):
                ejemplos.append({
                    "ubicacion": ubicacion,
                    "linea": numero,
                    "grupos": encontrados,
                    "texto": linea[:700],
                })
                por_grupo.update(encontrados)
            if len(ejemplos) >= 35:
                break

    return sorted(grupos), ejemplos


def analizar_fuente(ruta):
    texto, encoding = leer_texto(ruta)

    if ruta.suffix.lower() == ".ipynb":
        cuaderno = json.loads(texto)
        # Se revisa source, NO outputs: evita confundir una salida
        # impresa con una instrucción realmente presente en el código.
        bloques = []
        for i, celda in enumerate(cuaderno.get("cells", []), start=1):
            source = celda.get("source", "")
            if isinstance(source, list):
                source = "".join(source)
            if isinstance(source, str):
                bloques.append((
                    f"celda_{i}_{celda.get('cell_type', 'desconocido')}",
                    source,
                ))
    else:
        bloques = [("archivo", texto)]

    grupos, ejemplos = fragmentos_y_grupos(bloques)
    pista_ruta = bool(re.search(
        r"unet|u[_ -]net|entren|training|predic|modelo|checkpoint",
        str(ruta), re.IGNORECASE
    ))

    # Priorización para revisión humana, NO certificación de procedencia.
    puntaje = (
        8 * ("unet" in grupos)
        + 5 * ("entrenamiento" in grupos)
        + 3 * ("dataset" in grupos)
        + 2 * ("normalizacion" in grupos)
        + 2 * ("etiquetas" in grupos)
        + int(pista_ruta)
    )
    candidato = (
        "unet" in grupos
        or (
            "entrenamiento" in grupos
            and any(g in grupos for g in ("pesos", "dataset", "otras_redes"))
        )
        or (
            pista_ruta
            and any(g in grupos for g in (
                "normalizacion", "escalado", "etiquetas", "pesos", "bandas"
            ))
        )
    )
    return grupos, ejemplos, puntaje, candidato, encoding


# ============================================================
# 3. PREPARAR REGISTROS
# ============================================================

raices_36 = discos_disponibles() + [Path(p) for p in RUTAS_ADICIONALES]
raices_36 = list(dict.fromkeys(str(p.absolute()) for p in raices_36))

conteos_36 = Counter()
candidatos_36 = []
pesos_36 = []
inicio_36 = time.monotonic()
ultimo_aviso_36 = inicio_36

ruta_inventario = SALIDA_36 / "Inventario_archivos.csv"
ruta_incidencias = SALIDA_36 / "Incidencias_y_omisiones.jsonl"
ruta_evidencias = SALIDA_36 / "Evidencias_candidatos.jsonl"

print("Equipo:", EQUIPO)
print("Raíces:", raices_36)
print("Informes:", SALIDA_36)
print("\nBuscando. Los pesos grandes pueden tardar en procesarse.", flush=True)

with (
    ruta_inventario.open("x", newline="", encoding="utf-8-sig") as f_inv,
    ruta_incidencias.open("x", encoding="utf-8") as f_inc,
    ruta_evidencias.open("x", encoding="utf-8") as f_evi,
):
    columnas = [
        "ruta", "tipo", "bytes", "modificacion",
        "sha256", "estado", "grupos", "puntaje",
    ]
    escritor = csv.DictWriter(f_inv, fieldnames=columnas)
    escritor.writeheader()

    def incidencia(ruta, motivo, detalle=""):
        conteos_36[f"incidencia_{motivo}"] += 1
        f_inc.write(json.dumps({
            "ruta": str(ruta),
            "motivo": motivo,
            "detalle": str(detalle),
        }, ensure_ascii=False) + "\n")
        f_inc.flush()

    def error_recorrido(error):
        incidencia(getattr(error, "filename", ""), "acceso", error)

    for raiz in raices_36:
        if not Path(raiz).is_dir():
            incidencia(raiz, "raiz_no_disponible")
            continue

        print("\nRevisando:", raiz, flush=True)

        for carpeta, subcarpetas, archivos in os.walk(
            raiz, topdown=True, followlinks=False, onerror=error_recorrido
        ):
            permitidas = []
            for nombre in subcarpetas:
                ruta_dir = Path(carpeta) / nombre
                try:
                    if nombre.lower() in EXCLUIR:
                        incidencia(ruta_dir, "carpeta_excluida_por_politica")
                    elif es_enlace_o_junction(ruta_dir):
                        incidencia(ruta_dir, "directorio_enlace_no_recorrido")
                    else:
                        permitidas.append(nombre)
                except OSError as exc:
                    incidencia(ruta_dir, "inspeccion_directorio", exc)
            subcarpetas[:] = permitidas
            conteos_36["directorios_revisados"] += 1

            for nombre in archivos:
                conteos_36["archivos_vistos"] += 1
                ruta = Path(carpeta) / nombre
                ext = ruta.suffix.lower()

                if ext not in EXT_TEXTO | EXT_PESOS | EXT_COMPRIMIDOS:
                    continue

                fila = dict.fromkeys(columnas, "")
                fila["ruta"] = str(ruta)
                try:
                    stat_inicial = ruta.stat()
                    fila["bytes"] = stat_inicial.st_size
                    fila["modificacion"] = datetime.fromtimestamp(
                        stat_inicial.st_mtime
                    ).astimezone().isoformat()

                    if ruta.is_symlink():
                        fila["estado"] = "enlace_no_leido"
                        incidencia(ruta, fila["estado"])

                    elif ext in EXT_PESOS:
                        fila["tipo"] = "pesos"
                        print(
                            f"  Pesos: {ruta} "
                            f"({stat_inicial.st_size / 1024**2:.1f} MB)",
                            flush=True,
                        )
                        if CALCULAR_HASH_PESOS:
                            fila["sha256"] = sha256_archivo(ruta)
                        fila["estado"] = "inventariado_sin_cargar"
                        pesos_36.append(fila)

                    elif ext in EXT_COMPRIMIDOS:
                        fila["tipo"] = "comprimido"
                        fila["estado"] = "pendiente_revision_interna"
                        conteos_36["comprimidos"] += 1

                    else:
                        fila["tipo"] = "fuente_o_metadato"
                        if stat_inicial.st_size > MAX_TEXTO_MB * 1024**2:
                            fila["estado"] = "omitido_por_tamano"
                            incidencia(ruta, fila["estado"], stat_inicial.st_size)
                        else:
                            grupos, ejemplos, puntaje, candidato, encoding = (
                                analizar_fuente(ruta)
                            )
                            conteos_36["textos_leidos"] += 1
                            fila.update({
                                "estado": "candidato" if candidato else "leido",
                                "grupos": ";".join(grupos),
                                "puntaje": puntaje,
                            })
                            if candidato:
                                fila["sha256"] = sha256_archivo(ruta)
                                candidatos_36.append(fila)
                                f_evi.write(json.dumps({
                                    **fila,
                                    "encoding": encoding,
                                    "fragmentos": ejemplos,
                                }, ensure_ascii=False) + "\n")
                                f_evi.flush()

                    stat_final = ruta.stat()
                    if (
                        stat_inicial.st_size != stat_final.st_size
                        or stat_inicial.st_mtime_ns != stat_final.st_mtime_ns
                    ):
                        fila["estado"] += "|cambio_durante_lectura"
                        incidencia(ruta, "cambio_durante_lectura")

                except Exception as exc:
                    fila["estado"] = "error_lectura"
                    incidencia(ruta, "lectura", f"{type(exc).__name__}: {exc}")

                escritor.writerow(fila)

            ahora = time.monotonic()
            if ahora - ultimo_aviso_36 >= 15:
                print(
                    f"  Archivos vistos: {conteos_36['archivos_vistos']:,} | "
                    f"Textos leídos: {conteos_36['textos_leidos']:,} | "
                    f"Candidatos: {len(candidatos_36)} | "
                    f"Pesos: {len(pesos_36)} | "
                    f"Minutos: {(ahora - inicio_36) / 60:.1f}",
                    flush=True,
                )
                f_inv.flush()
                ultimo_aviso_36 = ahora

# ============================================================
# 4. GUARDAR RESULTADOS Y MOSTRAR RESUMEN
# ============================================================

candidatos_36.sort(key=lambda x: x["puntaje"], reverse=True)

for nombre, filas in (
    ("Candidatos_priorizados.csv", candidatos_36),
    ("Pesos_SHA256.csv", pesos_36),
):
    with (SALIDA_36 / nombre).open(
        "x", newline="", encoding="utf-8-sig"
    ) as f:
        w = csv.DictWriter(f, fieldnames=columnas)
        w.writeheader()
        w.writerows(filas)

resumen_36 = {
    "equipo": EQUIPO,
    "fecha_inicio": FECHA,
    "fecha_fin": datetime.now().astimezone().isoformat(),
    "raices_revisadas": raices_36,
    "carpetas_excluidas_por_nombre": sorted(EXCLUIR),
    "max_texto_mb": MAX_TEXTO_MB,
    "hash_pesos_activado": CALCULAR_HASH_PESOS,
    "conteos": dict(conteos_36),
    "candidatos": len(candidatos_36),
    "pesos": len(pesos_36),
    "duracion_minutos": (time.monotonic() - inicio_36) / 60,
    "carpeta_resultados": str(SALIDA_36),
    "alcance": (
        "Búsqueda de archivos accesibles en las raíces indicadas. "
        "No ejecuta código, no carga checkpoints, no examina el interior "
        "de comprimidos ni recupera archivos borrados. "
        "Los candidatos requieren revisión humana."
    ),
}
with (SALIDA_36 / "celda_36_resumen.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(resumen_36, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 75)
print("RESUMEN")
print("=" * 75)
print(json.dumps(resumen_36, ensure_ascii=False, indent=2))

print("\nPRIMEROS 25 CANDIDATOS — EL CSV CONTIENE TODOS")
for i, fila in enumerate(candidatos_36[:25], start=1):
    print(f"\n{i}. Puntaje: {fila['puntaje']} | {fila['ruta']}")
    print("   Coincidencias:", fila["grupos"])

print("\nCELDA 36 TERMINADA.")
print("Informes guardados en:", SALIDA_36)
print("Los archivos originales no se modificaron.")

Equipo: DESKTOP-NRE2L20
Raíces: ['C:\\', 'D:\\']
Informes: C:\Users\Ivan sua\Auditoria_busqueda_modelos\DESKTOP-NRE2L20_20261008_200431_209504

Buscando. Los pesos grandes pueden tardar en procesarse.

Revisando: C:\
  Archivos vistos: 796 | Textos leídos: 389 | Candidatos: 0 | Pesos: 0 | Minutos: 0.4
  Archivos vistos: 2,143 | Textos leídos: 1,402 | Candidatos: 1 | Pesos: 0 | Minutos: 0.8
  Archivos vistos: 14,144 | Textos leídos: 2,171 | Candidatos: 2 | Pesos: 0 | Minutos: 1.1
  Archivos vistos: 17,541 | Textos leídos: 2,832 | Candidatos: 2 | Pesos: 0 | Minutos: 1.4
  Archivos vistos: 18,584 | Textos leídos: 3,696 | Candidatos: 2 | Pesos: 0 | Minutos: 2.1
  Archivos vistos: 22,452 | Textos leídos: 4,286 | Candidatos: 83 | Pesos: 0 | Minutos: 2.4
  Archivos vistos: 26,319 | Textos leídos: 5,140 | Candidatos: 191 | Pesos: 0 | Minutos: 2.6
  Archivos vistos: 26,329 | Textos leídos: 5,149 | Candidatos: 191 | Pesos: 0 | Minutos: 3.9
  Archivos vistos: 26,916 | Textos leídos: 5,701 | Candi

In [44]:
# CELDA 37 — Recuperación dirigida de evidencias de U-Net
from pathlib import Path
from datetime import datetime
import hashlib
import json
import platform
import zipfile

fecha = datetime.now().astimezone()
base = Path.home() / "anaconda3"
carpeta_modelo = base / "Modelos_UNET"
zip_original = base / "Modelos_UNET.zip"

salida = (
    Path.home()
    / "Auditoria_busqueda_modelos"
    / f"Evidencias_UNET_{fecha.strftime('%Y%m%d_%H%M%S_%f')}"
)
salida.mkdir(parents=True, exist_ok=False)
paquete = salida / "Evidencias_UNET_para_revision.zip"

extensiones = {
    ".json", ".csv", ".py", ".ipynb",
    ".yaml", ".yml", ".txt", ".md"
}
limite_archivo = 25 * 1024**2
limite_total = 150 * 1024**2

registro = {
    "fecha": fecha.isoformat(),
    "equipo": platform.node(),
    "incluidos": [],
    "incidencias": [],
    "contenido_zip_original": [],
}
total = 0

def incorporar(destino, origen, nombre, obtener_bytes, tamano):
    global total

    if tamano > limite_archivo or total + tamano > limite_total:
        registro["incidencias"].append({
            "origen": origen,
            "motivo": "Omitido por límite de tamaño",
            "bytes": tamano,
        })
        return

    try:
        contenido = obtener_bytes()
        destino.writestr(nombre, contenido)
        total += len(contenido)
        registro["incluidos"].append({
            "origen": origen,
            "archivo_en_paquete": nombre,
            "bytes": len(contenido),
            "sha256": hashlib.sha256(contenido).hexdigest(),
        })
    except Exception as exc:
        registro["incidencias"].append({
            "origen": origen,
            "motivo": f"{type(exc).__name__}: {exc}",
        })

with zipfile.ZipFile(
    paquete, mode="x", compression=zipfile.ZIP_DEFLATED
) as destino:

    archivos = []

    if carpeta_modelo.is_dir():
        archivos.extend(
            p for p in carpeta_modelo.rglob("*")
            if p.is_file() and p.suffix.lower() in extensiones
        )
    else:
        registro["incidencias"].append({
            "origen": str(carpeta_modelo),
            "motivo": "Carpeta no encontrada",
        })

    # Fuentes de predicción: se registran como tales,
    # sin considerarlas automáticamente fuentes de entrenamiento.
    for carpeta in (Path.home(), Path.home() / "Downloads"):
        if carpeta.is_dir():
            archivos.extend(carpeta.glob("PREDICCION_UNET*.ipynb"))

    for numero, archivo in enumerate(
        sorted(set(archivos), key=str), start=1
    ):
        try:
            incorporar(
                destino,
                str(archivo),
                f"archivos/{numero:04d}_{archivo.name}",
                archivo.read_bytes,
                archivo.stat().st_size,
            )
        except OSError as exc:
            registro["incidencias"].append({
                "origen": str(archivo),
                "motivo": str(exc),
            })

    if zip_original.is_file():
        try:
            with zipfile.ZipFile(zip_original, "r") as fuente:
                for numero, miembro in enumerate(fuente.infolist(), 1):
                    registro["contenido_zip_original"].append({
                        "nombre": miembro.filename,
                        "bytes": miembro.file_size,
                        "carpeta": miembro.is_dir(),
                    })

                    nombre = miembro.filename.replace("\\", "/").split("/")[-1]
                    extension = Path(nombre).suffix.lower()

                    if not miembro.is_dir() and extension in extensiones:
                        incorporar(
                            destino,
                            f"{zip_original} :: {miembro.filename}",
                            f"contenido_zip/{numero:04d}{extension}",
                            lambda m=miembro: fuente.read(m),
                            miembro.file_size,
                        )
        except Exception as exc:
            registro["incidencias"].append({
                "origen": str(zip_original),
                "motivo": f"{type(exc).__name__}: {exc}",
            })
    else:
        registro["incidencias"].append({
            "origen": str(zip_original),
            "motivo": "ZIP no encontrado",
        })

    registro["bytes_recopilados"] = total
    destino.writestr(
        "registro_recopilacion.json",
        json.dumps(registro, ensure_ascii=False, indent=2),
    )

print("Equipo:", registro["equipo"])
print("Archivos incluidos:", len(registro["incluidos"]))
print("Incidencias u omisiones:", len(registro["incidencias"]))
print("Paquete generado:", paquete)
print("\nCELDA 37 TERMINADA. Insumos originales conservados.")

Equipo: DESKTOP-NRE2L20
Archivos incluidos: 15
Incidencias u omisiones: 0
Paquete generado: C:\Users\Ivan sua\Auditoria_busqueda_modelos\Evidencias_UNET_20261008_203741_136553\Evidencias_UNET_para_revision.zip

CELDA 37 TERMINADA. Insumos originales conservados.


In [1]:
# C00 — Configuración. Ejecutar una vez antes de C01–C06.
from pathlib import Path, PureWindowsPath
from datetime import datetime, timezone
import os, json, hashlib, platform, csv, gc
import numpy as np
import pandas as pd

# Si el segundo equipo tiene otra ubicación, cambiar solamente estas rutas.
AUDITORIA = Path(r"D:\PROYECTO_GRADO_1\Auditoria_comparacion_modelos\20261007_191328_200990")
DATASET = Path(r"C:\Users\Ivan sua\anaconda3\Dataset")
if not DATASET.exists():
    alternativa = Path.home() / "anaconda3" / "Dataset"
    if alternativa.exists():
        DATASET = alternativa
RASTER_DIR = Path(r"D:\PROYECTO_GRADO_1\PROYECTO_GRADO\Raster_CLC")
RAICES = [Path.home()/"anaconda3", Path(r"D:\PROYECTO_GRADO_1")]
RAICES = [p for p in RAICES if p.exists()]
if not RAICES:
    raise FileNotFoundError("Indique en RAICES la carpeta copiada al segundo equipo.")
marca = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
SALIDA = Path.home()/"Auditoria_correcciones_CLC"/(platform.node()+"_"+marca)
SALIDA.mkdir(parents=True, exist_ok=False)

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloque in iter(lambda:f.read(8*1024*1024), b""):
            h.update(bloque)
    return h.hexdigest()

def guardar(nombre, datos):
    p=SALIDA/(nombre+".json")
    payload={"fecha_UTC":datetime.now(timezone.utc).isoformat(),
             "equipo":platform.node(),"datos":datos}
    with p.open("x",encoding="utf-8") as f:
        json.dump(payload,f,ensure_ascii=False,indent=2,default=str,allow_nan=False)
    print("Registro:",p)
    return p

def archivos(raices, extensiones):
    ignorar={"envs","pkgs","node_modules",".git","__pycache__",".ipynb_checkpoints"}
    vistos=set()
    for raiz in raices:
        def error(e): print("NO ACCESIBLE:",e)
        for carpeta, dirs, files in os.walk(raiz,onerror=error):
            dirs[:]=[d for d in dirs if d not in ignorar]
            for nombre in files:
                p=Path(carpeta)/nombre
                if p.suffix.lower() in extensiones and str(p.resolve()) not in vistos:
                    vistos.add(str(p.resolve())); yield p

CAT={
 1:{1:"2. Territorios agrícolas",2:"3. Bosques y áreas seminaturales",
    3:"5. Superficies de agua",4:"1. Territorios artificializados",5:"4. Áreas húmedas"},
 2:{1:"2.4 Áreas agrícolas heterogéneas",2:"3.2 Áreas con vegetación herbácea o arbustiva",
    3:"3.1 Bosques",4:"2.3 Pastos",5:"5.1 Aguas continentales",6:"1.1 Zonas urbanizadas",
    7:"1.3 Zonas de extracción minera y escombreras",8:"4.1 Áreas húmedas continentales",
    9:"1.2 Zonas industriales o comerciales y redes de comunicación",
    10:"1.4 Zonas verdes artificializadas no agrícolas",11:"2.1 Cultivos transitorios",
    12:"2.2 Cultivos permanentes",13:"3.3 Áreas abiertas sin o con poca vegetación"}}
PADRE={1:1,2:2,3:2,4:1,5:3,6:4,7:4,8:5,9:4,10:4,11:1,12:1,13:2}
COLORES={1:["#e8c547","#39824a","#2477c7","#cb4444","#7ac6c0"],
2:["#ecd46d","#8ebd69","#28663b","#c7dc8e","#2477c7","#bd3030",
   "#92705b","#7ac6c0","#8b497d","#e98cad","#c89537","#deaa68","#b8b4a8"]}
CARPETAS={
("UNET",1):"Inferencia_UNET_nivel_1_documentada_20261008_003424_347570",
("UNET",2):"Inferencia_UNET_nivel_2_documentada_20261008_080417_807532",
("RESNET50",1):"Inferencia_RESNET50_nivel_1_20261007_235816_276690",
("RESNET50",2):"Inferencia_RESNET50_nivel_2_20261008_000718_751568",
("DENSENET121",1):"Inferencia_DENSENET121_nivel_1_20261008_001317_776842",
("DENSENET121",2):"Inferencia_DENSENET121_nivel_2_20261008_001719_131759"}
MAPAS={(m,n):AUDITORIA/c/f"{m}_nivel_{n}_prediccion.tif" for (m,n),c in CARPETAS.items()}
REFDIR=AUDITORIA/"Referencias_homologadas_20261007_220856_013511"
REFS={n:REFDIR/f"Referencia_IDEAM_2022_nivel_{n}_codigos_dataset.tif" for n in (1,2)}
EVAL=AUDITORIA/"Evaluacion_comun_20261008_081815_052051"
guardar("C00_rutas",{"auditoria":AUDITORIA,"dataset":DATASET,"raices":RAICES,
                     "salida":SALIDA,"nota":"Ningún insumo se sobrescribe."})
print("C00 terminada. Las rutas ausentes se informarán en la celda que las utilice.")

Registro: C:\Users\Ivan sua\Auditoria_correcciones_CLC\DESKTOP-NRE2L20_20261013_024043_976490\C00_rutas.json
C00 terminada. Las rutas ausentes se informarán en la celda que las utilice.


In [2]:
# C01 — CSV, geometrías, soporte de etiquetas y solapamientos.
# Requiere C00. No identifica por sí sola qué CSV utilizó cada checkpoint.
import rasterio
if not DATASET.exists(): raise FileNotFoundError(DATASET)
indice={}
for p in archivos([DATASET],{'.tif','.tiff'}):
    indice.setdefault(p.name,[]).append(p)

def resolver(valor):
    s=str(valor).strip()
    if not s or s.lower()=="nan": raise ValueError("Ruta vacía en manifiesto")
    p=Path(s)
    if p.is_file(): return p
    piezas=PureWindowsPath(s).parts
    for i,x in enumerate(piezas):
        if x.lower()=="dataset":
            q=DATASET.joinpath(*piezas[i+1:])
            if q.is_file():return q
    q=DATASET/s
    if q.is_file():return q
    opciones=indice.get(PureWindowsPath(s).name,[])
    if len(opciones)!=1:raise ValueError(f"Ruta no resuelta de forma única: {s}; {opciones}")
    return opciones[0]

def columna(df,opciones):
    encontradas=[x for x in opciones if x in df.columns]
    if not encontradas:raise ValueError(f"No identifico columna. Cabeceras: {list(df.columns)}")
    return encontradas[0]

manifiestos=[]; geo={}; reportes=[]
for n in (1,2):
 for part in ('train','val','test'):
    p=DATASET/'metadata'/'splits'/f'nivel_{n}'/f'{part}_samples.csv'
    df=pd.read_csv(p,sep=None,engine='python')
    ci=columna(df,['image_path_local','image_path','image','img_path','image_file'])
    cm=columna(df,['mask_path_local','mask_path','mask','label_path','mask_file'])
    soporte={k:0 for k in CAT[n]}; limites=[]; crs_set=set()
    for j,(_,fila) in enumerate(df.iterrows(),1):
        ip,mp=resolver(fila[ci]),resolver(fila[cm])
        with rasterio.open(ip) as im,rasterio.open(mp) as ma:
            if not (im.shape==ma.shape and im.transform==ma.transform and im.crs==ma.crs):
                raise ValueError(f"Geometría distinta: {ip} / {mp}")
            if im.count!=4:raise ValueError(f"Bandas inesperadas: {ip}")
            a=ma.read(1); v=ma.read_masks(1)>0
            cod,cnt=np.unique(a[v],return_counts=True)
            desconocidos=set(map(int,cod))-set(CAT[n])
            if desconocidos:raise ValueError(f"Códigos inesperados {mp}: {desconocidos}")
            for k,c in zip(cod,cnt):soporte[int(k)]+=int(c)
            b=im.bounds; crs_set.add(im.crs.to_wkt() if im.crs else 'SIN_CRS')
            limites.append([b.left,b.bottom,b.right,b.top])
            manifiestos.append({'nivel':n,'particion':part,'image_path':str(ip),
                 'mask_path':str(mp),'left':b.left,'bottom':b.bottom,'right':b.right,'top':b.top,
                 'origen_aumento':'PENDIENTE: verificar linaje; no inferirlo del nombre'})
        if j%500==0:print(n,part,j,'/',len(df),flush=True)
    if len(crs_set)!=1 or 'SIN_CRS' in crs_set:raise ValueError('CRS no homogéneo')
    geo[(n,part)]=(np.array(limites),next(iter(crs_set)))
    reportes.append({'nivel':n,'particion':part,'pares':len(df),
                     'soporte_suma_por_parche':soporte,'csv':str(p),'sha256_csv':sha256(p)})
solapes=[]
for n in (1,2):
 for pa,pb in [('train','val'),('train','test'),('val','test')]:
    A,ca=geo[n,pa]; B,cb=geo[n,pb]
    if ca!=cb:raise ValueError('CRS diferentes: no comparar límites directamente')
    na=set(); nb=set(); total=0
    for i,a in enumerate(A):
        hit=np.flatnonzero((B[:,0]<a[2])&(B[:,2]>a[0])&(B[:,1]<a[3])&(B[:,3]>a[1]))
        if len(hit):na.add(i);nb.update(hit.tolist());total+=len(hit)
    solapes.append({'nivel':n,'A':pa,'B':pb,'pares_con_area_superpuesta':total,
                     'parches_A':len(na),'parches_B':len(nb)})
pd.DataFrame(manifiestos).to_csv(SALIDA/'C01_geometrias.csv',index=False,encoding='utf-8-sig')
guardar('C01_particiones',{'particiones':reportes,'solapes':solapes,
 'pendientes':['Linaje original→aumento','Correspondencia de particiones con seis checkpoints',
 'Contexto completo de ventanas de inferencia','Semilla y procedimiento históricos de partición'],
 'advertencia':'Soportes sumados por parche; no equivalen a píxeles geográficos únicos.'})
print(pd.DataFrame(solapes).to_string(index=False))

1 train 500 / 1017
1 train 1000 / 1017
2 train 500 / 3727
2 train 1000 / 3727
2 train 1500 / 3727
2 train 2000 / 3727
2 train 2500 / 3727
2 train 3000 / 3727
2 train 3500 / 3727
2 val 500 / 799
2 test 500 / 799
Registro: C:\Users\Ivan sua\Auditoria_correcciones_CLC\DESKTOP-NRE2L20_20261013_024043_976490\C01_particiones.json
 nivel     A    B  pares_con_area_superpuesta  parches_A  parches_B
     1 train  val                         149        134        100
     1 train test                         144        116         90
     1   val test                          31         22         25
     2 train  val                         821        684        482
     2 train test                         833        694        482
     2   val test                         174        146        144


In [3]:
# C02 — Recuperar evidencia de entrenamiento y código sin ejecutarlo.
# Requiere C00. La extracción de checkpoints usa exclusivamente weights_only=True.
import sys, importlib.metadata as md
import torch
versiones={}
for nombre in ['numpy','pandas','rasterio','torch','torchvision','scikit-learn','openpyxl','pyproj']:
    try:versiones[nombre]=md.version(nombre)
    except md.PackageNotFoundError:versiones[nombre]='NO INSTALADO'
patrones=['class ','def forward','load_state_dict','argmax','10000','1000','clip(',
 'means','stds','ignore_index','CrossEntropy','dice','best_model','early_stop','patience',
 'random_state','seed','WeightedRandomSampler','GroupNorm','Dropout','interpolate','softmax']
configuraciones=[]; fragmentos=[]; pesos=[]; incidencias=[]; historiales=[]
for p in archivos(RAICES,{'.json','.ipynb','.py','.pt','.pth','.csv'}):
 try:
    if p.suffix.lower() in {'.pt','.pth'}:
        if not any(s in str(p).lower() for s in ['unet','u_net','resnet','densenet']):continue
        print('Leyendo pesos:',p,flush=True)
        fila={'ruta':str(p),'bytes':p.stat().st_size,'sha256':sha256(p)}
        try:
            ck=torch.load(p,map_location='cpu',weights_only=True)
            def resumen(x,depth=0):
                if isinstance(x,torch.Tensor):
                    z={'forma':list(x.shape),'tipo':str(x.dtype)}
                    if x.numel()<=16:z['valores']=x.detach().cpu().tolist()
                    return z
                if isinstance(x,dict) and depth<5:return {str(k):resumen(v,depth+1) for k,v in x.items()}
                if isinstance(x,(list,tuple)) and depth<5:return [resumen(v,depth+1) for v in x[:100]]
                if isinstance(x,(str,int,bool,type(None))):return x
                if isinstance(x,float):return x if np.isfinite(x) else str(x)
                return {'tipo':type(x).__name__}
            fila['contenido']=resumen(ck)
            del ck;gc.collect()
        except Exception as e:
            fila['lectura_restringida_fallida']=str(e)
            fila['nota']='No se cambia a weights_only=False. Revisar archivo y procedencia.'
        pesos.append(fila);continue
    if p.stat().st_size>30*1024*1024:
        incidencias.append({'ruta':str(p),'motivo':'Texto >30 MiB no leído'});continue
    if p.suffix.lower()=='.csv':
        if any(s in p.name.lower() for s in ['history','historial','epoch','metric','log']):
            t=pd.read_csv(p,sep=None,engine='python')
            historiales.append({'ruta':str(p),'sha256':sha256(p),'filas':len(t),
                 'columnas':list(t.columns),'ultimas':t.tail(5).fillna('NE').to_dict('records')})
        continue
    raw=p.read_text(encoding='utf-8-sig',errors='strict')
    if p.suffix.lower()=='.json':
        if any(s in p.name.lower() for s in ['config','mapping','decision','registro','celda_']):
            configuraciones.append({'ruta':str(p),'sha256':sha256(p),'contenido':json.loads(raw)})
        continue
    if p.suffix.lower()=='.ipynb':
        fuentes=[(i+1,''.join(c.get('source',[]))) for i,c in enumerate(json.loads(raw).get('cells',[]))
                 if c.get('cell_type')=='code']
    else:fuentes=[(None,raw)]
    if not any(s in raw.lower() for s in ['unet','u_net','resnet','densenet']):continue
    for celda,texto in fuentes:
        lineas=texto.splitlines(); seleccion=set()
        for i,linea in enumerate(lineas):
            if any(x.lower() in linea.lower() for x in patrones):
                seleccion.update(range(max(0,i-3),min(len(lineas),i+5)))
        if seleccion:fragmentos.append({'ruta':str(p),'sha256':sha256(p),'celda':celda,
           'lineas':[{'numero':i+1,'texto':lineas[i]} for i in sorted(seleccion)]})
 except Exception as e:incidencias.append({'ruta':str(p),'error':str(e)})
# Registrar objetos ya cargados sin ejecutar entrenamiento ni inferencia.
modelos_en_memoria=[]
for nombre,objeto in list(globals().items()):
    candidatos=[(nombre,objeto)] if isinstance(objeto,torch.nn.Module) else []
    if isinstance(objeto,dict):
        candidatos=[(nombre+'['+str(k)+']',v) for k,v in list(objeto.items())
                    if isinstance(v,torch.nn.Module)]
    for etiqueta,modelo in candidatos:
        modelos_en_memoria.append({'variable':etiqueta,'clase':type(modelo).__name__,
            'parametros_totales':sum(x.numel() for x in modelo.parameters()),
            'parametros_entrenables':sum(x.numel() for x in modelo.parameters() if x.requires_grad),
            'modulos':str(modelo)})
# Estos objetos no prueban su vínculo histórico sin código y hashes asociados.
# Resumen de tensores acredita formas, no demuestra el forward ni el entrenamiento ejecutado.
guardar('C02_evidencia',{'configuraciones':configuraciones,'pesos':pesos,
 'codigo_estatico':fragmentos,'historiales':historiales,'incidencias':incidencias,
 'modelos_en_memoria':modelos_en_memoria,
 'entorno_actual':{'python':sys.version,'ejecutable':sys.executable,'bibliotecas':versiones,
 'cuda_disponible':torch.cuda.is_available(),'cuda_compilacion':torch.version.cuda},
 'pendiente':'Revisar las fuentes y vincularlas al checkpoint; no deducir ejecución solo por coincidencia textual.'})
print('Configuraciones:',len(configuraciones),'pesos:',len(pesos),'fuentes:',len(fragmentos))
print('No se ejecutó código de entrenamiento ni se modificaron pesos.')

Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_1_20260815_122755\best_model.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_1_20260815_122755\checkpoints\last_checkpoint.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_2_20260815_172335\best_model.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_2_20260815_172335\checkpoints\last_checkpoint.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion\ResNet50Seg_nivel_1_20260730_191354\best_model.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion\ResNet50Seg_nivel_1_20260730_191354\checkpoints\last_checkpoint.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion\ResNet50Seg_nivel_2_20260803_201945\best_model.pt
Leyendo pesos: C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segm